## Decision Intelligence Capability (DIC)

In [1]:
# ================================================================
# DIC CODING
# INPUT FILE: Themes.xlsx
# ================================================================

import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# ------------------------------------------------
# 1. INPUT
# ------------------------------------------------

input_file = Path("Themes.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"Themes.xlsx was not found in:\n{Path.cwd()}"
    )

excel = pd.ExcelFile(input_file)


# ------------------------------------------------
# 2. FIND PARTICIPANT SHEETS
# ------------------------------------------------

participant_sheets = []

for sheet in excel.sheet_names:

    match = re.fullmatch(
        r"P?\s*0*(\d+)",
        str(sheet).strip(),
        flags=re.IGNORECASE
    )

    if match:

        number = int(match.group(1))

        if 1 <= number <= 26:
            participant_sheets.append(
                (number, sheet)
            )

participant_sheets = sorted(
    participant_sheets,
    key=lambda x: x[0]
)

print("Participant sheets found:")

for number, sheet in participant_sheets:
    print(f"P{number:02d} -> {sheet}")


# ------------------------------------------------
# 3. EXTRACT DIC RESPONSES
# ------------------------------------------------

raw_rows = []

for number, sheet in participant_sheets:

    participant = f"P{number:02d}"

    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        header=None
    )

    for row_number, row in df.iterrows():

        values = []

        for value in row.tolist():

            if pd.isna(value):
                continue

            text = str(value).strip()

            if text != "":
                values.append(text)

        if len(values) < 2:
            continue

        construct_position = None

        for position, value in enumerate(values):

            if value.strip().upper() == "DIC":

                construct_position = position
                break

        if construct_position is None:
            continue

        remaining = values[
            construct_position + 1:
        ]

        if not remaining:
            continue

        response = " ".join(
            remaining
        ).strip()

        if response:

            raw_rows.append({

                "Participant":
                    participant,

                "Construct":
                    "DIC",

                "Original_Response":
                    response,

                "Source_Sheet":
                    sheet,

                "Source_Row":
                    row_number + 1
            })


# ------------------------------------------------
# 4. CHECK
# ------------------------------------------------

original_df = pd.DataFrame(
    raw_rows
)

if original_df.empty:

    raise ValueError(
        """
        No DIC responses were found.

        Check that DIC appears in the participant sheets.
        """
    )

print("\nDIC original responses:")
print(len(original_df))


# ------------------------------------------------
# 5. SPLIT INTO RAW THEMES
# ------------------------------------------------

theme_rows = []

for _, row in original_df.iterrows():

    response = str(
        row["Original_Response"]
    )

    response = response.replace(
        "\n", ","
    )

    response = response.replace(
        ";", ","
    )

    response = response.replace(
        "•", ","
    )

    themes = response.split(",")

    for theme in themes:

        theme = str(
            theme
        ).strip()

        if theme == "":
            continue

        theme_rows.append({

            "Participant":
                row["Participant"],

            "Construct":
                "DIC",

            "Raw_Theme":
                theme,

            "Source_Sheet":
                row["Source_Sheet"],

            "Source_Row":
                row["Source_Row"]
        })


raw_df = pd.DataFrame(
    theme_rows
)


# ------------------------------------------------
# 6. CLEAN THEMES
# ------------------------------------------------

clean_df = raw_df.copy()

clean_df["Theme_Clean"] = (
    clean_df["Raw_Theme"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

clean_df["Theme_Key"] = (
    clean_df["Theme_Clean"]
    .str.lower()
    .str.strip()
)

clean_df = clean_df[
    clean_df["Theme_Key"] != ""
]

clean_df = clean_df.drop_duplicates(
    subset=[
        "Participant",
        "Construct",
        "Theme_Key"
    ]
)

clean_df = clean_df.reset_index(
    drop=True
)


print("\nDIC cleaned theme observations:")
print(len(clean_df))


# ------------------------------------------------
# 7. DISPLAY RAW DIC THEMES
# ------------------------------------------------

raw_theme_list = (
    clean_df[
        [
            "Raw_Theme",
            "Theme_Clean",
            "Theme_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Theme_Key"
    )
    .reset_index(
        drop=True
    )
)

print("\n")
print("=" * 60)
print("DIC RAW THEMES")
print("=" * 60)

print(
    raw_theme_list.to_string(
        index=False
    )
)


# ------------------------------------------------
# 8. DIC NORMALIZATION DICTIONARY
# ------------------------------------------------
#
# Add the researcher-approved DIC mappings here.
#
# Example:
#
# "data interpretation":
#     "Data interpretation",
#
# "information interpretation":
#     "Data interpretation",
#
# ------------------------------------------------

DIC_NORMALIZATION = {

    # ADD DIC MAPPINGS HERE

}


# ------------------------------------------------
# 9. APPLY NORMALIZATION
# ------------------------------------------------

coding_df = raw_theme_list.copy()

coding_df["Normalized_Theme"] = ""

coding_df["Decision"] = ""

coding_df["Reason"] = ""


for i, row in coding_df.iterrows():

    theme_key = str(
        row["Theme_Key"]
    ).strip().lower()

    if theme_key in DIC_NORMALIZATION:

        normalized = DIC_NORMALIZATION[
            theme_key
        ]

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = normalized

        if theme_key == normalized.lower():

            coding_df.loc[
                i,
                "Decision"
            ] = "Keep"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Retained as a distinct conceptual theme."
            )

        else:

            coding_df.loc[
                i,
                "Decision"
            ] = "Merge"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Merged with a semantically equivalent "
                "theme."
            )

    else:

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = row["Theme_Clean"]

        coding_df.loc[
            i,
            "Decision"
        ] = "Keep"

        coding_df.loc[
            i,
            "Reason"
        ] = (
            "Retained pending conceptual review."
        )


# ------------------------------------------------
# 10. MAP TO PARTICIPANTS
# ------------------------------------------------

mapping = coding_df[
    [
        "Theme_Key",
        "Normalized_Theme",
        "Decision"
    ]
]

coded_df = clean_df.merge(
    mapping,
    on="Theme_Key",
    how="left"
)


# ------------------------------------------------
# 11. PARTICIPANT × DIC THEME MATRIX
# ------------------------------------------------

participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

matrix_source = (
    coded_df
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
)

matrix = pd.crosstab(
    matrix_source["Normalized_Theme"],
    matrix_source["Participant"]
)

matrix = matrix.reindex(
    columns=participants,
    fill_value=0
)

matrix = matrix.reset_index()

matrix["Frequency"] = matrix[
    participants
].sum(axis=1)

matrix["Percentage"] = (
    matrix["Frequency"]
    / len(participants)
    * 100
).round(1)


# ------------------------------------------------
# 12. THEME SUMMARY
# ------------------------------------------------

theme_summary = matrix[
    [
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].sort_values(
    "Frequency",
    ascending=False
)


# ------------------------------------------------
# 13. DECISION SUMMARY
# ------------------------------------------------

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Count"
    )
)


# ------------------------------------------------
# 14. PARTICIPANT COVERAGE
# ------------------------------------------------

participant_coverage = (
    coded_df
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "DIC_Theme_Count"
]


# ------------------------------------------------
# 15. SAVE EXCEL
# ------------------------------------------------

output_file = Path(
    "DIC_Coding_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".xlsx"
)


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_df.to_excel(
        writer,
        sheet_name="04_Coding_Dictionary",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="07_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="08_Participant_Coverage",
        index=False
    )


# ------------------------------------------------
# 16. FINAL REPORT
# ------------------------------------------------

print("\n")
print("=" * 60)
print("DIC CODING COMPLETED")
print("=" * 60)

print(
    "Participants:",
    clean_df["Participant"].nunique()
)

print(
    "Raw theme observations:",
    len(clean_df)
)

print(
    "Unique DIC raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Normalized DIC themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print("\nDECISION SUMMARY:")

print(
    decision_summary.to_string(
        index=False
    )
)

print("\nOUTPUT FILE:")

print(
    output_file.resolve()
)

print("\nDONE.")

Participant sheets found:
P01 -> 1
P02 -> 2
P03 -> 3
P04 -> 4
P05 -> 5
P06 -> 6
P07 -> 7
P08 -> 8
P09 -> 9
P10 -> 10
P11 -> 11
P12 -> 12
P13 -> 13
P14 -> 14
P15 -> 15
P16 -> 16
P17 -> 17
P18 -> 18
P19 -> 19
P20 -> 20
P21 -> 21
P22 -> 22
P23 -> 23
P24 -> 24
P25 -> 25
P26 -> 26

DIC original responses:
26

DIC cleaned theme observations:
82


DIC RAW THEMES
                             Raw_Theme                            Theme_Clean                              Theme_Key
                alternative comparison                 alternative comparison                 alternative comparison
                 alternative selection                  alternative selection                  alternative selection
                          alternatives                           alternatives                           alternatives
                          Alternatives                           Alternatives                           alternatives
                 business consequences                  b

In [2]:
# ================================================================
# COMPLETE DIC QUALITATIVE CODING ANALYSIS
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# ================================================================
# 1. FIND DIC INPUT FILE
# ================================================================

files = list(Path(".").glob("DIC_Coding_20260827_130957*.xlsx"))

if len(files) == 0:
    raise FileNotFoundError(
        "\nNo file beginning with 'DIC_Coding_' was found.\n"
        "Make sure your DIC Excel file is in the same folder as "
        "your Python notebook."
    )

if len(files) > 1:
    print("DIC files found:")
    for f in files:
        print(" -", f.name)

    # Use most recently modified file
    input_file = max(files, key=lambda x: x.stat().st_mtime)
    print("\nUsing the most recently modified DIC file:")
else:
    input_file = files[0]

print(input_file.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

excel = pd.ExcelFile(input_file)

print("\nSheets found:")
for s in excel.sheet_names:
    print(" -", s)


# ================================================================
# 3. FLEXIBLE SHEET FINDER
# ================================================================

def find_sheet(possible_names):

    # Exact
    for name in possible_names:
        if name in excel.sheet_names:
            return name

    # Flexible
    for sheet in excel.sheet_names:

        a = (
            str(sheet).lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for name in possible_names:

            b = (
                str(name).lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if b in a:
                return sheet

    return None


original_sheet = find_sheet([
    "01_Original_DIC",
    "01_Original",
    "Original_DIC",
    "Original"
])

raw_sheet = find_sheet([
    "02_Raw_DIC",
    "02_Raw",
    "Raw_DIC",
    "Raw"
])

clean_sheet = find_sheet([
    "03_Cleaned_DIC",
    "03_Cleaned",
    "Cleaned_DIC",
    "Cleaned"
])

coding_sheet = find_sheet([
    "04_Normalized_Coding",
    "04_Coding",
    "Normalized_Coding",
    "Coding"
])

print("\nSelected sheets:")
print("Original:", original_sheet)
print("Raw:", raw_sheet)
print("Cleaned:", clean_sheet)
print("Coding:", coding_sheet)


# ================================================================
# 4. READ SHEETS
# ================================================================

original_df = (
    pd.read_excel(input_file, sheet_name=original_sheet)
    if original_sheet else pd.DataFrame()
)

raw_df = (
    pd.read_excel(input_file, sheet_name=raw_sheet)
    if raw_sheet else pd.DataFrame()
)

clean_df = (
    pd.read_excel(input_file, sheet_name=clean_sheet)
    if clean_sheet else pd.DataFrame()
)

if coding_sheet is None:
    raise ValueError(
        "\nCould not find the DIC Coding/Normalized Coding sheet."
    )

coding_df = pd.read_excel(
    input_file,
    sheet_name=coding_sheet
)


# ================================================================
# 5. STANDARDIZE COLUMN NAMES
# ================================================================

for df in [
    original_df,
    raw_df,
    clean_df,
    coding_df
]:

    if not df.empty:
        df.columns = [
            str(c).strip()
            for c in df.columns
        ]


print("\nCoding columns:")
print(list(coding_df.columns))


# ================================================================
# 6. COLUMN FINDER
# ================================================================

def find_column(df, candidates):

    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    for column in df.columns:

        a = (
            str(column)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for candidate in candidates:

            b = (
                str(candidate)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if a == b:
                return column

    return None


theme_key_col = find_column(
    coding_df,
    ["Theme_Key", "Theme Key", "ThemeKey"]
)

raw_theme_col = find_column(
    coding_df,
    ["Raw_Theme", "Raw Theme", "RawTheme"]
)

normalized_col = find_column(
    coding_df,
    [
        "Normalized_Theme",
        "Normalized Theme",
        "NormalizedTheme"
    ]
)

decision_col = find_column(
    coding_df,
    ["Decision"]
)


if theme_key_col is None:
    raise ValueError("Theme_Key column not found.")

if raw_theme_col is None:
    raise ValueError("Raw_Theme column not found.")

if normalized_col is None:
    raise ValueError(
        "Normalized_Theme column not found."
    )

if decision_col is None:
    raise ValueError(
        "Decision column not found."
    )


coding_df = coding_df.rename(
    columns={
        theme_key_col: "Theme_Key",
        raw_theme_col: "Raw_Theme",
        normalized_col: "Normalized_Theme",
        decision_col: "Decision"
    }
)


# ================================================================
# 7. PARTICIPANT COLUMN
# ================================================================

participant_col = find_column(
    clean_df,
    [
        "Participant",
        "Participant_ID",
        "Participant ID",
        "ParticipantID"
    ]
)

if participant_col is None:
    raise ValueError(
        "\nParticipant column not found in Cleaned DIC sheet.\n"
        "Available columns:\n"
        + str(list(clean_df.columns))
    )

if participant_col != "Participant":
    clean_df = clean_df.rename(
        columns={
            participant_col: "Participant"
        }
    )


# ================================================================
# 8. THEME KEY IN CLEANED DATA
# ================================================================

clean_theme_key = find_column(
    clean_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

if clean_theme_key is None:
    raise ValueError(
        "\nTheme_Key not found in Cleaned DIC sheet."
    )

if clean_theme_key != "Theme_Key":
    clean_df = clean_df.rename(
        columns={
            clean_theme_key: "Theme_Key"
        }
    )


# ================================================================
# 9. CLEAN CODING DICTIONARY
# ================================================================

coding_df["Theme_Key"] = (
    coding_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

coding_df["Raw_Theme"] = (
    coding_df["Raw_Theme"]
    .fillna("")
    .astype(str)
    .str.strip()
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .replace(
        ["nan", "None", "", "NaN"],
        np.nan
    )
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .astype("string")
    .str.strip()
)

coding_df["Decision"] = (
    coding_df["Decision"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 10. CLEAN PARTICIPANT DATA
# ================================================================

clean_df["Theme_Key"] = (
    clean_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

clean_df["Participant"] = (
    clean_df["Participant"]
    .astype(str)
    .str.strip()
)


# ================================================================
# 11. PARTICIPANTS
# ================================================================

participants = sorted(
    clean_df["Participant"]
    .dropna()
    .unique()
)

print("\nParticipants:")
print(participants)

print(
    "\nNumber of participants:",
    len(participants)
)


# ================================================================
# 12. MERGE CLEANED THEMES WITH CODING
# ================================================================

coded_df = clean_df.merge(
    coding_df[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision"
        ]
    ],
    on="Theme_Key",
    how="left"
)


# ================================================================
# 13. UNMAPPED THEMES
# ================================================================

missing_mapping = coded_df[
    coded_df["Normalized_Theme"].isna()
].copy()

print(
    "\nNumber of unmapped themes:",
    len(missing_mapping)
)

if len(missing_mapping) > 0:

    print("\nUnmapped themes:")

    print(
        missing_mapping[
            [
                "Participant",
                "Theme_Key"
            ]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )


# ================================================================
# 14. PARTICIPANT × NORMALIZED THEME
# ================================================================

participant_theme = (
    coded_df[
        [
            "Participant",
            "Normalized_Theme"
        ]
    ]
    .dropna()
    .drop_duplicates()
)


# ================================================================
# 15. PARTICIPANT × THEME MATRIX
# ================================================================

matrix = pd.crosstab(
    participant_theme["Normalized_Theme"],
    participant_theme["Participant"]
)


# P01–P26 first

expected_participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

existing = [
    p for p in expected_participants
    if p in matrix.columns
]

other = [
    p for p in matrix.columns
    if p not in existing
]

matrix = matrix.reindex(
    columns=existing + other,
    fill_value=0
)

matrix = matrix.reset_index()


# ================================================================
# 16. FREQUENCY
# ================================================================

participant_columns = existing + other

matrix["Frequency"] = (
    matrix[participant_columns]
    .sum(axis=1)
)


# ================================================================
# 17. PERCENTAGE
# ================================================================

total_participants = len(participants)

matrix["Percentage"] = (
    matrix["Frequency"]
    / total_participants
    * 100
).round(1)


# ================================================================
# 18. RANK
# ================================================================

matrix = matrix.sort_values(
    [
        "Frequency",
        "Normalized_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)

matrix.insert(
    0,
    "Rank",
    range(1, len(matrix) + 1)
)


# ================================================================
# 19. THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Rank",
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].copy()

theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme":
            "Final_DIC_Theme",

        "Frequency":
            "Experts_Mentioning",

        "Percentage":
            "Percentage_of_Experts"
    }
)


# ================================================================
# 20. PREVALENCE
# ================================================================

def prevalence_category(p):

    if p >= 75:
        return "Very High"

    elif p >= 50:
        return "High"

    elif p >= 25:
        return "Moderate"

    else:
        return "Low"


theme_summary[
    "Prevalence_Category"
] = (
    theme_summary[
        "Percentage_of_Experts"
    ].apply(
        prevalence_category
    )
)


# ================================================================
# 21. CODING AUDIT
# ================================================================

theme_participant_counts = (
    participant_theme
    .groupby(
        "Normalized_Theme"
    )
    .size()
    .reset_index(
        name="Experts_Mentioning"
    )
)


coding_audit = coding_df.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)


coding_audit[
    "Experts_Mentioning"
] = (
    coding_audit[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


coding_audit[
    "Percentage_of_Experts"
] = (
    coding_audit[
        "Experts_Mentioning"
    ]
    / total_participants
    * 100
).round(1)


coding_audit = coding_audit.sort_values(
    [
        "Normalized_Theme",
        "Decision",
        "Raw_Theme"
    ]
).reset_index(drop=True)


# ================================================================
# 22. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Number_of_Raw_Themes"
    )
)

decision_summary[
    "Percentage"
] = (
    decision_summary[
        "Number_of_Raw_Themes"
    ]
    / len(coding_df)
    * 100
).round(1)


# ================================================================
# 23. NORMALIZATION SUMMARY
# ================================================================

normalization_summary = (
    coding_df
    .dropna(
        subset=["Normalized_Theme"]
    )
    .groupby(
        "Normalized_Theme"
    )
    .agg(
        Raw_Themes=(
            "Raw_Theme",
            "count"
        ),

        Keep_Count=(
            "Decision",
            lambda x:
            (x == "Keep").sum()
        ),

        Merge_Count=(
            "Decision",
            lambda x:
            (x == "Merge").sum()
        )
    )
    .reset_index()
)


normalization_summary = (
    normalization_summary.merge(
        theme_participant_counts,
        on="Normalized_Theme",
        how="left"
    )
)


normalization_summary[
    "Experts_Mentioning"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


normalization_summary[
    "Percentage_of_Experts"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    / total_participants
    * 100
).round(1)


normalization_summary = (
    normalization_summary
    .sort_values(
        "Experts_Mentioning",
        ascending=False
    )
    .reset_index(drop=True)
)


# ================================================================
# 24. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    participant_theme
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "Number_of_Normalized_Themes"
]


# ================================================================
# 25. DIC CONSTRUCT STATISTICS
# ================================================================

construct_statistics = pd.DataFrame({

    "Construct": ["DIC"],

    "Participants": [
        total_participants
    ],

    "Original_Response_Rows": [
        len(original_df)
    ],

    "Raw_Theme_Observations": [
        len(raw_df)
    ],

    "Cleaned_Theme_Observations": [
        len(clean_df)
    ],

    "Unique_Raw_Themes": [
        coding_df["Theme_Key"].nunique()
    ],

    "Final_Normalized_Themes": [
        coding_df[
            "Normalized_Theme"
        ].nunique()
    ],

    "Merged_Raw_Themes": [
        (
            coding_df["Decision"]
            == "Merge"
        ).sum()
    ],

    "Kept_Raw_Themes": [
        (
            coding_df["Decision"]
            == "Keep"
        ).sum()
    ],

    "Unmapped_Themes": [
        len(missing_mapping)
    ]
})


# ================================================================
# 26. FINAL DIC EVIDENCE
# ================================================================

final_evidence = normalization_summary[
    [
        "Normalized_Theme",
        "Raw_Themes",
        "Experts_Mentioning",
        "Percentage_of_Experts",
        "Keep_Count",
        "Merge_Count"
    ]
].copy()


final_evidence = final_evidence.rename(
    columns={
        "Normalized_Theme":
            "Final_DIC_Theme",

        "Raw_Themes":
            "Number_of_Raw_Themes",

        "Percentage_of_Experts":
            "Expert_Prevalence_%",

        "Keep_Count":
            "Raw_Themes_Kept",

        "Merge_Count":
            "Raw_Themes_Merged"
    }
)


final_evidence[
    "Prevalence_Category"
] = (
    final_evidence[
        "Expert_Prevalence_%"
    ].apply(
        prevalence_category
    )
)


# ================================================================
# 27. QUALITY CHECKS
# ================================================================

duplicate_count = (
    len(coded_df)
    -
    len(
        coded_df[
            [
                "Participant",
                "Normalized_Theme"
            ]
        ]
        .dropna()
        .drop_duplicates()
    )
)


quality_checks = pd.DataFrame({

    "Check": [

        "Number of participants",

        "Original response rows",

        "Raw theme observations",

        "Unique raw themes",

        "Final normalized themes",

        "Unmapped themes",

        "Duplicate participant-theme records",

        "Keep decisions",

        "Merge decisions"

    ],

    "Result": [

        total_participants,

        len(original_df),

        len(raw_df),

        coding_df[
            "Theme_Key"
        ].nunique(),

        coding_df[
            "Normalized_Theme"
        ].nunique(),

        len(missing_mapping),

        duplicate_count,

        (
            coding_df["Decision"]
            == "Keep"
        ).sum(),

        (
            coding_df["Decision"]
            == "Merge"
        ).sum()

    ],

    "Status": [

        "PASS"
        if total_participants > 0
        else "CHECK",

        "PASS"
        if len(original_df) > 0
        else "CHECK",

        "PASS"
        if len(raw_df) > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Theme_Key"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Normalized_Theme"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if len(missing_mapping) == 0
        else "CHECK",

        "INFO",

        "PASS",

        "PASS"

    ]
})


# ================================================================
# 28. SAVE COMPLETE DIC WORKBOOK
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_file = Path(
    f"DIC_FINAL_QUALITATIVE_ANALYSIS_{timestamp}.xlsx"
)


# Avoid PermissionError if file already exists/open
if output_file.exists():

    output_file = Path(
        f"DIC_FINAL_QUALITATIVE_ANALYSIS_{timestamp}_NEW.xlsx"
    )


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_DIC_Evidence",
        index=False
    )

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    missing_mapping.to_excel(
        writer,
        sheet_name="13_Unmapped_Themes",
        index=False
    )


# ================================================================
# 29. FINAL REPORT
# ================================================================

print("\n")
print("=" * 70)
print("DIC ANALYSIS COMPLETED")
print("=" * 70)

print(
    "\nParticipants:",
    total_participants
)

print(
    "Original responses:",
    len(original_df)
)

print(
    "Raw theme observations:",
    len(raw_df)
)

print(
    "Cleaned theme observations:",
    len(clean_df)
)

print(
    "Unique raw themes:",
    coding_df[
        "Theme_Key"
    ].nunique()
)

print(
    "Final normalized DIC themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print(
    "Keep decisions:",
    (
        coding_df[
            "Decision"
        ] == "Keep"
    ).sum()
)

print(
    "Merge decisions:",
    (
        coding_df[
            "Decision"
        ] == "Merge"
    ).sum()
)

print(
    "Unmapped themes:",
    len(missing_mapping)
)


# ================================================================
# 30. DISPLAY FINAL DIC THEMES
# ================================================================

print("\n")
print("=" * 70)
print("FINAL DIC THEMES")
print("=" * 70)

print(
    final_evidence[
        [
            "Final_DIC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Prevalence_Category"
        ]
    ].to_string(index=False)
)


# ================================================================
# 31. OUTPUT LOCATION
# ================================================================

print("\n")
print("=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(
    output_file.resolve()
)

print(
    "\nComplete DIC qualitative analysis workbook "
    "created successfully."
)

C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Decision_Intelligence_Risk_Coordination_Supply_Chain_Resilience_1\DIC_Coding_20260827_130957.xlsx

Sheets found:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Dictionary
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Decision_Summary
 - 08_Participant_Coverage

Selected sheets:
Original: 01_Original_Responses
Raw: 02_Raw_Themes
Cleaned: 03_Cleaned_Themes
Coding: 04_Coding_Dictionary

Coding columns:
['Raw_Theme', 'Theme_Clean', 'Theme_Key', 'Normalized_Theme', 'Decision', 'Reason']

Participants:
['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26']

Number of participants: 26

Number of unmapped themes: 0


DIC ANALYSIS COMPLETED

Participants: 26
Original responses: 26
Raw theme observations: 82
Cleaned theme observations: 82
Unique raw them

In [ ]:
# ================================================================
# DIC — FINAL QUALITATIVE EVIDENCE + DIMENSION ANALYSIS
# ================================================================
#
# INPUT:
# DIC_FINAL_QUALITATIVE_ANALYSIS_20260827_131057.xlsx
#
# MAIN OUTPUT:
# DIC_FINAL_EVIDENCE_AND_DIMENSIONS_20260827_131151.xlsx
#
# PURPOSE:
# 1. Preserve the completed DIC qualitative coding
# 2. Extract final DIC evidence
# 3. Organize themes into conceptually meaningful DIC dimensions
# 4. Calculate expert prevalence
# 5. Identify themes requiring manual review
# 6. Produce dimension-level evidence for questionnaire development
#
# IMPORTANT:
# The dimensions generated here are QUALITATIVE DIMENSIONS.
# They are NOT yet statistically validated measurement dimensions.
# ================================================================


import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import os


# ================================================================
# 1. INPUT FILE
# ================================================================

TARGET = "DIC_FINAL_QUALITATIVE_ANALYSIS_20260827_131057"

possible_files = []

for ext in [".xlsx", ".xlsm", ".xls"]:
    possible_files.extend(
        Path(".").rglob(TARGET + ext)
    )

if len(possible_files) == 0:
    raise FileNotFoundError(
        "\nDIC input file was not found.\n\n"
        f"Expected:\n{TARGET}.xlsx\n\n"
        "Put the file in the same folder as the notebook."
    )

INPUT_FILE = possible_files[0]

print("=" * 80)
print("DIC FINAL QUALITATIVE ANALYSIS")
print("=" * 80)

print("\nInput file:")
print(INPUT_FILE.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

xls = pd.ExcelFile(
    INPUT_FILE,
    engine="openpyxl"
)

print("\nAvailable sheets:")

for s in xls.sheet_names:
    print(" -", s)


# ================================================================
# 3. LOAD ALL SHEETS
# ================================================================

sheets = {}

for sheet in xls.sheet_names:

    try:
        sheets[sheet] = pd.read_excel(
            INPUT_FILE,
            sheet_name=sheet
        )
    except Exception as e:
        print(
            f"Warning: could not read {sheet}: {e}"
        )


# ================================================================
# 4. HELPER FUNCTIONS
# ================================================================

def find_sheet(keyword):

    for s in sheets.keys():

        if keyword.lower() in s.lower():
            return s

    return None


def clean_text(x):

    if pd.isna(x):
        return ""

    x = str(x).strip()

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x


def prevalence_category(x):

    try:
        x = float(x)
    except:
        return "Not available"

    if x >= 75:
        return "Very High"

    elif x >= 50:
        return "High"

    elif x >= 25:
        return "Moderate"

    else:
        return "Low"


# ================================================================
# 5. IDENTIFY EXISTING SHEETS
# ================================================================

original_sheet = find_sheet(
    "01_Original_Responses"
)

raw_sheet = find_sheet(
    "02_Raw_Themes"
)

cleaned_sheet = find_sheet(
    "03_Cleaned_Themes"
)

audit_sheet = find_sheet(
    "04_Coding_Audit"
)

matrix_sheet = find_sheet(
    "05_Participant_Matrix"
)

theme_sheet = find_sheet(
    "06_Theme_Summary"
)

normalization_sheet = find_sheet(
    "07_Normalization_Summary"
)

decision_sheet = find_sheet(
    "08_Decision_Summary"
)

coverage_sheet = find_sheet(
    "09_Participant_Coverage"
)

statistics_sheet = find_sheet(
    "10_Construct_Statistics"
)

final_sheet = find_sheet(
    "11_Final_DIC_Evidence"
)

quality_sheet = find_sheet(
    "12_Quality_Checks"
)

unmapped_sheet = find_sheet(
    "13_Unmapped_Themes"
)


print("\nDetected final evidence sheet:")
print(final_sheet)


# ================================================================
# 6. READ FINAL DIC EVIDENCE
# ================================================================

if final_sheet is None:

    raise ValueError(
        "11_Final_DIC_Evidence was not found."
    )

final_evidence = sheets[
    final_sheet
].copy()

final_evidence.columns = [
    str(c).strip()
    for c in final_evidence.columns
]


print("\nFinal DIC evidence columns:")
print(
    list(
        final_evidence.columns
    )
)


# ================================================================
# 7. IDENTIFY DIC THEME COLUMN
# ================================================================

theme_col = None

for c in final_evidence.columns:

    if str(c).strip() == "Final_DIC_Theme":

        theme_col = c
        break


if theme_col is None:

    for c in final_evidence.columns:

        if (
            "DIC" in str(c)
            and "Theme" in str(c)
        ):

            theme_col = c
            break


if theme_col is None:

    for c in final_evidence.columns:

        if "Theme" in str(c):

            theme_col = c
            break


if theme_col is None:

    raise ValueError(
        "Could not identify Final DIC Theme column."
    )


final_evidence[
    "Final_DIC_Theme"
] = (
    final_evidence[
        theme_col
    ]
    .apply(clean_text)
)


# ================================================================
# 8. STANDARDIZE PREVALENCE COLUMNS
# ================================================================

if (
    "Experts_Mentioning"
    not in final_evidence.columns
):

    for c in final_evidence.columns:

        if (
            "Experts" in str(c)
            and "Mention" in str(c)
        ):

            final_evidence[
                "Experts_Mentioning"
            ] = final_evidence[c]

            break


if (
    "Expert_Prevalence_%"
    not in final_evidence.columns
):

    for c in final_evidence.columns:

        if (
            "Prevalence" in str(c)
            and "%" in str(c)
        ):

            final_evidence[
                "Expert_Prevalence_%"
            ] = final_evidence[c]

            break


# ================================================================
# 9. REMOVE EMPTY THEMES
# ================================================================

final_evidence = final_evidence[
    final_evidence[
        "Final_DIC_Theme"
    ] != ""
].copy()


# ================================================================
# 10. DETERMINE NUMBER OF PARTICIPANTS
# ================================================================

participant_matrix = None

if matrix_sheet is not None:

    participant_matrix = sheets[
        matrix_sheet
    ].copy()


participants = []

if participant_matrix is not None:

    for c in participant_matrix.columns:

        cstr = str(c).strip()

        if re.match(
            r"^P\d+$",
            cstr,
            flags=re.IGNORECASE
        ):

            participants.append(
                cstr
            )


if len(participants) > 0:

    n_participants = len(
        participants
    )

else:

    # Your DIC study uses 26 experts.
    # This is used only if the participant matrix
    # does not expose P01...P26 columns.

    n_participants = 26


print(
    "\nNumber of participants:",
    n_participants
)


# ================================================================
# 11. PREVALENCE CATEGORY
# ================================================================

if (
    "Expert_Prevalence_%"
    in final_evidence.columns
):

    final_evidence[
        "Prevalence_Category"
    ] = (
        final_evidence[
            "Expert_Prevalence_%"
        ]
        .apply(
            prevalence_category
        )
    )


# ================================================================
# 12. SORT FINAL DIC EVIDENCE
# ================================================================

sort_columns = []

if "Experts_Mentioning" in final_evidence.columns:
    sort_columns.append(
        "Experts_Mentioning"
    )

sort_columns.append(
    "Final_DIC_Theme"
)


final_evidence = (
    final_evidence
    .sort_values(
        sort_columns,
        ascending=[
            False
            if c == "Experts_Mentioning"
            else True
            for c in sort_columns
        ]
    )
    .reset_index(drop=True)
)


# ================================================================
# 13. FINAL DIC EVIDENCE TABLE
# ================================================================

preferred_columns = [

    "Final_DIC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Raw_Themes_Kept",

    "Raw_Themes_Merged",

    "Prevalence_Category"

]

evidence_columns = [
    c
    for c in preferred_columns
    if c in final_evidence.columns
]


final_dic_evidence = final_evidence[
    evidence_columns
].copy()


# IMPORTANT:
# Avoid "Rank already exists" errors.
# Remove any previous Rank column first.

if "Rank" in final_dic_evidence.columns:

    final_dic_evidence = (
        final_dic_evidence
        .drop(
            columns=["Rank"]
        )
    )


final_dic_evidence.insert(
    0,
    "Rank",
    range(
        1,
        len(final_dic_evidence) + 1
    )
)


# ================================================================
# 14. DIC DIMENSION MAPPING
# ================================================================
#
# The dimensions below are derived from the actual themes found
# in your DIC qualitative evidence.
#
# Dimension 1:
# Information Integration & Interpretation
#
# Dimension 2:
# Evidence-Based Decision Framing
#
# Dimension 3:
# Alternative & Option Evaluation
#
# Dimension 4:
# Consequence & Scenario Analysis
#
# Dimension 5:
# Multi-Criteria & Multi-Perspective Assessment
#
# Dimension 6:
# Decision-Relevant Information & Knowledge Integration
#
# Themes that cannot be assigned confidently are marked
# "Review Required".
#
# ================================================================


def map_dic_dimension(theme):

    t = str(theme).lower().strip()


    # ------------------------------------------------------------
    # 1. INFORMATION INTEGRATION & INTERPRETATION
    # ------------------------------------------------------------

    information_words = [

        "multi-source information",
        "multi source information",
        "information integration",
        "evidence integration",
        "interpretation",
        "conflicting information",
        "relevant information",
        "filtering",
        "multiple indicators",
        "cross-functional information",
        "fact–assumption distinction",
        "fact-assumption distinction"

    ]

    if any(
        word in t
        for word in information_words
    ):

        return (
            "Information Integration & Interpretation"
        )


    # ------------------------------------------------------------
    # 2. EVIDENCE-BASED DECISION FRAMING
    # ------------------------------------------------------------

    decision_framing_words = [

        "decision framing",
        "decision focus",
        "decision-oriented information",
        "decision oriented information",
        "informed decisions",
        "evidence + experience",
        "data + experience",
        "data + expertise",
        "data + knowledge",
        "data + operational knowledge",
        "operational knowledge",
        "managerial knowledge",
        "experience",
        "capacity",
        "suitability"

    ]

    if any(
        word in t
        for word in decision_framing_words
    ):

        return (
            "Evidence-Based Decision Framing"
        )


    # ------------------------------------------------------------
    # 3. ALTERNATIVE & OPTION EVALUATION
    # ------------------------------------------------------------

    alternative_words = [

        "alternatives",
        "alternative comparison",
        "alternative selection",
        "option comparison",
        "option selection",
        "options",
        "systematic comparison",
        "suitability"

    ]

    if any(
        word in t
        for word in alternative_words
    ):

        return (
            "Alternative & Option Evaluation"
        )


    # ------------------------------------------------------------
    # 4. CONSEQUENCE & SCENARIO ANALYSIS
    # ------------------------------------------------------------

    consequence_words = [

        "consequence",
        "consequences",
        "consequence analysis",
        "consequence assessment",
        "business consequences",
        "downstream consequences",
        "knock-on effects",
        "immediate/long-term consequences",
        "immediate and long-term effects",
        "temporal consequences",
        "scenario",
        "scenarios",
        "scenario comparison",
        "scenario consideration",
        "scenario testing",
        "forward-looking assessment",
        "future continuity"

    ]

    if any(
        word in t
        for word in consequence_words
    ):

        return (
            "Consequence & Scenario Analysis"
        )


    # ------------------------------------------------------------
    # 5. MULTI-CRITERIA & MULTI-PERSPECTIVE ASSESSMENT
    # ------------------------------------------------------------

    multi_criteria_words = [

        "multi-criteria evaluation",
        "multi criteria evaluation",
        "multi-criteria decisions",
        "multi criteria decisions",
        "multi-criteria assessment",
        "multi criteria assessment",
        "multi-perspective assessment",
        "multi perspective assessment",
        "multi-outcome assessment",
        "multi-dimensional consequences",
        "trade-offs",
        "financial/customer assessment",
        "financial/sustainability effects",
        "operational/financial/customer effects",
        "overall outcomes",
        "systemic thinking",
        "interdependencies",
        "risk tolerance"

    ]

    if any(
        word in t
        for word in multi_criteria_words
    ):

        return (
            "Multi-Criteria & Multi-Perspective Assessment"
        )


    # ------------------------------------------------------------
    # 6. DECISION-RELEVANT INFORMATION & KNOWLEDGE INTEGRATION
    # ------------------------------------------------------------

    knowledge_words = [

        "data + knowledge",
        "data + operational knowledge",
        "data + experience",
        "data + expertise",
        "operational knowledge",
        "managerial knowledge",
        "cross-functional information",
        "evidence + experience",
        "relevant information",
        "information integration"

    ]

    if any(
        word in t
        for word in knowledge_words
    ):

        return (
            "Decision-Relevant Information & Knowledge Integration"
        )


    # ------------------------------------------------------------
    # 7. REVIEW REQUIRED
    # ------------------------------------------------------------

    return "Review Required"


final_dic_evidence[
    "DIC_Dimension"
] = (
    final_dic_evidence[
        "Final_DIC_Theme"
    ]
    .apply(
        map_dic_dimension
    )
)


# ================================================================
# 15. MANUALLY RESOLVE SPECIFIC AMBIGUITIES
# ================================================================
#
# These explicit mappings prevent generic words such as
# "Alternatives", "capacity", or "suitability" from being
# incorrectly grouped.
#
# ================================================================

manual_mapping = {

    "alternatives":
        "Alternative & Option Evaluation",

    "Alternatives":
        "Alternative & Option Evaluation",

    "alternative comparison":
        "Alternative & Option Evaluation",

    "alternative selection":
        "Alternative & Option Evaluation",

    "option comparison":
        "Alternative & Option Evaluation",

    "Option comparison":
        "Alternative & Option Evaluation",

    "systematic comparison":
        "Alternative & Option Evaluation",

    "consequences":
        "Consequence & Scenario Analysis",

    "Consequence analysis":
        "Consequence & Scenario Analysis",

    "consequence analysis":
        "Consequence & Scenario Analysis",

    "Consequence assessment":
        "Consequence & Scenario Analysis",

    "business consequences":
        "Consequence & Scenario Analysis",

    "downstream consequences":
        "Consequence & Scenario Analysis",

    "knock-on effects":
        "Consequence & Scenario Analysis",

    "immediate/long-term consequences":
        "Consequence & Scenario Analysis",

    "immediate and long-term effects":
        "Consequence & Scenario Analysis",

    "temporal consequences":
        "Consequence & Scenario Analysis",

    "scenario consideration":
        "Consequence & Scenario Analysis",

    "scenario testing":
        "Consequence & Scenario Analysis",

    "scenario comparison":
        "Consequence & Scenario Analysis",

    "scenarios":
        "Consequence & Scenario Analysis",

    "Multi-criteria evaluation":
        "Multi-Criteria & Multi-Perspective Assessment",

    "multi-criteria evaluation":
        "Multi-Criteria & Multi-Perspective Assessment",

    "multi-criteria assessment":
        "Multi-Criteria & Multi-Perspective Assessment",

    "multi-criteria decisions":
        "Multi-Criteria & Multi-Perspective Assessment",

    "trade-offs":
        "Multi-Criteria & Multi-Perspective Assessment",

    "Systemic thinking":
        "Multi-Criteria & Multi-Perspective Assessment",

    "interdependencies":
        "Multi-Criteria & Multi-Perspective Assessment",

    "risk tolerance":
        "Multi-Criteria & Multi-Perspective Assessment",

    "Multi-perspective assessment":
        "Multi-Criteria & Multi-Perspective Assessment",

    "multi-outcome assessment":
        "Multi-Criteria & Multi-Perspective Assessment",

    "multi-dimensional consequences":
        "Multi-Criteria & Multi-Perspective Assessment",

    "financial/customer assessment":
        "Multi-Criteria & Multi-Perspective Assessment",

    "financial/sustainability effects":
        "Multi-Criteria & Multi-Perspective Assessment",

    "operational/financial/customer effects":
        "Multi-Criteria & Multi-Perspective Assessment",

    "overall outcomes":
        "Multi-Criteria & Multi-Perspective Assessment",

    "future continuity":
        "Consequence & Scenario Analysis",

    "Forward-looking assessment":
        "Consequence & Scenario Analysis",

    "Decision framing":
        "Evidence-Based Decision Framing",

    "Decision focus":
        "Evidence-Based Decision Framing",

    "Decision-oriented information":
        "Evidence-Based Decision Framing",

    "informed decisions":
        "Evidence-Based Decision Framing",

    "Evidence + experience":
        "Evidence-Based Decision Framing",

    "data + experience":
        "Evidence-Based Decision Framing",

    "data + expertise":
        "Evidence-Based Decision Framing",

    "data + knowledge":
        "Decision-Relevant Information & Knowledge Integration",

    "data + operational knowledge":
        "Decision-Relevant Information & Knowledge Integration",

    "operational knowledge":
        "Decision-Relevant Information & Knowledge Integration",

    "managerial knowledge":
        "Decision-Relevant Information & Knowledge Integration",

    "cross-functional information":
        "Decision-Relevant Information & Knowledge Integration",

    "experience":
        "Evidence-Based Decision Framing",

    "capacity":
        "Evidence-Based Decision Framing",

    "suitability":
        "Evidence-Based Decision Framing",

    "Multi-source information":
        "Information Integration & Interpretation",

    "multi-source information":
        "Information Integration & Interpretation",

    "Evidence integration":
        "Information Integration & Interpretation",

    "information integration":
        "Information Integration & Interpretation",

    "Interpretation":
        "Information Integration & Interpretation",

    "filtering":
        "Information Integration & Interpretation",

    "Multiple indicators":
        "Information Integration & Interpretation",

    "Conflicting information":
        "Information Integration & Interpretation",

    "relevant information":
        "Information Integration & Interpretation",

    "Fact–assumption distinction":
        "Information Integration & Interpretation"

}


for theme, dimension in manual_mapping.items():

    mask = (
        final_dic_evidence[
            "Final_DIC_Theme"
        ]
        .astype(str)
        .str.strip()
        .eq(theme)
    )

    final_dic_evidence.loc[
        mask,
        "DIC_Dimension"
    ] = dimension


# ================================================================
# 16. CREATE DIMENSION SUMMARY
# ================================================================

dimension_rows = []


for dimension, group in (
    final_dic_evidence
    .groupby(
        "DIC_Dimension",
        dropna=False
    )
):

    themes = (
        group[
            "Final_DIC_Theme"
        ]
        .astype(str)
        .tolist()
    )


    # ------------------------------------------------------------
    # Calculate experts mentioning dimension
    # ------------------------------------------------------------

    experts_mentioning = 0


    if (
        participant_matrix is not None
        and len(participants) > 0
    ):

        temp = participant_matrix.copy()

        first_col = temp.columns[0]

        temp[
            "_Theme"
        ] = (
            temp[
                first_col
            ]
            .astype(str)
            .str.strip()
        )


        # Case-insensitive matching
        theme_lower = {
            str(x).strip().lower()
            for x in themes
        }


        matched = temp[
            temp[
                "_Theme"
            ]
            .str.lower()
            .isin(
                theme_lower
            )
        ]


        if len(matched) > 0:

            available_participants = [
                p
                for p in participants
                if p in matched.columns
            ]


            if len(
                available_participants
            ) > 0:

                vals = (
                    matched[
                        available_participants
                    ]
                    .apply(
                        pd.to_numeric,
                        errors="coerce"
                    )
                    .fillna(0)
                )


                experts_mentioning = int(
                    (
                        vals.sum(axis=0) > 0
                    ).sum()
                )


    # ------------------------------------------------------------
    # Fallback if participant matrix cannot be used
    # ------------------------------------------------------------

    if experts_mentioning == 0:

        if (
            "Experts_Mentioning"
            in group.columns
        ):

            experts_mentioning = int(
                group[
                    "Experts_Mentioning"
                ]
                .max()
            )


    prevalence = (

        experts_mentioning
        /
        n_participants
        *
        100

        if n_participants > 0
        else 0
    )


    dimension_rows.append({

        "DIC_Dimension":
            dimension,

        "Number_of_Normalized_Themes":
            len(themes),

        "Experts_Mentioning":
            experts_mentioning,

        "Expert_Prevalence_%":
            round(
                prevalence,
                1
            ),

        "Prevalence_Category":
            prevalence_category(
                prevalence
            ),

        "Included_DIC_Themes":
            "; ".join(
                themes
            )

    })


dimension_summary = pd.DataFrame(
    dimension_rows
)


# ================================================================
# 17. SORT DIMENSIONS
# ================================================================

if len(dimension_summary) > 0:

    dimension_summary = (
        dimension_summary
        .sort_values(
            [
                "Experts_Mentioning",
                "Number_of_Normalized_Themes"
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )


    if "Rank" in dimension_summary.columns:

        dimension_summary = (
            dimension_summary
            .drop(
                columns=["Rank"]
            )
        )


    dimension_summary.insert(
        0,
        "Rank",
        range(
            1,
            len(dimension_summary) + 1
        )
    )


# ================================================================
# 18. DIMENSION × THEME TABLE
# ================================================================

dimension_theme_columns = [

    "DIC_Dimension",

    "Final_DIC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Prevalence_Category"

]


dimension_theme_columns = [
    c
    for c in dimension_theme_columns
    if c in final_dic_evidence.columns
]


dimension_themes = final_dic_evidence[
    dimension_theme_columns
].copy()


dimension_themes = (
    dimension_themes
    .sort_values(
        [
            "DIC_Dimension",
            "Experts_Mentioning"
        ],
        ascending=[
            True,
            False
        ]
    )
)


# ================================================================
# 19. REVIEW-REQUIRED THEMES
# ================================================================

unmapped_check = final_dic_evidence[
    final_dic_evidence[
        "DIC_Dimension"
    ]
    ==
    "Review Required"
].copy()


if len(unmapped_check) > 0:

    unmapped_check = unmapped_check[
        [
            "Final_DIC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%"
        ]
    ].copy()

    unmapped_check.insert(
        0,
        "Status",
        "REVIEW REQUIRED"
    )

else:

    unmapped_check = pd.DataFrame({

        "Status": [
            "PASS — All DIC themes mapped to a dimension."
        ]

    })


# ================================================================
# 20. DECISION CHECK
# ================================================================

coding_audit = None

if audit_sheet is not None:

    coding_audit = sheets[
        audit_sheet
    ].copy()


if coding_audit is not None:

    decision_columns = [

        c
        for c in coding_audit.columns
        if "Decision" in str(c)

    ]


    if len(decision_columns) > 0:

        decision_col = decision_columns[0]

        missing = coding_audit[
            coding_audit[
                decision_col
            ].isna()
        ]


        if len(missing) == 0:

            decision_check = pd.DataFrame({

                "Status": [
                    "PASS — All DIC coding decisions are present."
                ]

            })

        else:

            decision_check = (
                missing.copy()
            )


    else:

        decision_check = pd.DataFrame({

            "Status": [
                "CHECK — Decision column not detected."
            ]

        })

else:

    decision_check = pd.DataFrame({

        "Status": [
            "CHECK — Coding audit unavailable."
        ]

    })


# ================================================================
# 21. QUALITY CHECKS
# ================================================================

quality_rows = []


quality_rows.append({

    "Quality_Check":
        "Input workbook found",

    "Result":
        "PASS",

    "Details":
        INPUT_FILE.name

})


quality_rows.append({

    "Quality_Check":
        "Final DIC evidence available",

    "Result":
        "PASS"
        if len(final_dic_evidence) > 0
        else "FAIL",

    "Details":
        f"{len(final_dic_evidence)} final DIC themes"

})


quality_rows.append({

    "Quality_Check":
        "Participants",

    "Result":
        n_participants,

    "Details":
        "Participant count used for prevalence"

})


quality_rows.append({

    "Quality_Check":
        "DIC dimensions generated",

    "Result":
        len(dimension_summary),

    "Details":
        "Qualitative dimensions"

})


quality_rows.append({

    "Quality_Check":
        "Themes requiring review",

    "Result":
        len(unmapped_check)
        if "Final_DIC_Theme"
        in unmapped_check.columns
        else 0,

    "Details":
        "Themes assigned to Review Required"

})


quality_rows.append({

    "Quality_Check":
        "Duplicate theme labels",

    "Result":
        int(
            final_dic_evidence[
                "Final_DIC_Theme"
            ]
            .str.lower()
            .duplicated()
            .sum()
        ),

    "Details":
        "Case-insensitive duplicate labels detected"

})


quality_checks_new = pd.DataFrame(
    quality_rows
)


# ================================================================
# 22. LOAD ORIGINAL SUPPORTING SHEETS
# ================================================================

def get_sheet_or_empty(sheet):

    if sheet is not None:

        return sheets[
            sheet
        ].copy()

    return pd.DataFrame()


original_responses = get_sheet_or_empty(
    original_sheet
)

raw_themes = get_sheet_or_empty(
    raw_sheet
)

cleaned_themes = get_sheet_or_empty(
    cleaned_sheet
)

theme_summary_existing = get_sheet_or_empty(
    theme_sheet
)

normalization_summary = get_sheet_or_empty(
    normalization_sheet
)

decision_summary = get_sheet_or_empty(
    decision_sheet
)

participant_coverage = get_sheet_or_empty(
    coverage_sheet
)

construct_statistics = get_sheet_or_empty(
    statistics_sheet
)


# ================================================================
# 23. WRITE FINAL EXCEL FILE
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"DIC_FINAL_EVIDENCE_AND_DIMENSIONS_{timestamp}.xlsx"
)


with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:


    original_responses.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )


    raw_themes.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )


    cleaned_themes.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )


    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )


    if participant_matrix is not None:

        participant_matrix.to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )


    theme_summary_existing.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )


    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )


    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )


    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )


    unmapped_check.to_excel(
        writer,
        sheet_name="10_Unmapped_Check",
        index=False
    )


    decision_check.to_excel(
        writer,
        sheet_name="11_Decision_Check",
        index=False
    )


    quality_checks_new.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )


    # ------------------------------------------------------------
    # MAIN DIC EVIDENCE
    # ------------------------------------------------------------

    final_dic_evidence.to_excel(
        writer,
        sheet_name="13_Final_DIC_Evidence",
        index=False
    )


    # ------------------------------------------------------------
    # DIC DIMENSION SUMMARY
    # ------------------------------------------------------------

    dimension_summary.to_excel(
        writer,
        sheet_name="14_DIC_Dimension_Summary",
        index=False
    )


    # ------------------------------------------------------------
    # DIC DIMENSION × THEME
    # ------------------------------------------------------------

    dimension_themes.to_excel(
        writer,
        sheet_name="15_DIC_Dimension_Themes",
        index=False
    )


    # ------------------------------------------------------------
    # ORIGINAL CONSTRUCT STATISTICS
    # ------------------------------------------------------------

    construct_statistics.to_excel(
        writer,
        sheet_name="16_Construct_Statistics",
        index=False
    )


# ================================================================
# 24. PRINT RESULTS
# ================================================================

print("\n")
print("=" * 80)
print("DIC PROCESS COMPLETED SUCCESSFULLY")
print("=" * 80)


print("\nOutput file:")

print(
    OUTPUT_FILE.resolve()
)


print("\n")
print("-" * 80)
print("FINAL DIC EVIDENCE")
print("-" * 80)


print(
    final_dic_evidence.to_string(
        index=False
    )
)


print("\n")
print("-" * 80)
print("DIC DIMENSION SUMMARY")
print("-" * 80)


print(
    dimension_summary.to_string(
        index=False
    )
)


print("\n")
print("=" * 80)
print("OUTPUT SHEETS")
print("=" * 80)

print("""
01_Original_Responses
02_Raw_Themes
03_Cleaned_Themes
04_Coding_Audit
05_Participant_Matrix
06_Theme_Summary
07_Normalization_Summary
08_Decision_Summary
09_Participant_Coverage
10_Unmapped_Check
11_Decision_Check
12_Quality_Checks
13_Final_DIC_Evidence
14_DIC_Dimension_Summary
15_DIC_Dimension_Themes
16_Construct_Statistics
""")


print("\n")
print("=" * 80)
print("NEXT STAGE")
print("=" * 80)

print("""
The two most important sheets for questionnaire development are:

14_DIC_Dimension_Summary
15_DIC_Dimension_Themes

These provide the bridge from expert qualitative evidence
to candidate DIC questionnaire items.

DO NOT treat these dimensions as statistically validated
subdimensions yet. They are evidence-based qualitative
groupings that will be used to formulate candidate items.
""")

DIC FINAL QUALITATIVE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Decision_Intelligence_Risk_Coordination_Supply_Chain_Resilience_1\DIC_FINAL_QUALITATIVE_ANALYSIS_20260827_131057.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Audit
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Normalization_Summary
 - 08_Decision_Summary
 - 09_Participant_Coverage
 - 10_Construct_Statistics
 - 11_Final_DIC_Evidence
 - 12_Quality_Checks
 - 13_Unmapped_Themes

Detected final evidence sheet:
11_Final_DIC_Evidence

Final DIC evidence columns:
['Final_DIC_Theme', 'Number_of_Raw_Themes', 'Experts_Mentioning', 'Expert_Prevalence_%', 'Raw_Themes_Kept', 'Raw_Themes_Merged', 'Prevalence_Category']

Number of participants: 26


DIC PROCESS COMPLETED SUCCESSFULLY

Output file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Decision_Intelligence_Risk_Coordinati

## Risk Intelligence Capability

In [4]:
# ================================================================
# RIC CODING
# INPUT FILE: Themes.xlsx
# ================================================================

import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# ------------------------------------------------
# 1. INPUT
# ------------------------------------------------

input_file = Path("Themes.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"Themes.xlsx was not found in:\n{Path.cwd()}"
    )

excel = pd.ExcelFile(input_file)


# ------------------------------------------------
# 2. FIND PARTICIPANT SHEETS
# ------------------------------------------------

participant_sheets = []

for sheet in excel.sheet_names:

    match = re.fullmatch(
        r"P?\s*0*(\d+)",
        str(sheet).strip(),
        flags=re.IGNORECASE
    )

    if match:

        number = int(match.group(1))

        if 1 <= number <= 26:
            participant_sheets.append(
                (number, sheet)
            )

participant_sheets = sorted(
    participant_sheets,
    key=lambda x: x[0]
)

print("Participant sheets found:")

for number, sheet in participant_sheets:
    print(f"P{number:02d} -> {sheet}")


# ------------------------------------------------
# 3. EXTRACT RIC RESPONSES
# ------------------------------------------------

raw_rows = []

for number, sheet in participant_sheets:

    participant = f"P{number:02d}"

    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        header=None
    )

    for row_number, row in df.iterrows():

        values = []

        for value in row.tolist():

            if pd.isna(value):
                continue

            text = str(value).strip()

            if text != "":
                values.append(text)

        if len(values) < 2:
            continue

        construct_position = None

        for position, value in enumerate(values):

            if value.strip().upper() == "RIC":

                construct_position = position
                break

        if construct_position is None:
            continue

        remaining = values[
            construct_position + 1:
        ]

        if not remaining:
            continue

        response = " ".join(
            remaining
        ).strip()

        if response:

            raw_rows.append({

                "Participant":
                    participant,

                "Construct":
                    "RIC",

                "Original_Response":
                    response,

                "Source_Sheet":
                    sheet,

                "Source_Row":
                    row_number + 1
            })


# ------------------------------------------------
# 4. CHECK
# ------------------------------------------------

original_df = pd.DataFrame(
    raw_rows
)

if original_df.empty:

    raise ValueError(
        """
        No RIC responses were found.

        Check that RIC appears in the participant sheets.
        """
    )

print("\nRIC original responses:")
print(len(original_df))


# ------------------------------------------------
# 5. SPLIT INTO RAW THEMES
# ------------------------------------------------

theme_rows = []

for _, row in original_df.iterrows():

    response = str(
        row["Original_Response"]
    )

    response = response.replace(
        "\n", ","
    )

    response = response.replace(
        ";", ","
    )

    response = response.replace(
        "•", ","
    )

    themes = response.split(",")

    for theme in themes:

        theme = str(
            theme
        ).strip()

        if theme == "":
            continue

        theme_rows.append({

            "Participant":
                row["Participant"],

            "Construct":
                "RIC",

            "Raw_Theme":
                theme,

            "Source_Sheet":
                row["Source_Sheet"],

            "Source_Row":
                row["Source_Row"]
        })


raw_df = pd.DataFrame(
    theme_rows
)


# ------------------------------------------------
# 6. CLEAN THEMES
# ------------------------------------------------

clean_df = raw_df.copy()

clean_df["Theme_Clean"] = (
    clean_df["Raw_Theme"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

clean_df["Theme_Key"] = (
    clean_df["Theme_Clean"]
    .str.lower()
    .str.strip()
)

clean_df = clean_df[
    clean_df["Theme_Key"] != ""
]

clean_df = clean_df.drop_duplicates(
    subset=[
        "Participant",
        "Construct",
        "Theme_Key"
    ]
)

clean_df = clean_df.reset_index(
    drop=True
)

print("\nRIC cleaned theme observations:")
print(len(clean_df))


# ------------------------------------------------
# 7. DISPLAY RAW RIC THEMES
# ------------------------------------------------

raw_theme_list = (
    clean_df[
        [
            "Raw_Theme",
            "Theme_Clean",
            "Theme_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Theme_Key"
    )
    .reset_index(
        drop=True
    )
)

print("\n")
print("=" * 60)
print("RIC RAW THEMES")
print("=" * 60)

print(
    raw_theme_list.to_string(
        index=False
    )
)


# ------------------------------------------------
# 8. RIC NORMALIZATION DICTIONARY
# ------------------------------------------------
#
# Add researcher-approved RIC mappings here after reviewing
# the actual RIC raw themes.
#
# Example format:
#
# "risk identification":
#     "Risk identification",
#
# "identifying emerging risks":
#     "Risk identification",
#
# ------------------------------------------------

RIC_NORMALIZATION = {

    # ADD RIC MAPPINGS HERE

}


# ------------------------------------------------
# 9. APPLY NORMALIZATION
# ------------------------------------------------

coding_df = raw_theme_list.copy()

coding_df["Normalized_Theme"] = ""

coding_df["Decision"] = ""

coding_df["Reason"] = ""


for i, row in coding_df.iterrows():

    theme_key = str(
        row["Theme_Key"]
    ).strip().lower()

    if theme_key in RIC_NORMALIZATION:

        normalized = RIC_NORMALIZATION[
            theme_key
        ]

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = normalized

        if theme_key == normalized.lower():

            coding_df.loc[
                i,
                "Decision"
            ] = "Keep"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Retained as a distinct conceptual theme."
            )

        else:

            coding_df.loc[
                i,
                "Decision"
            ] = "Merge"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Merged with a semantically equivalent "
                "theme."
            )

    else:

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = row["Theme_Clean"]

        coding_df.loc[
            i,
            "Decision"
        ] = "Keep"

        coding_df.loc[
            i,
            "Reason"
        ] = (
            "Retained pending conceptual review."
        )


# ------------------------------------------------
# 10. MAP TO PARTICIPANTS
# ------------------------------------------------

mapping = coding_df[
    [
        "Theme_Key",
        "Normalized_Theme",
        "Decision"
    ]
]

coded_df = clean_df.merge(
    mapping,
    on="Theme_Key",
    how="left"
)


# ------------------------------------------------
# 11. PARTICIPANT × RIC THEME MATRIX
# ------------------------------------------------

participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

matrix_source = (
    coded_df
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
)

matrix = pd.crosstab(
    matrix_source["Normalized_Theme"],
    matrix_source["Participant"]
)

matrix = matrix.reindex(
    columns=participants,
    fill_value=0
)

matrix = matrix.reset_index()

matrix["Frequency"] = matrix[
    participants
].sum(axis=1)

matrix["Percentage"] = (
    matrix["Frequency"]
    / len(participants)
    * 100
).round(1)


# ------------------------------------------------
# 12. THEME SUMMARY
# ------------------------------------------------

theme_summary = matrix[
    [
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].sort_values(
    "Frequency",
    ascending=False
)


# ------------------------------------------------
# 13. DECISION SUMMARY
# ------------------------------------------------

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Count"
    )
)


# ------------------------------------------------
# 14. PARTICIPANT COVERAGE
# ------------------------------------------------

participant_coverage = (
    coded_df
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "RIC_Theme_Count"
]


# ------------------------------------------------
# 15. SAVE EXCEL
# ------------------------------------------------

output_file = Path(
    "RIC_Coding_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".xlsx"
)

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_df.to_excel(
        writer,
        sheet_name="04_Coding_Dictionary",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="07_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="08_Participant_Coverage",
        index=False
    )


# ------------------------------------------------
# 16. FINAL REPORT
# ------------------------------------------------

print("\n")
print("=" * 60)
print("RIC CODING COMPLETED")
print("=" * 60)

print(
    "Participants:",
    clean_df["Participant"].nunique()
)

print(
    "Raw theme observations:",
    len(clean_df)
)

print(
    "Unique RIC raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Normalized RIC themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print("\nDECISION SUMMARY:")

print(
    decision_summary.to_string(
        index=False
    )
)

print("\nOUTPUT FILE:")

print(
    output_file.resolve()
)

print("\nDONE.")

Participant sheets found:
P01 -> 1
P02 -> 2
P03 -> 3
P04 -> 4
P05 -> 5
P06 -> 6
P07 -> 7
P08 -> 8
P09 -> 9
P10 -> 10
P11 -> 11
P12 -> 12
P13 -> 13
P14 -> 14
P15 -> 15
P16 -> 16
P17 -> 17
P18 -> 18
P19 -> 19
P20 -> 20
P21 -> 21
P22 -> 22
P23 -> 23
P24 -> 24
P25 -> 25
P26 -> 26

RIC original responses:
26

RIC cleaned theme observations:
98


RIC RAW THEMES
                           Raw_Theme                          Theme_Clean                            Theme_Key
                        alternatives                         alternatives                         alternatives
                        anticipation                         anticipation                         anticipation
             Business interpretation              Business interpretation              business interpretation
                  causal connections                   causal connections                   causal connections
                 combined indicators                  combined indicators              

In [5]:
# ================================================================
# COMPLETE RIC QUALITATIVE CODING ANALYSIS
# Input: RIC_Coding_20260821_092922.xlsx
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# ================================================================
# 1. INPUT FILE
# ================================================================

input_file = Path("RIC_Coding_20260827_131335.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"\nFile not found:\n{input_file.resolve()}\n\n"
        "Make sure the Excel file is in the same folder as your "
        "Python notebook."
    )

print("=" * 70)
print("RIC QUALITATIVE CODING ANALYSIS")
print("=" * 70)

print("\nInput file:")
print(input_file.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

excel = pd.ExcelFile(input_file)

print("\nSheets found:")
for s in excel.sheet_names:
    print(" -", s)


# ================================================================
# 3. FLEXIBLE SHEET FINDER
# ================================================================

def find_sheet(possible_names):

    for name in possible_names:
        if name in excel.sheet_names:
            return name

    for sheet in excel.sheet_names:

        a = (
            str(sheet)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for name in possible_names:

            b = (
                str(name)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if b in a:
                return sheet

    return None


original_sheet = find_sheet([
    "01_Original_RIC",
    "01_Original",
    "Original_RIC",
    "Original"
])

raw_sheet = find_sheet([
    "02_Raw_RIC",
    "02_Raw",
    "Raw_RIC",
    "Raw"
])

clean_sheet = find_sheet([
    "03_Cleaned_RIC",
    "03_Cleaned",
    "Cleaned_RIC",
    "Cleaned"
])

coding_sheet = find_sheet([
    "04_Normalized_Coding",
    "04_Coding",
    "Normalized_Coding",
    "Coding"
])


print("\nSelected sheets:")
print("Original:", original_sheet)
print("Raw:", raw_sheet)
print("Cleaned:", clean_sheet)
print("Coding:", coding_sheet)


# ================================================================
# 4. READ SHEETS
# ================================================================

original_df = (
    pd.read_excel(
        input_file,
        sheet_name=original_sheet
    )
    if original_sheet
    else pd.DataFrame()
)

raw_df = (
    pd.read_excel(
        input_file,
        sheet_name=raw_sheet
    )
    if raw_sheet
    else pd.DataFrame()
)

clean_df = (
    pd.read_excel(
        input_file,
        sheet_name=clean_sheet
    )
    if clean_sheet
    else pd.DataFrame()
)

if coding_sheet is None:
    raise ValueError(
        "\nCould not find the RIC Coding/Normalized Coding sheet.\n"
        "Available sheets are:\n"
        + "\n".join(
            str(x)
            for x in excel.sheet_names
        )
    )

coding_df = pd.read_excel(
    input_file,
    sheet_name=coding_sheet
)


# ================================================================
# 5. STANDARDIZE COLUMN NAMES
# ================================================================

for df in [
    original_df,
    raw_df,
    clean_df,
    coding_df
]:

    if not df.empty:
        df.columns = [
            str(c).strip()
            for c in df.columns
        ]


print("\nCoding columns:")
print(list(coding_df.columns))


# ================================================================
# 6. COLUMN FINDER
# ================================================================

def find_column(df, candidates):

    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    for column in df.columns:

        a = (
            str(column)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for candidate in candidates:

            b = (
                str(candidate)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if a == b:
                return column

    return None


theme_key_col = find_column(
    coding_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

raw_theme_col = find_column(
    coding_df,
    [
        "Raw_Theme",
        "Raw Theme",
        "RawTheme"
    ]
)

normalized_col = find_column(
    coding_df,
    [
        "Normalized_Theme",
        "Normalized Theme",
        "NormalizedTheme"
    ]
)

decision_col = find_column(
    coding_df,
    [
        "Decision"
    ]
)


if theme_key_col is None:
    raise ValueError(
        "\nTheme_Key column not found."
    )

if raw_theme_col is None:
    raise ValueError(
        "\nRaw_Theme column not found."
    )

if normalized_col is None:
    raise ValueError(
        "\nNormalized_Theme column not found."
    )

if decision_col is None:
    raise ValueError(
        "\nDecision column not found."
    )


coding_df = coding_df.rename(
    columns={
        theme_key_col: "Theme_Key",
        raw_theme_col: "Raw_Theme",
        normalized_col: "Normalized_Theme",
        decision_col: "Decision"
    }
)


# ================================================================
# 7. FIND PARTICIPANT COLUMN
# ================================================================

participant_col = find_column(
    clean_df,
    [
        "Participant",
        "Participant_ID",
        "Participant ID",
        "ParticipantID"
    ]
)

if participant_col is None:

    raise ValueError(
        "\nParticipant column not found in Cleaned RIC sheet.\n\n"
        "Available columns:\n"
        + str(list(clean_df.columns))
    )


if participant_col != "Participant":

    clean_df = clean_df.rename(
        columns={
            participant_col:
            "Participant"
        }
    )


# ================================================================
# 8. FIND THEME KEY IN CLEANED DATA
# ================================================================

clean_theme_key = find_column(
    clean_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

if clean_theme_key is None:

    raise ValueError(
        "\nTheme_Key not found in Cleaned RIC sheet.\n\n"
        "Available columns:\n"
        + str(list(clean_df.columns))
    )


if clean_theme_key != "Theme_Key":

    clean_df = clean_df.rename(
        columns={
            clean_theme_key:
            "Theme_Key"
        }
    )


# ================================================================
# 9. CLEAN CODING DICTIONARY
# ================================================================

coding_df["Theme_Key"] = (
    coding_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

coding_df["Raw_Theme"] = (
    coding_df["Raw_Theme"]
    .fillna("")
    .astype(str)
    .str.strip()
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .replace(
        [
            "nan",
            "None",
            "",
            "NaN"
        ],
        np.nan
    )
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .astype("string")
    .str.strip()
)

coding_df["Decision"] = (
    coding_df["Decision"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 10. CLEAN PARTICIPANT DATA
# ================================================================

clean_df["Theme_Key"] = (
    clean_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

clean_df["Participant"] = (
    clean_df["Participant"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 11. PARTICIPANT LIST
# ================================================================

participants = sorted(
    clean_df[
        "Participant"
    ]
    .replace("", np.nan)
    .dropna()
    .unique()
)

print("\nParticipants:")
print(participants)

print(
    "\nNumber of participants:",
    len(participants)
)


# ================================================================
# 12. MERGE CLEANED DATA WITH CODING DICTIONARY
# ================================================================

coded_df = clean_df.merge(
    coding_df[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision"
        ]
    ],
    on="Theme_Key",
    how="left"
)


# ================================================================
# 13. CHECK UNMAPPED THEMES
# ================================================================

missing_mapping = coded_df[
    coded_df[
        "Normalized_Theme"
    ].isna()
].copy()


print(
    "\nNumber of unmapped themes:",
    len(missing_mapping)
)


if len(missing_mapping) > 0:

    print("\nUnmapped themes:")

    print(
        missing_mapping[
            [
                "Participant",
                "Theme_Key"
            ]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )


# ================================================================
# 14. PARTICIPANT × NORMALIZED RIC THEME
# ================================================================

participant_theme = (
    coded_df[
        [
            "Participant",
            "Normalized_Theme"
        ]
    ]
    .dropna()
    .drop_duplicates()
)


# ================================================================
# 15. PARTICIPANT × THEME MATRIX
# ================================================================

matrix = pd.crosstab(
    participant_theme[
        "Normalized_Theme"
    ],
    participant_theme[
        "Participant"
    ]
)


# ================================================================
# 16. ORDER PARTICIPANTS P01, P02, ...
# ================================================================

expected_participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

existing = [
    p
    for p in expected_participants
    if p in matrix.columns
]

other = [
    p
    for p in matrix.columns
    if p not in existing
]

matrix = matrix.reindex(
    columns=existing + other,
    fill_value=0
)

matrix = matrix.reset_index()


participant_columns = (
    existing + other
)


# ================================================================
# 17. FREQUENCY
# ================================================================

matrix["Frequency"] = (
    matrix[
        participant_columns
    ]
    .sum(axis=1)
)


# ================================================================
# 18. PERCENTAGE OF PARTICIPANTS
# ================================================================

total_participants = len(participants)

if total_participants > 0:

    matrix["Percentage"] = (
        matrix["Frequency"]
        / total_participants
        * 100
    ).round(1)

else:

    matrix["Percentage"] = 0


# ================================================================
# 19. RANK THEMES
# ================================================================

matrix = matrix.sort_values(
    [
        "Frequency",
        "Normalized_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)


matrix.insert(
    0,
    "Rank",
    range(
        1,
        len(matrix) + 1
    )
)


# ================================================================
# 20. RIC THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Rank",
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].copy()


theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme":
            "Final_RIC_Theme",

        "Frequency":
            "Experts_Mentioning",

        "Percentage":
            "Percentage_of_Experts"
    }
)


# ================================================================
# 21. PREVALENCE CATEGORY
# ================================================================

def prevalence_category(p):

    if p >= 75:
        return "Very High"

    elif p >= 50:
        return "High"

    elif p >= 25:
        return "Moderate"

    else:
        return "Low"


theme_summary[
    "Prevalence_Category"
] = (
    theme_summary[
        "Percentage_of_Experts"
    ]
    .apply(
        prevalence_category
    )
)


# ================================================================
# 22. CODING AUDIT
# ================================================================

theme_participant_counts = (
    participant_theme
    .groupby(
        "Normalized_Theme"
    )
    .size()
    .reset_index(
        name="Experts_Mentioning"
    )
)


coding_audit = coding_df.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)


coding_audit[
    "Experts_Mentioning"
] = (
    coding_audit[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


if total_participants > 0:

    coding_audit[
        "Percentage_of_Experts"
    ] = (
        coding_audit[
            "Experts_Mentioning"
        ]
        / total_participants
        * 100
    ).round(1)

else:

    coding_audit[
        "Percentage_of_Experts"
    ] = 0


coding_audit = coding_audit.sort_values(
    [
        "Normalized_Theme",
        "Decision",
        "Raw_Theme"
    ]
).reset_index(drop=True)


# ================================================================
# 23. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby(
        "Decision"
    )
    .size()
    .reset_index(
        name="Number_of_Raw_Themes"
    )
)


if len(coding_df) > 0:

    decision_summary[
        "Percentage"
    ] = (
        decision_summary[
            "Number_of_Raw_Themes"
        ]
        / len(coding_df)
        * 100
    ).round(1)

else:

    decision_summary[
        "Percentage"
    ] = 0


# ================================================================
# 24. NORMALIZATION SUMMARY
# ================================================================

normalization_summary = (
    coding_df
    .dropna(
        subset=[
            "Normalized_Theme"
        ]
    )
    .groupby(
        "Normalized_Theme"
    )
    .agg(
        Raw_Themes=(
            "Raw_Theme",
            "count"
        ),

        Keep_Count=(
            "Decision",
            lambda x:
            (x == "Keep").sum()
        ),

        Merge_Count=(
            "Decision",
            lambda x:
            (x == "Merge").sum()
        )
    )
    .reset_index()
)


normalization_summary = (
    normalization_summary.merge(
        theme_participant_counts,
        on="Normalized_Theme",
        how="left"
    )
)


normalization_summary[
    "Experts_Mentioning"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


if total_participants > 0:

    normalization_summary[
        "Percentage_of_Experts"
    ] = (
        normalization_summary[
            "Experts_Mentioning"
        ]
        / total_participants
        * 100
    ).round(1)

else:

    normalization_summary[
        "Percentage_of_Experts"
    ] = 0


normalization_summary = (
    normalization_summary
    .sort_values(
        "Experts_Mentioning",
        ascending=False
    )
    .reset_index(drop=True)
)


# ================================================================
# 25. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    participant_theme
    .groupby(
        "Participant"
    )
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)


participant_coverage.columns = [
    "Participant",
    "Number_of_Normalized_RIC_Themes"
]


# ================================================================
# 26. RIC CONSTRUCT STATISTICS
# ================================================================

construct_statistics = pd.DataFrame({

    "Construct": [
        "RIC"
    ],

    "Participants": [
        total_participants
    ],

    "Original_Response_Rows": [
        len(original_df)
    ],

    "Raw_Theme_Observations": [
        len(raw_df)
    ],

    "Cleaned_Theme_Observations": [
        len(clean_df)
    ],

    "Unique_Raw_Themes": [
        coding_df[
            "Theme_Key"
        ].nunique()
    ],

    "Final_Normalized_Themes": [
        coding_df[
            "Normalized_Theme"
        ].nunique()
    ],

    "Merged_Raw_Themes": [
        (
            coding_df[
                "Decision"
            ]
            == "Merge"
        ).sum()
    ],

    "Kept_Raw_Themes": [
        (
            coding_df[
                "Decision"
            ]
            == "Keep"
        ).sum()
    ],

    "Unmapped_Themes": [
        len(missing_mapping)
    ]
})


# ================================================================
# 27. FINAL RIC EVIDENCE
# ================================================================

final_evidence = normalization_summary[
    [
        "Normalized_Theme",
        "Raw_Themes",
        "Experts_Mentioning",
        "Percentage_of_Experts",
        "Keep_Count",
        "Merge_Count"
    ]
].copy()


final_evidence = final_evidence.rename(
    columns={
        "Normalized_Theme":
            "Final_RIC_Theme",

        "Raw_Themes":
            "Number_of_Raw_Themes",

        "Percentage_of_Experts":
            "Expert_Prevalence_%",

        "Keep_Count":
            "Raw_Themes_Kept",

        "Merge_Count":
            "Raw_Themes_Merged"
    }
)


final_evidence[
    "Prevalence_Category"
] = (
    final_evidence[
        "Expert_Prevalence_%"
    ]
    .apply(
        prevalence_category
    )
)


# ================================================================
# 28. QUALITY CHECKS
# ================================================================

duplicate_count = (
    len(coded_df)
    -
    len(
        coded_df[
            [
                "Participant",
                "Normalized_Theme"
            ]
        ]
        .dropna()
        .drop_duplicates()
    )
)


quality_checks = pd.DataFrame({

    "Check": [

        "Number of participants",

        "Original response rows",

        "Raw theme observations",

        "Unique raw themes",

        "Final normalized themes",

        "Unmapped themes",

        "Duplicate participant-theme records",

        "Keep decisions",

        "Merge decisions"

    ],

    "Result": [

        total_participants,

        len(original_df),

        len(raw_df),

        coding_df[
            "Theme_Key"
        ].nunique(),

        coding_df[
            "Normalized_Theme"
        ].nunique(),

        len(missing_mapping),

        duplicate_count,

        (
            coding_df[
                "Decision"
            ]
            == "Keep"
        ).sum(),

        (
            coding_df[
                "Decision"
            ]
            == "Merge"
        ).sum()

    ],

    "Status": [

        "PASS"
        if total_participants > 0
        else "CHECK",

        "PASS"
        if len(original_df) > 0
        else "CHECK",

        "PASS"
        if len(raw_df) > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Theme_Key"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Normalized_Theme"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if len(missing_mapping) == 0
        else "CHECK",

        "INFO",

        "PASS",

        "PASS"

    ]
})


# ================================================================
# 29. SAVE COMPLETE RIC WORKBOOK
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_file = Path(
    f"RIC_FINAL_QUALITATIVE_ANALYSIS_{timestamp}.xlsx"
)


# Avoid overwrite / PermissionError

counter = 1

while output_file.exists():

    output_file = Path(
        f"RIC_FINAL_QUALITATIVE_ANALYSIS_"
        f"{timestamp}_{counter}.xlsx"
    )

    counter += 1


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_RIC_Evidence",
        index=False
    )

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    missing_mapping.to_excel(
        writer,
        sheet_name="13_Unmapped_Themes",
        index=False
    )


# ================================================================
# 30. FINAL REPORT
# ================================================================

print("\n")
print("=" * 70)
print("RIC ANALYSIS COMPLETED")
print("=" * 70)

print(
    "\nParticipants:",
    total_participants
)

print(
    "Original responses:",
    len(original_df)
)

print(
    "Raw theme observations:",
    len(raw_df)
)

print(
    "Cleaned theme observations:",
    len(clean_df)
)

print(
    "Unique raw themes:",
    coding_df[
        "Theme_Key"
    ].nunique()
)

print(
    "Final normalized RIC themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print(
    "Keep decisions:",
    (
        coding_df[
            "Decision"
        ] == "Keep"
    ).sum()
)

print(
    "Merge decisions:",
    (
        coding_df[
            "Decision"
        ] == "Merge"
    ).sum()
)

print(
    "Unmapped themes:",
    len(missing_mapping)
)


# ================================================================
# 31. DISPLAY FINAL RIC THEMES
# ================================================================

print("\n")
print("=" * 70)
print("FINAL RIC THEMES")
print("=" * 70)

print(
    final_evidence[
        [
            "Final_RIC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Prevalence_Category"
        ]
    ].to_string(index=False)
)


# ================================================================
# 32. OUTPUT FILE
# ================================================================

print("\n")
print("=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(
    output_file.resolve()
)

print(
    "\nComplete RIC qualitative analysis workbook "
    "created successfully."
)

RIC QUALITATIVE CODING ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Decision_Intelligence_Risk_Coordination_Supply_Chain_Resilience_1\RIC_Coding_20260827_131335.xlsx

Sheets found:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Dictionary
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Decision_Summary
 - 08_Participant_Coverage

Selected sheets:
Original: 01_Original_Responses
Raw: 02_Raw_Themes
Cleaned: 03_Cleaned_Themes
Coding: 04_Coding_Dictionary

Coding columns:
['Raw_Theme', 'Theme_Clean', 'Theme_Key', 'Normalized_Theme', 'Decision', 'Reason']

Participants:
['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26']

Number of participants: 26

Number of unmapped themes: 0


RIC ANALYSIS COMPLETED

Participants: 26
Original responses: 26
Raw theme observations: 98
C

In [ ]:
# ================================================================
# RIC — FINAL QUALITATIVE EVIDENCE + DIMENSION ANALYSIS
# ================================================================
#
# INPUT:
# RIC_FINAL_QUALITATIVE_ANALYSIS_20260827_131423.xlsx
#
# OUTPUT:
# RIC_FINAL_EVIDENCE_AND_DIMENSIONS_20260827_131622.xlsx
#
# PURPOSE:
# 1. Preserve the completed RIC qualitative coding
# 2. Extract final RIC evidence
# 3. Organize RIC themes into evidence-based dimensions
# 4. Calculate expert prevalence
# 5. Identify themes requiring review
# 6. Produce dimension × theme evidence tables
#
# IMPORTANT:
# The dimensions are qualitative groupings for questionnaire
# development. They are NOT statistically validated dimensions.
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import os


# ================================================================
# 1. FIND INPUT FILE
# ================================================================

TARGET = "RIC_FINAL_QUALITATIVE_ANALYSIS_20260827_131423"

possible_files = []

search_locations = [
    Path("."),
    Path.home(),
    Path.home() / "Downloads",
    Path.home() / "Documents",
    Path.home() / "Desktop",
]

for location in search_locations:

    if location.exists():

        for ext in [".xlsx", ".xlsm", ".xls"]:

            try:
                possible_files.extend(
                    location.rglob(TARGET + ext)
                )
            except Exception:
                pass


# Remove duplicates
possible_files = list(
    dict.fromkeys(
        [p.resolve() for p in possible_files]
    )
)


if len(possible_files) == 0:

    raise FileNotFoundError(
        "\nRIC input file was not found.\n\n"
        f"Expected:\n{TARGET}.xlsx\n\n"
        "Put the Excel file in the same folder as your "
        "Python notebook or change TARGET to the exact "
        "filename."
    )


INPUT_FILE = possible_files[0]


print("=" * 80)
print("RIC FINAL QUALITATIVE ANALYSIS")
print("=" * 80)

print("\nInput file:")
print(INPUT_FILE)


# ================================================================
# 2. READ WORKBOOK
# ================================================================

xls = pd.ExcelFile(
    INPUT_FILE,
    engine="openpyxl"
)

print("\nAvailable sheets:")

for s in xls.sheet_names:
    print(" -", s)


# ================================================================
# 3. LOAD ALL SHEETS
# ================================================================

sheets = {}

for sheet in xls.sheet_names:

    try:

        sheets[sheet] = pd.read_excel(
            INPUT_FILE,
            sheet_name=sheet
        )

    except Exception as e:

        print(
            f"Warning: could not read {sheet}: {e}"
        )


# ================================================================
# 4. HELPER FUNCTIONS
# ================================================================

def find_sheet(keyword):

    for s in sheets.keys():

        if keyword.lower() in s.lower():

            return s

    return None


def clean_text(x):

    if pd.isna(x):

        return ""

    x = str(x).strip()

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x


def prevalence_category(x):

    try:

        x = float(x)

    except:

        return "Not available"

    if x >= 75:

        return "Very High"

    elif x >= 50:

        return "High"

    elif x >= 25:

        return "Moderate"

    else:

        return "Low"


# ================================================================
# 5. IDENTIFY SHEETS
# ================================================================

original_sheet = find_sheet(
    "01_Original_Responses"
)

raw_sheet = find_sheet(
    "02_Raw_Themes"
)

cleaned_sheet = find_sheet(
    "03_Cleaned_Themes"
)

audit_sheet = find_sheet(
    "04_Coding_Audit"
)

matrix_sheet = find_sheet(
    "05_Participant_Matrix"
)

theme_sheet = find_sheet(
    "06_Theme_Summary"
)

normalization_sheet = find_sheet(
    "07_Normalization_Summary"
)

decision_sheet = find_sheet(
    "08_Decision_Summary"
)

coverage_sheet = find_sheet(
    "09_Participant_Coverage"
)

statistics_sheet = find_sheet(
    "10_Construct_Statistics"
)

final_sheet = find_sheet(
    "11_Final_RIC_Evidence"
)

quality_sheet = find_sheet(
    "12_Quality_Checks"
)

unmapped_sheet = find_sheet(
    "13_Unmapped_Themes"
)


if final_sheet is None:

    raise ValueError(
        "11_Final_RIC_Evidence was not found."
    )


print(
    "\nFinal RIC evidence sheet:",
    final_sheet
)


# ================================================================
# 6. READ FINAL RIC EVIDENCE
# ================================================================

final_evidence = sheets[
    final_sheet
].copy()

final_evidence.columns = [
    str(c).strip()
    for c in final_evidence.columns
]


# ================================================================
# 7. IDENTIFY RIC THEME COLUMN
# ================================================================

theme_col = None


if "Final_RIC_Theme" in final_evidence.columns:

    theme_col = "Final_RIC_Theme"


else:

    for c in final_evidence.columns:

        if (
            "RIC" in str(c)
            and "Theme" in str(c)
        ):

            theme_col = c
            break


if theme_col is None:

    for c in final_evidence.columns:

        if "Theme" in str(c):

            theme_col = c
            break


if theme_col is None:

    raise ValueError(
        "Could not identify the Final RIC Theme column."
    )


final_evidence[
    "Final_RIC_Theme"
] = (
    final_evidence[
        theme_col
    ]
    .apply(clean_text)
)


# ================================================================
# 8. STANDARDIZE IMPORTANT COLUMNS
# ================================================================

if (
    "Experts_Mentioning"
    not in final_evidence.columns
):

    for c in final_evidence.columns:

        if (
            "Experts" in str(c)
            and "Mention" in str(c)
        ):

            final_evidence[
                "Experts_Mentioning"
            ] = final_evidence[c]

            break


if (
    "Expert_Prevalence_%"
    not in final_evidence.columns
):

    for c in final_evidence.columns:

        if (
            "Prevalence" in str(c)
            and "%" in str(c)
        ):

            final_evidence[
                "Expert_Prevalence_%"
            ] = final_evidence[c]

            break


# ================================================================
# 9. REMOVE EMPTY THEMES
# ================================================================

final_evidence = final_evidence[
    final_evidence[
        "Final_RIC_Theme"
    ] != ""
].copy()


# ================================================================
# 10. PARTICIPANT COUNT
# ================================================================

participant_matrix = None

if matrix_sheet is not None:

    participant_matrix = sheets[
        matrix_sheet
    ].copy()


participants = []


if participant_matrix is not None:

    for c in participant_matrix.columns:

        cstr = str(c).strip()

        if re.match(
            r"^P\d+$",
            cstr,
            flags=re.IGNORECASE
        ):

            participants.append(
                cstr
            )


if len(participants) > 0:

    n_participants = len(
        participants
    )

else:

    # Study participant count
    n_participants = 26


print(
    "\nParticipants used:",
    n_participants
)


# ================================================================
# 11. PREVALENCE CATEGORY
# ================================================================

if (
    "Expert_Prevalence_%"
    in final_evidence.columns
):

    final_evidence[
        "Prevalence_Category"
    ] = (
        final_evidence[
            "Expert_Prevalence_%"
        ]
        .apply(
            prevalence_category
        )
    )


# ================================================================
# 12. SORT FINAL EVIDENCE
# ================================================================

if (
    "Experts_Mentioning"
    in final_evidence.columns
):

    final_evidence = (
        final_evidence
        .sort_values(
            "Experts_Mentioning",
            ascending=False
        )
        .reset_index(drop=True)
    )


# ================================================================
# 13. FINAL RIC EVIDENCE TABLE
# ================================================================

preferred_columns = [

    "Final_RIC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Raw_Themes_Kept",

    "Raw_Themes_Merged",

    "Prevalence_Category"

]


evidence_columns = [

    c
    for c in preferred_columns
    if c in final_evidence.columns

]


final_ric_evidence = final_evidence[
    evidence_columns
].copy()


# Remove existing Rank if present
if "Rank" in final_ric_evidence.columns:

    final_ric_evidence = (
        final_ric_evidence
        .drop(columns=["Rank"])
    )


final_ric_evidence.insert(
    0,
    "Rank",
    range(
        1,
        len(final_ric_evidence) + 1
    )
)


# ================================================================
# 14. RIC DIMENSION MAPPING
# ================================================================
#
# These dimensions are derived from the 61 RIC themes present
# in the attached workbook.
#
# ================================================================


def map_ric_dimension(theme):

    t = str(theme).strip().lower()


    # ------------------------------------------------------------
    # 1. RISK DETECTION & EARLY IDENTIFICATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "early detection",
            "early identification",
            "warning signs",
            "external warnings",
            "connecting signals",
            "signal combination",
            "signal connection",
            "monitoring"

        ]
    ):

        return (
            "Risk Detection & Early Identification"
        )


    # ------------------------------------------------------------
    # 2. RISK EXPOSURE & VULNERABILITY ASSESSMENT
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "exposure",
            "vulnerability",
            "overall exposure",
            "systemic exposure",
            "internal exposure",
            "exposure assessment",
            "contextual exposure",
            "inventory vulnerability",
            "internal vulnerabilities",
            "likelihood",
            "probability",
            "severity",
            "significance"

        ]
    ):

        return (
            "Risk Exposure & Vulnerability Assessment"
        )


    # ------------------------------------------------------------
    # 3. CONTEXTUAL RISK INTERPRETATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "contextual assessment",
            "contextual interpretation",
            "contextual risk interpretation",
            "exposure interpretation",
            "business interpretation",
            "event interpretation",
            "interpretation",
            "conditions",
            "historical patterns",
            "external developments",
            "risk development"

        ]
    ):

        return (
            "Contextual Risk Interpretation"
        )


    # ------------------------------------------------------------
    # 4. RISK INTERCONNECTEDNESS & SYSTEMIC UNDERSTANDING
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "dependencies",
            "dependency",
            "interactions",
            "relationships",
            "knock-on effects",
            "event interactions",
            "causal connections",
            "interconnected",
            "reinforcing risks",
            "combined risks",
            "exposure combinations",
            "combined indicators"

        ]
    ):

        return (
            "Risk Interconnectedness & Systemic Understanding"
        )


    # ------------------------------------------------------------
    # 5. RISK AGGREGATION & PRIORITIZATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "combined-risk assessment",
            "risk aggregation",
            "prioritization",
            "ranking",
            "resource focus",
            "alternatives",
            "multiple consequences",
            "consequences",
            "impact"

        ]
    ):

        return (
            "Risk Aggregation & Prioritization"
        )


    # ------------------------------------------------------------
    # 6. RISK INFORMATION INTEGRATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "integrated assessment",
            "inventory",
            "critical materials",
            "external developments",
            "internal vulnerabilities",
            "combined indicators"

        ]
    ):

        return (
            "Risk Information Integration"
        )


    # ------------------------------------------------------------
    # 7. REVIEW REQUIRED
    # ------------------------------------------------------------

    return "Review Required"


final_ric_evidence[
    "RIC_Dimension"
] = (
    final_ric_evidence[
        "Final_RIC_Theme"
    ]
    .apply(
        map_ric_dimension
    )
)


# ================================================================
# 15. EXPLICIT MAPPING FOR AMBIGUOUS / DUPLICATE LABELS
# ================================================================

manual_mapping = {

    # ------------------------------------------------------------
    # Detection
    # ------------------------------------------------------------

    "early detection":
        "Risk Detection & Early Identification",

    "early identification":
        "Risk Detection & Early Identification",

    "warning signs":
        "Risk Detection & Early Identification",

    "External warnings":
        "Risk Detection & Early Identification",

    "Monitoring":
        "Risk Detection & Early Identification",

    "Connecting signals":
        "Risk Detection & Early Identification",

    "Signal combination":
        "Risk Detection & Early Identification",

    "Signal connection":
        "Risk Detection & Early Identification",


    # ------------------------------------------------------------
    # Exposure / vulnerability
    # ------------------------------------------------------------

    "Exposure":
        "Risk Exposure & Vulnerability Assessment",

    "exposure":
        "Risk Exposure & Vulnerability Assessment",

    "Exposure interpretation":
        "Contextual Risk Interpretation",

    "Contextual exposure":
        "Risk Exposure & Vulnerability Assessment",

    "Systemic exposure":
        "Risk Exposure & Vulnerability Assessment",

    "internal exposure":
        "Risk Exposure & Vulnerability Assessment",

    "exposure assessment":
        "Risk Exposure & Vulnerability Assessment",

    "inventory vulnerability":
        "Risk Exposure & Vulnerability Assessment",

    "internal vulnerabilities":
        "Risk Exposure & Vulnerability Assessment",

    "vulnerability":
        "Risk Exposure & Vulnerability Assessment",

    "vulnerabilities":
        "Risk Exposure & Vulnerability Assessment",

    "likelihood":
        "Risk Exposure & Vulnerability Assessment",

    "Likelihood":
        "Risk Exposure & Vulnerability Assessment",

    "Probability":
        "Risk Exposure & Vulnerability Assessment",

    "severity":
        "Risk Exposure & Vulnerability Assessment",

    "significance":
        "Risk Exposure & Vulnerability Assessment",

    "Overall exposure":
        "Risk Exposure & Vulnerability Assessment",


    # ------------------------------------------------------------
    # Contextual interpretation
    # ------------------------------------------------------------

    "Contextual assessment":
        "Contextual Risk Interpretation",

    "contextual assessment":
        "Contextual Risk Interpretation",

    "Contextual interpretation":
        "Contextual Risk Interpretation",

    "contextual interpretation":
        "Contextual Risk Interpretation",

    "Contextual risk interpretation":
        "Contextual Risk Interpretation",

    "Business interpretation":
        "Contextual Risk Interpretation",

    "Event interpretation":
        "Contextual Risk Interpretation",

    "interpretation":
        "Contextual Risk Interpretation",

    "conditions":
        "Contextual Risk Interpretation",

    "historical patterns":
        "Contextual Risk Interpretation",

    "Risk development":
        "Contextual Risk Interpretation",


    # ------------------------------------------------------------
    # Interconnectedness
    # ------------------------------------------------------------

    "dependencies":
        "Risk Interconnectedness & Systemic Understanding",

    "dependency":
        "Risk Interconnectedness & Systemic Understanding",

    "interactions":
        "Risk Interconnectedness & Systemic Understanding",

    "relationships":
        "Risk Interconnectedness & Systemic Understanding",

    "knock-on effects":
        "Risk Interconnectedness & Systemic Understanding",

    "Event interactions":
        "Risk Interconnectedness & Systemic Understanding",

    "causal connections":
        "Risk Interconnectedness & Systemic Understanding",

    "Interconnected and reinforcing risks":
        "Risk Interconnectedness & Systemic Understanding",

    "combined risks":
        "Risk Interconnectedness & Systemic Understanding",

    "exposure combinations":
        "Risk Interconnectedness & Systemic Understanding",

    "combined indicators":
        "Risk Interconnectedness & Systemic Understanding",


    # ------------------------------------------------------------
    # Aggregation / prioritization
    # ------------------------------------------------------------

    "prioritization":
        "Risk Aggregation & Prioritization",

    "ranking":
        "Risk Aggregation & Prioritization",

    "resource focus":
        "Risk Aggregation & Prioritization",

    "alternatives":
        "Risk Aggregation & Prioritization",

    "Consequences":
        "Risk Aggregation & Prioritization",

    "consequences":
        "Risk Aggregation & Prioritization",

    "impact":
        "Risk Aggregation & Prioritization",

    "multiple consequences":
        "Risk Aggregation & Prioritization",

    "risk aggregation":
        "Risk Aggregation & Prioritization",

    "combined-risk assessment":
        "Risk Aggregation & Prioritization",


    # ------------------------------------------------------------
    # Information integration
    # ------------------------------------------------------------

    "integrated assessment":
        "Risk Information Integration",

    "inventory":
        "Risk Information Integration",

    "critical materials":
        "Risk Information Integration",

}


for theme, dimension in manual_mapping.items():

    mask = (
        final_ric_evidence[
            "Final_RIC_Theme"
        ]
        .astype(str)
        .str.strip()
        .eq(theme)
    )

    final_ric_evidence.loc[
        mask,
        "RIC_Dimension"
    ] = dimension


# ================================================================
# 16. DIMENSION SUMMARY
# ================================================================

dimension_rows = []


for dimension, group in (
    final_ric_evidence
    .groupby(
        "RIC_Dimension",
        dropna=False
    )
):

    themes = (
        group[
            "Final_RIC_Theme"
        ]
        .astype(str)
        .tolist()
    )


    # ------------------------------------------------------------
    # Calculate expert prevalence
    # ------------------------------------------------------------

    experts_mentioning = 0


    if (
        participant_matrix is not None
        and len(participants) > 0
    ):

        temp = participant_matrix.copy()

        # Identify theme column
        theme_candidates = [
            c
            for c in temp.columns
            if "theme" in str(c).lower()
        ]

        if len(theme_candidates) > 0:

            matrix_theme_col = (
                theme_candidates[0]
            )

        else:

            matrix_theme_col = (
                temp.columns[0]
            )


        temp["_Theme"] = (
            temp[
                matrix_theme_col
            ]
            .astype(str)
            .str.strip()
        )


        theme_lower = {
            str(x).strip().lower()
            for x in themes
        }


        matched = temp[
            temp[
                "_Theme"
            ]
            .str.lower()
            .isin(
                theme_lower
            )
        ]


        available_participants = [

            p

            for p in participants

            if p in matched.columns

        ]


        if (
            len(matched) > 0
            and len(available_participants) > 0
        ):

            vals = (
                matched[
                    available_participants
                ]
                .apply(
                    pd.to_numeric,
                    errors="coerce"
                )
                .fillna(0)
            )


            experts_mentioning = int(
                (
                    vals.sum(axis=0) > 0
                ).sum()
            )


    # ------------------------------------------------------------
    # Fallback to final evidence
    # ------------------------------------------------------------

    if experts_mentioning == 0:

        if (
            "Experts_Mentioning"
            in group.columns
        ):

            experts_mentioning = int(
                group[
                    "Experts_Mentioning"
                ]
                .max()
            )


    prevalence = (

        experts_mentioning
        /
        n_participants
        *
        100

        if n_participants > 0
        else 0

    )


    dimension_rows.append({

        "RIC_Dimension":
            dimension,

        "Number_of_Normalized_Themes":
            len(themes),

        "Experts_Mentioning":
            experts_mentioning,

        "Expert_Prevalence_%":
            round(
                prevalence,
                1
            ),

        "Prevalence_Category":
            prevalence_category(
                prevalence
            ),

        "Included_RIC_Themes":
            "; ".join(
                themes
            )

    })


dimension_summary = pd.DataFrame(
    dimension_rows
)


# ================================================================
# 17. SORT DIMENSION SUMMARY
# ================================================================

if len(dimension_summary) > 0:

    dimension_summary = (
        dimension_summary
        .sort_values(
            [
                "Experts_Mentioning",
                "Number_of_Normalized_Themes"
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )


    if "Rank" in dimension_summary.columns:

        dimension_summary = (
            dimension_summary
            .drop(columns=["Rank"])
        )


    dimension_summary.insert(
        0,
        "Rank",
        range(
            1,
            len(dimension_summary) + 1
        )
    )


# ================================================================
# 18. DIMENSION × THEME TABLE
# ================================================================

dimension_theme_columns = [

    "RIC_Dimension",

    "Final_RIC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Prevalence_Category"

]


dimension_theme_columns = [
    c
    for c in dimension_theme_columns
    if c in final_ric_evidence.columns
]


dimension_themes = final_ric_evidence[
    dimension_theme_columns
].copy()


dimension_themes = (
    dimension_themes
    .sort_values(
        [
            "RIC_Dimension",
            "Experts_Mentioning"
        ],
        ascending=[
            True,
            False
        ]
    )
)


# ================================================================
# 19. REVIEW-REQUIRED THEMES
# ================================================================

unmapped_check = final_ric_evidence[
    final_ric_evidence[
        "RIC_Dimension"
    ]
    ==
    "Review Required"
].copy()


if len(unmapped_check) > 0:

    unmapped_check = unmapped_check[
        [
            "Final_RIC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%"
        ]
    ].copy()


    unmapped_check.insert(
        0,
        "Status",
        "REVIEW REQUIRED"
    )


else:

    unmapped_check = pd.DataFrame({

        "Status": [
            "PASS — All RIC themes mapped to a dimension."
        ]

    })


# ================================================================
# 20. DECISION CHECK
# ================================================================

coding_audit = None

if audit_sheet is not None:

    coding_audit = sheets[
        audit_sheet
    ].copy()


if coding_audit is not None:

    decision_columns = [

        c

        for c in coding_audit.columns

        if "Decision" in str(c)

    ]


    if len(decision_columns) > 0:

        decision_col = (
            decision_columns[0]
        )


        missing = coding_audit[
            coding_audit[
                decision_col
            ].isna()
        ]


        if len(missing) == 0:

            decision_check = pd.DataFrame({

                "Status": [
                    "PASS — All RIC coding decisions are present."
                ]

            })

        else:

            decision_check = missing.copy()


    else:

        decision_check = pd.DataFrame({

            "Status": [
                "CHECK — Decision column not detected."
            ]

        })


else:

    decision_check = pd.DataFrame({

        "Status": [
            "CHECK — Coding audit unavailable."
        ]

    })


# ================================================================
# 21. QUALITY CHECKS
# ================================================================

quality_rows = []


quality_rows.append({

    "Quality_Check":
        "Input workbook found",

    "Result":
        "PASS",

    "Details":
        INPUT_FILE.name

})


quality_rows.append({

    "Quality_Check":
        "Final RIC evidence available",

    "Result":
        "PASS"
        if len(final_ric_evidence) > 0
        else "FAIL",

    "Details":
        f"{len(final_ric_evidence)} final RIC themes"

})


quality_rows.append({

    "Quality_Check":
        "Participants",

    "Result":
        n_participants,

    "Details":
        "Participant count used for prevalence"

})


quality_rows.append({

    "Quality_Check":
        "RIC dimensions generated",

    "Result":
        len(dimension_summary),

    "Details":
        "Qualitative dimensions"

})


review_count = (

    len(unmapped_check)

    if "Final_RIC_Theme"
    in unmapped_check.columns

    else 0

)


quality_rows.append({

    "Quality_Check":
        "Themes requiring review",

    "Result":
        review_count,

    "Details":
        "Themes assigned to Review Required"

})


quality_rows.append({

    "Quality_Check":
        "Duplicate theme labels",

    "Result":
        int(
            final_ric_evidence[
                "Final_RIC_Theme"
            ]
            .str.lower()
            .duplicated()
            .sum()
        ),

    "Details":
        "Case-insensitive duplicate labels detected"

})


quality_checks_new = pd.DataFrame(
    quality_rows
)


# ================================================================
# 22. LOAD SUPPORTING SHEETS
# ================================================================

def get_sheet_or_empty(sheet):

    if sheet is not None:

        return sheets[
            sheet
        ].copy()

    return pd.DataFrame()


original_responses = get_sheet_or_empty(
    original_sheet
)

raw_themes = get_sheet_or_empty(
    raw_sheet
)

cleaned_themes = get_sheet_or_empty(
    cleaned_sheet
)

theme_summary_existing = get_sheet_or_empty(
    theme_sheet
)

normalization_summary = get_sheet_or_empty(
    normalization_sheet
)

decision_summary = get_sheet_or_empty(
    decision_sheet
)

participant_coverage = get_sheet_or_empty(
    coverage_sheet
)

construct_statistics = get_sheet_or_empty(
    statistics_sheet
)


# ================================================================
# 23. WRITE OUTPUT WORKBOOK
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"RIC_FINAL_EVIDENCE_AND_DIMENSIONS_{timestamp}.xlsx"
)


with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    original_responses.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )


    raw_themes.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )


    cleaned_themes.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )


    if coding_audit is not None:

        coding_audit.to_excel(
            writer,
            sheet_name="04_Coding_Audit",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="04_Coding_Audit",
            index=False
        )


    if participant_matrix is not None:

        participant_matrix.to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )


    theme_summary_existing.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )


    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )


    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )


    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )


    unmapped_check.to_excel(
        writer,
        sheet_name="10_Unmapped_Check",
        index=False
    )


    decision_check.to_excel(
        writer,
        sheet_name="11_Decision_Check",
        index=False
    )


    quality_checks_new.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )


    # ------------------------------------------------------------
    # FINAL RIC EVIDENCE
    # ------------------------------------------------------------

    final_ric_evidence.to_excel(
        writer,
        sheet_name="13_Final_RIC_Evidence",
        index=False
    )


    # ------------------------------------------------------------
    # RIC DIMENSION SUMMARY
    # ------------------------------------------------------------

    dimension_summary.to_excel(
        writer,
        sheet_name="14_RIC_Dimension_Summary",
        index=False
    )


    # ------------------------------------------------------------
    # RIC DIMENSION × THEME
    # ------------------------------------------------------------

    dimension_themes.to_excel(
        writer,
        sheet_name="15_RIC_Dimension_Themes",
        index=False
    )


    # ------------------------------------------------------------
    # CONSTRUCT STATISTICS
    # ------------------------------------------------------------

    construct_statistics.to_excel(
        writer,
        sheet_name="16_Construct_Statistics",
        index=False
    )


# ================================================================
# 24. PRINT RESULTS
# ================================================================

print("\n")
print("=" * 80)
print("RIC PROCESS COMPLETED SUCCESSFULLY")
print("=" * 80)


print("\nOutput file:")
print(
    OUTPUT_FILE.resolve()
)


print("\n")
print("-" * 80)
print("FINAL RIC EVIDENCE")
print("-" * 80)

print(
    final_ric_evidence.to_string(
        index=False
    )
)


print("\n")
print("-" * 80)
print("RIC DIMENSION SUMMARY")
print("-" * 80)

print(
    dimension_summary.to_string(
        index=False
    )
)


print("\n")
print("=" * 80)
print("OUTPUT SHEETS")
print("=" * 80)

print("""
01_Original_Responses
02_Raw_Themes
03_Cleaned_Themes
04_Coding_Audit
05_Participant_Matrix
06_Theme_Summary
07_Normalization_Summary
08_Decision_Summary
09_Participant_Coverage
10_Unmapped_Check
11_Decision_Check
12_Quality_Checks
13_Final_RIC_Evidence
14_RIC_Dimension_Summary
15_RIC_Dimension_Themes
16_Construct_Statistics
""")


print("\n")
print("=" * 80)
print("NEXT STAGE")
print("=" * 80)

print("""
For questionnaire development, the two important sheets are:

14_RIC_Dimension_Summary
15_RIC_Dimension_Themes

These provide the bridge from:

Expert responses
      ↓
Raw themes
      ↓
Normalized themes
      ↓
Final RIC themes
      ↓
RIC qualitative dimensions
      ↓
Candidate questionnaire items

The dimensions are qualitative evidence-based groupings.
They are NOT yet statistically validated measurement
dimensions.
""")

RIC FINAL QUALITATIVE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Decision_Intelligence_Risk_Coordination_Supply_Chain_Resilience_1\RIC_FINAL_QUALITATIVE_ANALYSIS_20260827_131423.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Audit
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Normalization_Summary
 - 08_Decision_Summary
 - 09_Participant_Coverage
 - 10_Construct_Statistics
 - 11_Final_RIC_Evidence
 - 12_Quality_Checks
 - 13_Unmapped_Themes

Final RIC evidence sheet: 11_Final_RIC_Evidence

Participants used: 26


RIC PROCESS COMPLETED SUCCESSFULLY

Output file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Decision_Intelligence_Risk_Coordination_Supply_Chain_Resilience_1\RIC_FINAL_EVIDENCE_AND_DIMENSIONS_20260827_131622.xlsx


--------------------------------------------------------------------------------
FINAL RIC EVIDENCE
-----

## Risk Orchestration Capability (ROC)

In [8]:
# ================================================================
# ROC CODING
# INPUT FILE: Themes.xlsx
# ================================================================

import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# ------------------------------------------------
# 1. INPUT
# ------------------------------------------------

input_file = Path("Themes.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"Themes.xlsx was not found in:\n{Path.cwd()}"
    )

excel = pd.ExcelFile(input_file)


# ------------------------------------------------
# 2. FIND PARTICIPANT SHEETS
# ------------------------------------------------

participant_sheets = []

for sheet in excel.sheet_names:

    match = re.fullmatch(
        r"P?\s*0*(\d+)",
        str(sheet).strip(),
        flags=re.IGNORECASE
    )

    if match:

        number = int(match.group(1))

        if 1 <= number <= 26:
            participant_sheets.append(
                (number, sheet)
            )

participant_sheets = sorted(
    participant_sheets,
    key=lambda x: x[0]
)

print("Participant sheets found:")

for number, sheet in participant_sheets:
    print(f"P{number:02d} -> {sheet}")


# ------------------------------------------------
# 3. EXTRACT ROC RESPONSES
# ------------------------------------------------

raw_rows = []

for number, sheet in participant_sheets:

    participant = f"P{number:02d}"

    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        header=None
    )

    for row_number, row in df.iterrows():

        values = []

        for value in row.tolist():

            if pd.isna(value):
                continue

            text = str(value).strip()

            if text != "":
                values.append(text)

        if len(values) < 2:
            continue

        construct_position = None

        for position, value in enumerate(values):

            if value.strip().upper() == "ROC":

                construct_position = position
                break

        if construct_position is None:
            continue

        remaining = values[
            construct_position + 1:
        ]

        if not remaining:
            continue

        response = " ".join(
            remaining
        ).strip()

        if response:

            raw_rows.append({

                "Participant":
                    participant,

                "Construct":
                    "ROC",

                "Original_Response":
                    response,

                "Source_Sheet":
                    sheet,

                "Source_Row":
                    row_number + 1
            })


# ------------------------------------------------
# 4. CHECK
# ------------------------------------------------

original_df = pd.DataFrame(
    raw_rows
)

if original_df.empty:

    raise ValueError(
        """
        No ROC responses were found.

        Check that ROC appears in the participant sheets.
        """
    )

print("\nROC original responses:")
print(len(original_df))


# ------------------------------------------------
# 5. SPLIT INTO RAW THEMES
# ------------------------------------------------

theme_rows = []

for _, row in original_df.iterrows():

    response = str(
        row["Original_Response"]
    )

    response = response.replace(
        "\n", ","
    )

    response = response.replace(
        ";", ","
    )

    response = response.replace(
        "•", ","
    )

    themes = response.split(",")

    for theme in themes:

        theme = str(
            theme
        ).strip()

        if theme == "":
            continue

        theme_rows.append({

            "Participant":
                row["Participant"],

            "Construct":
                "ROC",

            "Raw_Theme":
                theme,

            "Source_Sheet":
                row["Source_Sheet"],

            "Source_Row":
                row["Source_Row"]
        })


raw_df = pd.DataFrame(
    theme_rows
)


# ------------------------------------------------
# 6. CLEAN THEMES
# ------------------------------------------------

clean_df = raw_df.copy()

clean_df["Theme_Clean"] = (
    clean_df["Raw_Theme"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

clean_df["Theme_Key"] = (
    clean_df["Theme_Clean"]
    .str.lower()
    .str.strip()
)

clean_df = clean_df[
    clean_df["Theme_Key"] != ""
]

clean_df = clean_df.drop_duplicates(
    subset=[
        "Participant",
        "Construct",
        "Theme_Key"
    ]
)

clean_df = clean_df.reset_index(
    drop=True
)

print("\nROC cleaned theme observations:")
print(len(clean_df))


# ------------------------------------------------
# 7. DISPLAY RAW ROC THEMES
# ------------------------------------------------

raw_theme_list = (
    clean_df[
        [
            "Raw_Theme",
            "Theme_Clean",
            "Theme_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Theme_Key"
    )
    .reset_index(
        drop=True
    )
)

print("\n")
print("=" * 60)
print("ROC RAW THEMES")
print("=" * 60)

print(
    raw_theme_list.to_string(
        index=False
    )
)


# ------------------------------------------------
# 8. ROC NORMALIZATION DICTIONARY
# ------------------------------------------------
#
# Add researcher-approved ROC mappings here after reviewing
# the actual ROC raw themes.
#
# Example format:
#
# "coordinated risk response":
#     "Coordinated risk response",
#
# "risk response coordination":
#     "Coordinated risk response",
#
# ------------------------------------------------

ROC_NORMALIZATION = {

    # ADD ROC MAPPINGS HERE

}


# ------------------------------------------------
# 9. APPLY NORMALIZATION
# ------------------------------------------------

coding_df = raw_theme_list.copy()

coding_df["Normalized_Theme"] = ""

coding_df["Decision"] = ""

coding_df["Reason"] = ""


for i, row in coding_df.iterrows():

    theme_key = str(
        row["Theme_Key"]
    ).strip().lower()

    if theme_key in ROC_NORMALIZATION:

        normalized = ROC_NORMALIZATION[
            theme_key
        ]

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = normalized

        if theme_key == normalized.lower():

            coding_df.loc[
                i,
                "Decision"
            ] = "Keep"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Retained as a distinct conceptual theme."
            )

        else:

            coding_df.loc[
                i,
                "Decision"
            ] = "Merge"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Merged with a semantically equivalent "
                "theme."
            )

    else:

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = row["Theme_Clean"]

        coding_df.loc[
            i,
            "Decision"
        ] = "Keep"

        coding_df.loc[
            i,
            "Reason"
        ] = (
            "Retained pending conceptual review."
        )


# ------------------------------------------------
# 10. MAP TO PARTICIPANTS
# ------------------------------------------------

mapping = coding_df[
    [
        "Theme_Key",
        "Normalized_Theme",
        "Decision"
    ]
]

coded_df = clean_df.merge(
    mapping,
    on="Theme_Key",
    how="left"
)


# ------------------------------------------------
# 11. PARTICIPANT × ROC THEME MATRIX
# ------------------------------------------------

participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

matrix_source = (
    coded_df
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
)

matrix = pd.crosstab(
    matrix_source["Normalized_Theme"],
    matrix_source["Participant"]
)

matrix = matrix.reindex(
    columns=participants,
    fill_value=0
)

matrix = matrix.reset_index()

matrix["Frequency"] = matrix[
    participants
].sum(axis=1)

matrix["Percentage"] = (
    matrix["Frequency"]
    / len(participants)
    * 100
).round(1)


# ------------------------------------------------
# 12. THEME SUMMARY
# ------------------------------------------------

theme_summary = matrix[
    [
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].sort_values(
    "Frequency",
    ascending=False
)


# ------------------------------------------------
# 13. DECISION SUMMARY
# ------------------------------------------------

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Count"
    )
)


# ------------------------------------------------
# 14. PARTICIPANT COVERAGE
# ------------------------------------------------

participant_coverage = (
    coded_df
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "ROC_Theme_Count"
]


# ------------------------------------------------
# 15. SAVE EXCEL
# ------------------------------------------------

output_file = Path(
    "ROC_Coding_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".xlsx"
)

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_df.to_excel(
        writer,
        sheet_name="04_Coding_Dictionary",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="07_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="08_Participant_Coverage",
        index=False
    )


# ------------------------------------------------
# 16. FINAL REPORT
# ------------------------------------------------

print("\n")
print("=" * 60)
print("ROC CODING COMPLETED")
print("=" * 60)

print(
    "Participants:",
    clean_df["Participant"].nunique()
)

print(
    "Raw theme observations:",
    len(clean_df)
)

print(
    "Unique ROC raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Normalized ROC themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print("\nDECISION SUMMARY:")

print(
    decision_summary.to_string(
        index=False
    )
)

print("\nOUTPUT FILE:")

print(
    output_file.resolve()
)

print("\nDONE.")

Participant sheets found:
P01 -> 1
P02 -> 2
P03 -> 3
P04 -> 4
P05 -> 5
P06 -> 6
P07 -> 7
P08 -> 8
P09 -> 9
P10 -> 10
P11 -> 11
P12 -> 12
P13 -> 13
P14 -> 14
P15 -> 15
P16 -> 16
P17 -> 17
P18 -> 18
P19 -> 19
P20 -> 20
P21 -> 21
P22 -> 22
P23 -> 23
P24 -> 24
P25 -> 25
P26 -> 26

ROC original responses:
26

ROC cleaned theme observations:
89


ROC RAW THEMES
                               Raw_Theme                              Theme_Clean                                Theme_Key
                              adaptation                               adaptation                               adaptation
                     Adaptive allocation                      Adaptive allocation                      adaptive allocation
                              adjustment                               adjustment                               adjustment
                               alignment                                alignment                                alignment
                           

In [9]:
# ================================================================
# COMPLETE ROC QUALITATIVE CODING ANALYSIS
# Input: ROC_Coding_20260827_131834.xlsx
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# ================================================================
# 1. INPUT FILE
# ================================================================

input_file = Path("ROC_Coding_20260827_131834.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"\nFile not found:\n{input_file.resolve()}\n\n"
        "Make sure the Excel file is in the same folder as your "
        "Python notebook."
    )

print("=" * 70)
print("ROC QUALITATIVE CODING ANALYSIS")
print("=" * 70)
print("\nInput file:")
print(input_file.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

excel = pd.ExcelFile(input_file)

print("\nSheets found:")
for s in excel.sheet_names:
    print(" -", s)


# ================================================================
# 3. FLEXIBLE SHEET FINDER
# ================================================================

def find_sheet(possible_names):

    for name in possible_names:
        if name in excel.sheet_names:
            return name

    for sheet in excel.sheet_names:

        a = (
            str(sheet)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for name in possible_names:

            b = (
                str(name)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if b in a:
                return sheet

    return None


original_sheet = find_sheet([
    "01_Original_ROC",
    "01_Original",
    "Original_ROC",
    "Original"
])

raw_sheet = find_sheet([
    "02_Raw_ROC",
    "02_Raw",
    "Raw_ROC",
    "Raw"
])

clean_sheet = find_sheet([
    "03_Cleaned_ROC",
    "03_Cleaned",
    "Cleaned_ROC",
    "Cleaned"
])

coding_sheet = find_sheet([
    "04_Normalized_Coding",
    "04_Coding",
    "Normalized_Coding",
    "Coding"
])


print("\nSelected sheets:")
print("Original:", original_sheet)
print("Raw:", raw_sheet)
print("Cleaned:", clean_sheet)
print("Coding:", coding_sheet)


# ================================================================
# 4. READ SHEETS
# ================================================================

original_df = (
    pd.read_excel(
        input_file,
        sheet_name=original_sheet
    )
    if original_sheet
    else pd.DataFrame()
)

raw_df = (
    pd.read_excel(
        input_file,
        sheet_name=raw_sheet
    )
    if raw_sheet
    else pd.DataFrame()
)

clean_df = (
    pd.read_excel(
        input_file,
        sheet_name=clean_sheet
    )
    if clean_sheet
    else pd.DataFrame()
)

if coding_sheet is None:
    raise ValueError(
        "\nCould not find the ROC Coding/Normalized Coding sheet.\n"
        "Available sheets are:\n"
        + "\n".join(
            str(x)
            for x in excel.sheet_names
        )
    )

coding_df = pd.read_excel(
    input_file,
    sheet_name=coding_sheet
)


# ================================================================
# 5. STANDARDIZE COLUMN NAMES
# ================================================================

for df in [
    original_df,
    raw_df,
    clean_df,
    coding_df
]:

    if not df.empty:
        df.columns = [
            str(c).strip()
            for c in df.columns
        ]


print("\nCoding columns:")
print(list(coding_df.columns))


# ================================================================
# 6. COLUMN FINDER
# ================================================================

def find_column(df, candidates):

    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    for column in df.columns:

        a = (
            str(column)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for candidate in candidates:

            b = (
                str(candidate)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if a == b:
                return column

    return None


theme_key_col = find_column(
    coding_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

raw_theme_col = find_column(
    coding_df,
    [
        "Raw_Theme",
        "Raw Theme",
        "RawTheme"
    ]
)

normalized_col = find_column(
    coding_df,
    [
        "Normalized_Theme",
        "Normalized Theme",
        "NormalizedTheme"
    ]
)

decision_col = find_column(
    coding_df,
    [
        "Decision"
    ]
)


if theme_key_col is None:
    raise ValueError(
        "\nTheme_Key column not found."
    )

if raw_theme_col is None:
    raise ValueError(
        "\nRaw_Theme column not found."
    )

if normalized_col is None:
    raise ValueError(
        "\nNormalized_Theme column not found."
    )

if decision_col is None:
    raise ValueError(
        "\nDecision column not found."
    )


coding_df = coding_df.rename(
    columns={
        theme_key_col: "Theme_Key",
        raw_theme_col: "Raw_Theme",
        normalized_col: "Normalized_Theme",
        decision_col: "Decision"
    }
)


# ================================================================
# 7. FIND PARTICIPANT COLUMN
# ================================================================

participant_col = find_column(
    clean_df,
    [
        "Participant",
        "Participant_ID",
        "Participant ID",
        "ParticipantID"
    ]
)

if participant_col is None:

    raise ValueError(
        "\nParticipant column not found in Cleaned ROC sheet.\n\n"
        "Available columns:\n"
        + str(list(clean_df.columns))
    )


if participant_col != "Participant":

    clean_df = clean_df.rename(
        columns={
            participant_col:
            "Participant"
        }
    )


# ================================================================
# 8. FIND THEME KEY IN CLEANED DATA
# ================================================================

clean_theme_key = find_column(
    clean_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

if clean_theme_key is None:

    raise ValueError(
        "\nTheme_Key not found in Cleaned ROC sheet.\n\n"
        "Available columns:\n"
        + str(list(clean_df.columns))
    )


if clean_theme_key != "Theme_Key":

    clean_df = clean_df.rename(
        columns={
            clean_theme_key:
            "Theme_Key"
        }
    )


# ================================================================
# 9. CLEAN CODING DICTIONARY
# ================================================================

coding_df["Theme_Key"] = (
    coding_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

coding_df["Raw_Theme"] = (
    coding_df["Raw_Theme"]
    .fillna("")
    .astype(str)
    .str.strip()
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .replace(
        [
            "nan",
            "None",
            "",
            "NaN"
        ],
        np.nan
    )
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .astype("string")
    .str.strip()
)

coding_df["Decision"] = (
    coding_df["Decision"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 10. CLEAN PARTICIPANT DATA
# ================================================================

clean_df["Theme_Key"] = (
    clean_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

clean_df["Participant"] = (
    clean_df["Participant"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 11. PARTICIPANT LIST
# ================================================================

participants = sorted(
    clean_df[
        "Participant"
    ]
    .replace("", np.nan)
    .dropna()
    .unique()
)

print("\nParticipants:")
print(participants)

print(
    "\nNumber of participants:",
    len(participants)
)


# ================================================================
# 12. MERGE CLEANED DATA WITH CODING DICTIONARY
# ================================================================

coded_df = clean_df.merge(
    coding_df[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision"
        ]
    ],
    on="Theme_Key",
    how="left"
)


# ================================================================
# 13. CHECK UNMAPPED THEMES
# ================================================================

missing_mapping = coded_df[
    coded_df[
        "Normalized_Theme"
    ].isna()
].copy()


print(
    "\nNumber of unmapped themes:",
    len(missing_mapping)
)


if len(missing_mapping) > 0:

    print("\nUnmapped themes:")

    print(
        missing_mapping[
            [
                "Participant",
                "Theme_Key"
            ]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )


# ================================================================
# 14. PARTICIPANT × NORMALIZED ROC THEME
# ================================================================

participant_theme = (
    coded_df[
        [
            "Participant",
            "Normalized_Theme"
        ]
    ]
    .dropna()
    .drop_duplicates()
)


# ================================================================
# 15. PARTICIPANT × THEME MATRIX
# ================================================================

matrix = pd.crosstab(
    participant_theme[
        "Normalized_Theme"
    ],
    participant_theme[
        "Participant"
    ]
)


# ================================================================
# 16. ORDER PARTICIPANTS
# ================================================================

expected_participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

existing = [
    p
    for p in expected_participants
    if p in matrix.columns
]

other = [
    p
    for p in matrix.columns
    if p not in existing
]

matrix = matrix.reindex(
    columns=existing + other,
    fill_value=0
)

matrix = matrix.reset_index()

participant_columns = (
    existing + other
)


# ================================================================
# 17. FREQUENCY
# ================================================================

matrix["Frequency"] = (
    matrix[
        participant_columns
    ]
    .sum(axis=1)
)


# ================================================================
# 18. PERCENTAGE OF PARTICIPANTS
# ================================================================

total_participants = len(participants)

if total_participants > 0:

    matrix["Percentage"] = (
        matrix["Frequency"]
        / total_participants
        * 100
    ).round(1)

else:

    matrix["Percentage"] = 0


# ================================================================
# 19. RANK THEMES
# ================================================================

matrix = matrix.sort_values(
    [
        "Frequency",
        "Normalized_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)


matrix.insert(
    0,
    "Rank",
    range(
        1,
        len(matrix) + 1
    )
)


# ================================================================
# 20. ROC THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Rank",
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].copy()


theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme":
            "Final_ROC_Theme",

        "Frequency":
            "Experts_Mentioning",

        "Percentage":
            "Percentage_of_Experts"
    }
)


# ================================================================
# 21. PREVALENCE CATEGORY
# ================================================================

def prevalence_category(p):

    if p >= 75:
        return "Very High"

    elif p >= 50:
        return "High"

    elif p >= 25:
        return "Moderate"

    else:
        return "Low"


theme_summary[
    "Prevalence_Category"
] = (
    theme_summary[
        "Percentage_of_Experts"
    ]
    .apply(
        prevalence_category
    )
)


# ================================================================
# 22. CODING AUDIT
# ================================================================

theme_participant_counts = (
    participant_theme
    .groupby(
        "Normalized_Theme"
    )
    .size()
    .reset_index(
        name="Experts_Mentioning"
    )
)


coding_audit = coding_df.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)


coding_audit[
    "Experts_Mentioning"
] = (
    coding_audit[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


if total_participants > 0:

    coding_audit[
        "Percentage_of_Experts"
    ] = (
        coding_audit[
            "Experts_Mentioning"
        ]
        / total_participants
        * 100
    ).round(1)

else:

    coding_audit[
        "Percentage_of_Experts"
    ] = 0


coding_audit = coding_audit.sort_values(
    [
        "Normalized_Theme",
        "Decision",
        "Raw_Theme"
    ]
).reset_index(drop=True)


# ================================================================
# 23. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby(
        "Decision"
    )
    .size()
    .reset_index(
        name="Number_of_Raw_Themes"
    )
)


if len(coding_df) > 0:

    decision_summary[
        "Percentage"
    ] = (
        decision_summary[
            "Number_of_Raw_Themes"
        ]
        / len(coding_df)
        * 100
    ).round(1)

else:

    decision_summary[
        "Percentage"
    ] = 0


# ================================================================
# 24. NORMALIZATION SUMMARY
# ================================================================

normalization_summary = (
    coding_df
    .dropna(
        subset=[
            "Normalized_Theme"
        ]
    )
    .groupby(
        "Normalized_Theme"
    )
    .agg(
        Raw_Themes=(
            "Raw_Theme",
            "count"
        ),

        Keep_Count=(
            "Decision",
            lambda x:
            (x == "Keep").sum()
        ),

        Merge_Count=(
            "Decision",
            lambda x:
            (x == "Merge").sum()
        )
    )
    .reset_index()
)


normalization_summary = (
    normalization_summary.merge(
        theme_participant_counts,
        on="Normalized_Theme",
        how="left"
    )
)


normalization_summary[
    "Experts_Mentioning"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


if total_participants > 0:

    normalization_summary[
        "Percentage_of_Experts"
    ] = (
        normalization_summary[
            "Experts_Mentioning"
        ]
        / total_participants
        * 100
    ).round(1)

else:

    normalization_summary[
        "Percentage_of_Experts"
    ] = 0


normalization_summary = (
    normalization_summary
    .sort_values(
        "Experts_Mentioning",
        ascending=False
    )
    .reset_index(drop=True)
)


# ================================================================
# 25. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    participant_theme
    .groupby(
        "Participant"
    )
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)


participant_coverage.columns = [
    "Participant",
    "Number_of_Normalized_ROC_Themes"
]


# ================================================================
# 26. ROC CONSTRUCT STATISTICS
# ================================================================

construct_statistics = pd.DataFrame({

    "Construct": [
        "ROC"
    ],

    "Participants": [
        total_participants
    ],

    "Original_Response_Rows": [
        len(original_df)
    ],

    "Raw_Theme_Observations": [
        len(raw_df)
    ],

    "Cleaned_Theme_Observations": [
        len(clean_df)
    ],

    "Unique_Raw_Themes": [
        coding_df[
            "Theme_Key"
        ].nunique()
    ],

    "Final_Normalized_Themes": [
        coding_df[
            "Normalized_Theme"
        ].nunique()
    ],

    "Merged_Raw_Themes": [
        (
            coding_df[
                "Decision"
            ]
            == "Merge"
        ).sum()
    ],

    "Kept_Raw_Themes": [
        (
            coding_df[
                "Decision"
            ]
            == "Keep"
        ).sum()
    ],

    "Unmapped_Themes": [
        len(missing_mapping)
    ]
})


# ================================================================
# 27. FINAL ROC EVIDENCE
# ================================================================

final_evidence = normalization_summary[
    [
        "Normalized_Theme",
        "Raw_Themes",
        "Experts_Mentioning",
        "Percentage_of_Experts",
        "Keep_Count",
        "Merge_Count"
    ]
].copy()


final_evidence = final_evidence.rename(
    columns={
        "Normalized_Theme":
            "Final_ROC_Theme",

        "Raw_Themes":
            "Number_of_Raw_Themes",

        "Percentage_of_Experts":
            "Expert_Prevalence_%",

        "Keep_Count":
            "Raw_Themes_Kept",

        "Merge_Count":
            "Raw_Themes_Merged"
    }
)


final_evidence[
    "Prevalence_Category"
] = (
    final_evidence[
        "Expert_Prevalence_%"
    ]
    .apply(
        prevalence_category
    )
)


# ================================================================
# 28. QUALITY CHECKS
# ================================================================

duplicate_count = (
    len(coded_df)
    -
    len(
        coded_df[
            [
                "Participant",
                "Normalized_Theme"
            ]
        ]
        .dropna()
        .drop_duplicates()
    )
)


quality_checks = pd.DataFrame({

    "Check": [

        "Number of participants",
        "Original response rows",
        "Raw theme observations",
        "Unique raw themes",
        "Final normalized themes",
        "Unmapped themes",
        "Duplicate participant-theme records",
        "Keep decisions",
        "Merge decisions"

    ],

    "Result": [

        total_participants,
        len(original_df),
        len(raw_df),

        coding_df[
            "Theme_Key"
        ].nunique(),

        coding_df[
            "Normalized_Theme"
        ].nunique(),

        len(missing_mapping),

        duplicate_count,

        (
            coding_df[
                "Decision"
            ] == "Keep"
        ).sum(),

        (
            coding_df[
                "Decision"
            ] == "Merge"
        ).sum()

    ],

    "Status": [

        "PASS"
        if total_participants > 0
        else "CHECK",

        "PASS"
        if len(original_df) > 0
        else "CHECK",

        "PASS"
        if len(raw_df) > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Theme_Key"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Normalized_Theme"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if len(missing_mapping) == 0
        else "CHECK",

        "INFO",
        "PASS",
        "PASS"

    ]
})


# ================================================================
# 29. SAVE COMPLETE ROC WORKBOOK
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_file = Path(
    f"ROC_FINAL_QUALITATIVE_ANALYSIS_{timestamp}.xlsx"
)


# Avoid overwrite / PermissionError

counter = 1

while output_file.exists():

    output_file = Path(
        f"ROC_FINAL_QUALITATIVE_ANALYSIS_"
        f"{timestamp}_{counter}.xlsx"
    )

    counter += 1


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_ROC_Evidence",
        index=False
    )

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    missing_mapping.to_excel(
        writer,
        sheet_name="13_Unmapped_Themes",
        index=False
    )


# ================================================================
# 30. FINAL REPORT
# ================================================================

print("\n")
print("=" * 70)
print("ROC ANALYSIS COMPLETED")
print("=" * 70)

print("\nParticipants:", total_participants)

print("Original responses:", len(original_df))

print("Raw theme observations:", len(raw_df))

print("Cleaned theme observations:", len(clean_df))

print(
    "Unique raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Final normalized ROC themes:",
    coding_df["Normalized_Theme"].nunique()
)

print(
    "Keep decisions:",
    (
        coding_df["Decision"] == "Keep"
    ).sum()
)

print(
    "Merge decisions:",
    (
        coding_df["Decision"] == "Merge"
    ).sum()
)

print(
    "Unmapped themes:",
    len(missing_mapping)
)


# ================================================================
# 31. DISPLAY FINAL ROC THEMES
# ================================================================

print("\n")
print("=" * 70)
print("FINAL ROC THEMES")
print("=" * 70)

print(
    final_evidence[
        [
            "Final_ROC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Prevalence_Category"
        ]
    ].to_string(index=False)
)


# ================================================================
# 32. OUTPUT FILE
# ================================================================

print("\n")
print("=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(output_file.resolve())

print(
    "\nComplete ROC qualitative analysis workbook "
    "created successfully."
)

ROC QUALITATIVE CODING ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Decision_Intelligence_Risk_Coordination_Supply_Chain_Resilience_1\ROC_Coding_20260827_131834.xlsx

Sheets found:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Dictionary
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Decision_Summary
 - 08_Participant_Coverage

Selected sheets:
Original: 01_Original_Responses
Raw: 02_Raw_Themes
Cleaned: 03_Cleaned_Themes
Coding: 04_Coding_Dictionary

Coding columns:
['Raw_Theme', 'Theme_Clean', 'Theme_Key', 'Normalized_Theme', 'Decision', 'Reason']

Participants:
['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26']

Number of participants: 26

Number of unmapped themes: 0


ROC ANALYSIS COMPLETED

Participants: 26
Original responses: 26
Raw theme observations: 89
C

In [ ]:
# ================================================================
# ROC — FINAL QUALITATIVE EVIDENCE + DIMENSION ANALYSIS
# ================================================================
#
# INPUT:
# ROC_FINAL_QUALITATIVE_ANALYSIS_20260827_131920.xlsx
#
# OUTPUT:
# ROC_FINAL_EVIDENCE_AND_DIMENSIONS_20260827_132024.xlsx
#
# PURPOSE:
# Finalize ROC qualitative evidence and organize the themes
# into evidence-based dimensions for questionnaire development.
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import os


# ================================================================
# 1. FIND INPUT FILE
# ================================================================

TARGET = "ROC_FINAL_QUALITATIVE_ANALYSIS_20260827_131920"

possible_files = []

# First check the current working directory and common folders
search_locations = [
    Path("."),
    Path.home(),
    Path.home() / "Downloads",
    Path.home() / "Documents",
    Path.home() / "Desktop",
]

for location in search_locations:
    if location.exists():
        for ext in [".xlsx", ".xlsm", ".xls"]:
            try:
                possible_files.extend(
                    location.rglob(TARGET + ext)
                )
            except Exception:
                pass

# Also check /mnt/data for uploaded/copied files
try:
    possible_files.extend(
        Path("/mnt/data").glob("*.xlsx")
    )
except Exception:
    pass

possible_files = list(
    dict.fromkeys(
        [p.resolve() for p in possible_files]
    )
)

# Prefer an exact filename match
exact_files = [
    p for p in possible_files
    if p.stem == TARGET
]

if len(exact_files) > 0:
    INPUT_FILE = exact_files[0]

else:

    # If the original filename has been renamed by the upload system,
    # identify the workbook by its ROC-specific sheet.
    roc_candidates = []

    for p in possible_files:

        try:
            xl = pd.ExcelFile(
                p,
                engine="openpyxl"
            )

            if "11_Final_ROC_Evidence" in xl.sheet_names:
                roc_candidates.append(p)

        except Exception:
            pass

    if len(roc_candidates) > 0:
        INPUT_FILE = roc_candidates[0]

    else:
        raise FileNotFoundError(
            "\nROC input workbook could not be found.\n\n"
            "Expected filename:\n"
            f"{TARGET}.xlsx"
        )


print("=" * 80)
print("ROC FINAL QUALITATIVE ANALYSIS")
print("=" * 80)

print("\nInput file:")
print(INPUT_FILE)


# ================================================================
# 2. READ WORKBOOK
# ================================================================

xls = pd.ExcelFile(
    INPUT_FILE,
    engine="openpyxl"
)

print("\nAvailable sheets:")

for s in xls.sheet_names:
    print(" -", s)


# ================================================================
# 3. LOAD ALL SHEETS
# ================================================================

sheets = {}

for sheet in xls.sheet_names:

    try:

        sheets[sheet] = pd.read_excel(
            INPUT_FILE,
            sheet_name=sheet
        )

    except Exception as e:

        print(
            f"Warning: could not read {sheet}: {e}"
        )


# ================================================================
# 4. HELPER FUNCTIONS
# ================================================================

def find_sheet(keyword):

    for s in sheets.keys():

        if keyword.lower() in s.lower():
            return s

    return None


def clean_text(x):

    if pd.isna(x):
        return ""

    x = str(x).strip()

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x


def prevalence_category(x):

    try:
        x = float(x)
    except:
        return "Not available"

    if x >= 75:
        return "Very High"

    elif x >= 50:
        return "High"

    elif x >= 25:
        return "Moderate"

    else:
        return "Low"


# ================================================================
# 5. IDENTIFY SHEETS
# ================================================================

original_sheet = find_sheet(
    "01_Original_Responses"
)

raw_sheet = find_sheet(
    "02_Raw_Themes"
)

cleaned_sheet = find_sheet(
    "03_Cleaned_Themes"
)

audit_sheet = find_sheet(
    "04_Coding_Audit"
)

matrix_sheet = find_sheet(
    "05_Participant_Matrix"
)

theme_sheet = find_sheet(
    "06_Theme_Summary"
)

normalization_sheet = find_sheet(
    "07_Normalization_Summary"
)

decision_sheet = find_sheet(
    "08_Decision_Summary"
)

coverage_sheet = find_sheet(
    "09_Participant_Coverage"
)

statistics_sheet = find_sheet(
    "10_Construct_Statistics"
)

final_sheet = find_sheet(
    "11_Final_ROC_Evidence"
)

quality_sheet = find_sheet(
    "12_Quality_Checks"
)

unmapped_sheet = find_sheet(
    "13_Unmapped_Themes"
)


if final_sheet is None:

    raise ValueError(
        "11_Final_ROC_Evidence was not found."
    )


# ================================================================
# 6. READ FINAL ROC EVIDENCE
# ================================================================

final_evidence = sheets[
    final_sheet
].copy()

final_evidence.columns = [
    str(c).strip()
    for c in final_evidence.columns
]


# ================================================================
# 7. FIND ROC THEME COLUMN
# ================================================================

theme_col = None

if "Final_ROC_Theme" in final_evidence.columns:

    theme_col = "Final_ROC_Theme"

else:

    for c in final_evidence.columns:

        if (
            "ROC" in str(c)
            and "Theme" in str(c)
        ):

            theme_col = c
            break


if theme_col is None:

    for c in final_evidence.columns:

        if "Theme" in str(c):

            theme_col = c
            break


if theme_col is None:

    raise ValueError(
        "Could not identify the final ROC theme column."
    )


final_evidence[
    "Final_ROC_Theme"
] = (
    final_evidence[
        theme_col
    ]
    .apply(clean_text)
)


# ================================================================
# 8. STANDARDIZE EXPERT COLUMNS
# ================================================================

if "Experts_Mentioning" not in final_evidence.columns:

    for c in final_evidence.columns:

        if (
            "Experts" in str(c)
            and "Mention" in str(c)
        ):

            final_evidence[
                "Experts_Mentioning"
            ] = final_evidence[c]

            break


if "Expert_Prevalence_%" not in final_evidence.columns:

    for c in final_evidence.columns:

        if (
            "Prevalence" in str(c)
            and "%" in str(c)
        ):

            final_evidence[
                "Expert_Prevalence_%"
            ] = final_evidence[c]

            break


# ================================================================
# 9. REMOVE EMPTY THEMES
# ================================================================

final_evidence = final_evidence[
    final_evidence[
        "Final_ROC_Theme"
    ] != ""
].copy()


# ================================================================
# 10. PARTICIPANT COUNT
# ================================================================

participant_matrix = None

if matrix_sheet is not None:

    participant_matrix = sheets[
        matrix_sheet
    ].copy()


participants = []

if participant_matrix is not None:

    for c in participant_matrix.columns:

        cstr = str(c).strip()

        if re.match(
            r"^P\d+$",
            cstr,
            flags=re.IGNORECASE
        ):

            participants.append(cstr)


# Your qualitative dataset has 26 participants
if len(participants) > 0:

    n_participants = len(participants)

else:

    n_participants = 26


print(
    "\nParticipants used:",
    n_participants
)


# ================================================================
# 11. PREVALENCE CATEGORY
# ================================================================

if "Expert_Prevalence_%" in final_evidence.columns:

    final_evidence[
        "Prevalence_Category"
    ] = (
        final_evidence[
            "Expert_Prevalence_%"
        ]
        .apply(prevalence_category)
    )


# ================================================================
# 12. SORT FINAL EVIDENCE
# ================================================================

if "Experts_Mentioning" in final_evidence.columns:

    final_evidence = (
        final_evidence
        .sort_values(
            "Experts_Mentioning",
            ascending=False
        )
        .reset_index(drop=True)
    )


# ================================================================
# 13. FINAL ROC EVIDENCE TABLE
# ================================================================

preferred_columns = [

    "Final_ROC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Raw_Themes_Kept",

    "Raw_Themes_Merged",

    "Prevalence_Category"

]

evidence_columns = [
    c for c in preferred_columns
    if c in final_evidence.columns
]

final_roc_evidence = final_evidence[
    evidence_columns
].copy()


# Prevent Rank duplication
if "Rank" in final_roc_evidence.columns:

    final_roc_evidence = (
        final_roc_evidence
        .drop(columns=["Rank"])
    )


final_roc_evidence.insert(
    0,
    "Rank",
    range(
        1,
        len(final_roc_evidence) + 1
    )
)


# ================================================================
# 14. ROC DIMENSION MAPPING
# ================================================================
#
# These dimensions are intended for questionnaire-development
# purposes. They organize the final ROC themes according to
# what the experts describe as risk orchestration activities.
#
# They are NOT yet statistically validated subdimensions.
# ================================================================

def map_roc_dimension(theme):

    t = str(theme).strip().lower()


    # ------------------------------------------------------------
    # 1. RISK RESPONSE COORDINATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "coordination",
            "coordinating",
            "cross-functional",
            "cross functional",
            "interdepartmental",
            "between departments",
            "between functions",
            "stakeholder coordination",
            "supplier coordination",
            "partner coordination"

        ]
    ):

        return (
            "Risk Response Coordination"
        )


    # ------------------------------------------------------------
    # 2. RISK RESPONSE PRIORITIZATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "prioritization",
            "prioritisation",
            "priority",
            "prioritize",
            "prioritise",
            "resource allocation",
            "resource allocation decisions",
            "critical risks",
            "most critical"

        ]
    ):

        return (
            "Risk Response Prioritization"
        )


    # ------------------------------------------------------------
    # 3. RISK RESPONSE INTEGRATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "integrated response",
            "integrated action",
            "integrating",
            "integration",
            "combined response",
            "coordinated response",
            "holistic response",
            "response across"

        ]
    ):

        return (
            "Integrated Risk Response"
        )


    # ------------------------------------------------------------
    # 4. RESOURCE & ACTION ORCHESTRATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "resource",
            "capacity",
            "reallocate",
            "reallocation",
            "deploy",
            "deployment",
            "mobilize",
            "mobilise",
            "action",
            "response action",
            "corrective action"

        ]
    ):

        return (
            "Resource & Action Orchestration"
        )


    # ------------------------------------------------------------
    # 5. ADAPTIVE RISK RESPONSE
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "adapt",
            "adaptive",
            "adjust",
            "adjustment",
            "flexibility",
            "flexible response",
            "changing conditions",
            "changing circumstances",
            "dynamic response",
            "respond to changes"

        ]
    ):

        return (
            "Adaptive Risk Response"
        )


    # ------------------------------------------------------------
    # 6. RISK RESPONSE MONITORING & FOLLOW-UP
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "monitor response",
            "monitoring response",
            "follow-up",
            "follow up",
            "tracking",
            "response effectiveness",
            "response performance",
            "evaluate response",
            "review response"

        ]
    ):

        return (
            "Risk Response Monitoring & Follow-up"
        )


    # ------------------------------------------------------------
    # 7. REVIEW REQUIRED
    # ------------------------------------------------------------

    return "Review Required"


final_roc_evidence[
    "ROC_Dimension"
] = (
    final_roc_evidence[
        "Final_ROC_Theme"
    ]
    .apply(
        map_roc_dimension
    )
)


# ================================================================
# 15. ADDITIONAL MANUAL MAPPING
# ================================================================
#
# Only explicit labels are mapped here.
# Anything not confidently mapped remains Review Required.
# ================================================================

manual_mapping = {

    # Coordination
    "coordination":
        "Risk Response Coordination",

    "risk coordination":
        "Risk Response Coordination",

    "cross-functional coordination":
        "Risk Response Coordination",

    "cross functional coordination":
        "Risk Response Coordination",

    "stakeholder coordination":
        "Risk Response Coordination",

    "supplier coordination":
        "Risk Response Coordination",

    "partner coordination":
        "Risk Response Coordination",


    # Prioritization
    "prioritization":
        "Risk Response Prioritization",

    "prioritisation":
        "Risk Response Prioritization",

    "priority":
        "Risk Response Prioritization",

    "resource allocation":
        "Risk Response Prioritization",

    "critical risks":
        "Risk Response Prioritization",


    # Integration
    "integration":
        "Integrated Risk Response",

    "integrated response":
        "Integrated Risk Response",

    "integrated action":
        "Integrated Risk Response",

    "combined response":
        "Integrated Risk Response",

    "holistic response":
        "Integrated Risk Response",


    # Resource / action
    "resource":
        "Resource & Action Orchestration",

    "resource allocation decisions":
        "Resource & Action Orchestration",

    "reallocation":
        "Resource & Action Orchestration",

    "resource reallocation":
        "Resource & Action Orchestration",

    "deployment":
        "Resource & Action Orchestration",

    "mobilization":
        "Resource & Action Orchestration",

    "mobilisation":
        "Resource & Action Orchestration",


    # Adaptation
    "adaptation":
        "Adaptive Risk Response",

    "adaptive response":
        "Adaptive Risk Response",

    "flexibility":
        "Adaptive Risk Response",

    "flexible response":
        "Adaptive Risk Response",

    "dynamic response":
        "Adaptive Risk Response",


    # Monitoring
    "response monitoring":
        "Risk Response Monitoring & Follow-up",

    "monitoring response":
        "Risk Response Monitoring & Follow-up",

    "follow-up":
        "Risk Response Monitoring & Follow-up",

    "follow up":
        "Risk Response Monitoring & Follow-up",

    "response effectiveness":
        "Risk Response Monitoring & Follow-up",

    "response performance":
        "Risk Response Monitoring & Follow-up"

}


for theme, dimension in manual_mapping.items():

    mask = (
        final_roc_evidence[
            "Final_ROC_Theme"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq(
            theme.lower()
        )
    )

    final_roc_evidence.loc[
        mask,
        "ROC_Dimension"
    ] = dimension


# ================================================================
# 16. DIMENSION SUMMARY
# ================================================================

dimension_rows = []


for dimension, group in (
    final_roc_evidence
    .groupby(
        "ROC_Dimension",
        dropna=False
    )
):

    themes = (
        group[
            "Final_ROC_Theme"
        ]
        .astype(str)
        .tolist()
    )


    experts_mentioning = 0


    # ------------------------------------------------------------
    # Use participant matrix when available
    # ------------------------------------------------------------

    if (
        participant_matrix is not None
        and len(participants) > 0
    ):

        temp = participant_matrix.copy()


        theme_candidates = [
            c
            for c in temp.columns
            if "theme" in str(c).lower()
        ]


        if len(theme_candidates) > 0:

            matrix_theme_col = (
                theme_candidates[0]
            )

        else:

            matrix_theme_col = (
                temp.columns[0]
            )


        temp["_Theme"] = (
            temp[
                matrix_theme_col
            ]
            .astype(str)
            .str.strip()
        )


        theme_lower = {
            str(x).strip().lower()
            for x in themes
        }


        matched = temp[
            temp[
                "_Theme"
            ]
            .str.lower()
            .isin(
                theme_lower
            )
        ]


        available_participants = [

            p

            for p in participants

            if p in matched.columns

        ]


        if (
            len(matched) > 0
            and len(available_participants) > 0
        ):

            vals = (
                matched[
                    available_participants
                ]
                .apply(
                    pd.to_numeric,
                    errors="coerce"
                )
                .fillna(0)
            )


            experts_mentioning = int(
                (
                    vals.sum(axis=0) > 0
                ).sum()
            )


    # ------------------------------------------------------------
    # Fallback
    # ------------------------------------------------------------

    if experts_mentioning == 0:

        if (
            "Experts_Mentioning"
            in group.columns
        ):

            try:

                experts_mentioning = int(
                    pd.to_numeric(
                        group[
                            "Experts_Mentioning"
                        ],
                        errors="coerce"
                    )
                    .max()
                )

            except:

                experts_mentioning = 0


    prevalence = (

        experts_mentioning
        /
        n_participants
        *
        100

        if n_participants > 0
        else 0

    )


    dimension_rows.append({

        "ROC_Dimension":
            dimension,

        "Number_of_Normalized_Themes":
            len(themes),

        "Experts_Mentioning":
            experts_mentioning,

        "Expert_Prevalence_%":
            round(
                prevalence,
                1
            ),

        "Prevalence_Category":
            prevalence_category(
                prevalence
            ),

        "Included_ROC_Themes":
            "; ".join(
                themes
            )

    })


dimension_summary = pd.DataFrame(
    dimension_rows
)


# ================================================================
# 17. SORT DIMENSIONS
# ================================================================

if len(dimension_summary) > 0:

    dimension_summary = (
        dimension_summary
        .sort_values(
            [
                "Experts_Mentioning",
                "Number_of_Normalized_Themes"
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )


    if "Rank" in dimension_summary.columns:

        dimension_summary = (
            dimension_summary
            .drop(columns=["Rank"])
        )


    dimension_summary.insert(
        0,
        "Rank",
        range(
            1,
            len(dimension_summary) + 1
        )
    )


# ================================================================
# 18. DIMENSION × THEME TABLE
# ================================================================

dimension_theme_columns = [

    "ROC_Dimension",

    "Final_ROC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Prevalence_Category"

]


dimension_theme_columns = [
    c
    for c in dimension_theme_columns
    if c in final_roc_evidence.columns
]


dimension_themes = final_roc_evidence[
    dimension_theme_columns
].copy()


dimension_themes = (
    dimension_themes
    .sort_values(
        [
            "ROC_Dimension",
            "Experts_Mentioning"
        ],
        ascending=[
            True,
            False
        ]
    )
)


# ================================================================
# 19. REVIEW-REQUIRED THEMES
# ================================================================

unmapped_check = final_roc_evidence[
    final_roc_evidence[
        "ROC_Dimension"
    ]
    ==
    "Review Required"
].copy()


if len(unmapped_check) > 0:

    unmapped_check = unmapped_check[
        [
            "Final_ROC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%"
        ]
    ].copy()


    unmapped_check.insert(
        0,
        "Status",
        "REVIEW REQUIRED"
    )

else:

    unmapped_check = pd.DataFrame({

        "Status": [
            "PASS — All ROC themes mapped to a dimension."
        ]

    })


# ================================================================
# 20. CODING DECISION CHECK
# ================================================================

coding_audit = None

if audit_sheet is not None:

    coding_audit = sheets[
        audit_sheet
    ].copy()


if coding_audit is not None:

    decision_columns = [

        c
        for c in coding_audit.columns
        if "Decision" in str(c)

    ]


    if len(decision_columns) > 0:

        decision_col = (
            decision_columns[0]
        )


        missing = coding_audit[
            coding_audit[
                decision_col
            ].isna()
        ]


        if len(missing) == 0:

            decision_check = pd.DataFrame({

                "Status": [
                    "PASS — All ROC coding decisions are present."
                ]

            })

        else:

            decision_check = missing.copy()


    else:

        decision_check = pd.DataFrame({

            "Status": [
                "CHECK — Decision column not detected."
            ]

        })

else:

    decision_check = pd.DataFrame({

        "Status": [
            "CHECK — Coding audit unavailable."
        ]

    })


# ================================================================
# 21. QUALITY CHECKS
# ================================================================

quality_rows = []


quality_rows.append({

    "Quality_Check":
        "Input workbook found",

    "Result":
        "PASS",

    "Details":
        INPUT_FILE.name

})


quality_rows.append({

    "Quality_Check":
        "Final ROC evidence available",

    "Result":
        "PASS"
        if len(final_roc_evidence) > 0
        else "FAIL",

    "Details":
        f"{len(final_roc_evidence)} final ROC themes"

})


quality_rows.append({

    "Quality_Check":
        "Participants",

    "Result":
        n_participants,

    "Details":
        "Participant count used for prevalence"

})


quality_rows.append({

    "Quality_Check":
        "ROC dimensions generated",

    "Result":
        len(dimension_summary),

    "Details":
        "Qualitative evidence-based dimensions"

})


review_count = (

    len(unmapped_check)

    if "Final_ROC_Theme"
    in unmapped_check.columns

    else 0

)


quality_rows.append({

    "Quality_Check":
        "Themes requiring review",

    "Result":
        review_count,

    "Details":
        "Themes assigned to Review Required"

})


duplicate_count = int(
    final_roc_evidence[
        "Final_ROC_Theme"
    ]
    .str.lower()
    .duplicated()
    .sum()
)


quality_rows.append({

    "Quality_Check":
        "Duplicate theme labels",

    "Result":
        duplicate_count,

    "Details":
        "Case-insensitive duplicate labels"

})


quality_checks_new = pd.DataFrame(
    quality_rows
)


# ================================================================
# 22. LOAD SUPPORTING SHEETS
# ================================================================

def get_sheet_or_empty(sheet):

    if sheet is not None:
        return sheets[sheet].copy()

    return pd.DataFrame()


original_responses = get_sheet_or_empty(
    original_sheet
)

raw_themes = get_sheet_or_empty(
    raw_sheet
)

cleaned_themes = get_sheet_or_empty(
    cleaned_sheet
)

theme_summary_existing = get_sheet_or_empty(
    theme_sheet
)

normalization_summary = get_sheet_or_empty(
    normalization_sheet
)

decision_summary = get_sheet_or_empty(
    decision_sheet
)

participant_coverage = get_sheet_or_empty(
    coverage_sheet
)

construct_statistics = get_sheet_or_empty(
    statistics_sheet
)


# ================================================================
# 23. WRITE OUTPUT
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"ROC_FINAL_EVIDENCE_AND_DIMENSIONS_{timestamp}.xlsx"
)


with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    original_responses.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )


    raw_themes.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )


    cleaned_themes.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )


    if coding_audit is not None:

        coding_audit.to_excel(
            writer,
            sheet_name="04_Coding_Audit",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="04_Coding_Audit",
            index=False
        )


    if participant_matrix is not None:

        participant_matrix.to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )


    theme_summary_existing.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )


    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )


    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )


    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )


    unmapped_check.to_excel(
        writer,
        sheet_name="10_Unmapped_Check",
        index=False
    )


    decision_check.to_excel(
        writer,
        sheet_name="11_Decision_Check",
        index=False
    )


    quality_checks_new.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )


    final_roc_evidence.to_excel(
        writer,
        sheet_name="13_Final_ROC_Evidence",
        index=False
    )


    dimension_summary.to_excel(
        writer,
        sheet_name="14_ROC_Dimension_Summary",
        index=False
    )


    dimension_themes.to_excel(
        writer,
        sheet_name="15_ROC_Dimension_Themes",
        index=False
    )


    construct_statistics.to_excel(
        writer,
        sheet_name="16_Construct_Statistics",
        index=False
    )


# ================================================================
# 24. FINAL REPORT
# ================================================================

print("\n")
print("=" * 80)
print("ROC PROCESS COMPLETED SUCCESSFULLY")
print("=" * 80)

print("\nOutput file:")
print(
    OUTPUT_FILE.resolve()
)


print("\nNumber of final ROC themes:")
print(
    len(final_roc_evidence)
)


print("\nNumber of ROC dimensions:")
print(
    len(dimension_summary)
)


print("\n")
print("-" * 80)
print("ROC DIMENSION SUMMARY")
print("-" * 80)

print(
    dimension_summary.to_string(
        index=False
    )
)


print("\n")
print("-" * 80)
print("QUALITY CHECKS")
print("-" * 80)

print(
    quality_checks_new.to_string(
        index=False
    )
)


print("\n")
print("=" * 80)
print("IMPORTANT")
print("=" * 80)

print("""
The two sheets you should use for the next questionnaire stage are:

14_ROC_Dimension_Summary
15_ROC_Dimension_Themes

They connect your qualitative evidence to questionnaire-item
development.

The dimensions are NOT automatically treated as separate
statistical constructs. They are qualitative evidence-based
groupings that help us determine what aspects of ROC should
be represented in the questionnaire.

The next stage is to convert the strongest, non-overlapping
ROC themes into candidate reflective questionnaire items.
""")

ROC FINAL QUALITATIVE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Decision_Intelligence_Risk_Coordination_Supply_Chain_Resilience_1\ROC_FINAL_QUALITATIVE_ANALYSIS_20260827_131920.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Audit
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Normalization_Summary
 - 08_Decision_Summary
 - 09_Participant_Coverage
 - 10_Construct_Statistics
 - 11_Final_ROC_Evidence
 - 12_Quality_Checks
 - 13_Unmapped_Themes

Participants used: 26


ROC PROCESS COMPLETED SUCCESSFULLY

Output file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Decision_Intelligence_Risk_Coordination_Supply_Chain_Resilience_1\ROC_FINAL_EVIDENCE_AND_DIMENSIONS_20260827_132024.xlsx

Number of final ROC themes:
62

Number of ROC dimensions:
6


--------------------------------------------------------------------------------
ROC DIMENSIO

## Risk Coordination Capability (RCC)

In [11]:
# ================================================================
# RCC CODING
# INPUT FILE: Themes.xlsx
# ================================================================

import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# ------------------------------------------------
# 1. INPUT
# ------------------------------------------------

input_file = Path("Themes.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"Themes.xlsx was not found in:\n{Path.cwd()}"
    )

excel = pd.ExcelFile(input_file)


# ------------------------------------------------
# 2. FIND PARTICIPANT SHEETS
# ------------------------------------------------

participant_sheets = []

for sheet in excel.sheet_names:

    match = re.fullmatch(
        r"P?\s*0*(\d+)",
        str(sheet).strip(),
        flags=re.IGNORECASE
    )

    if match:

        number = int(match.group(1))

        if 1 <= number <= 26:
            participant_sheets.append(
                (number, sheet)
            )

participant_sheets = sorted(
    participant_sheets,
    key=lambda x: x[0]
)

print("Participant sheets found:")

for number, sheet in participant_sheets:
    print(f"P{number:02d} -> {sheet}")


# ------------------------------------------------
# 3. EXTRACT RCC RESPONSES
# ------------------------------------------------

raw_rows = []

for number, sheet in participant_sheets:

    participant = f"P{number:02d}"

    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        header=None
    )

    for row_number, row in df.iterrows():

        values = []

        for value in row.tolist():

            if pd.isna(value):
                continue

            text = str(value).strip()

            if text != "":
                values.append(text)

        if len(values) < 2:
            continue

        construct_position = None

        for position, value in enumerate(values):

            if value.strip().upper() == "RCC":

                construct_position = position
                break

        if construct_position is None:
            continue

        remaining = values[
            construct_position + 1:
        ]

        if not remaining:
            continue

        response = " ".join(
            remaining
        ).strip()

        if response:

            raw_rows.append({

                "Participant":
                    participant,

                "Construct":
                    "RCC",

                "Original_Response":
                    response,

                "Source_Sheet":
                    sheet,

                "Source_Row":
                    row_number + 1
            })


# ------------------------------------------------
# 4. CHECK
# ------------------------------------------------

original_df = pd.DataFrame(
    raw_rows
)

if original_df.empty:

    raise ValueError(
        """
        No RCC responses were found.

        Check that RCC appears in the participant sheets.
        """
    )

print("\nRCC original responses:")
print(len(original_df))


# ------------------------------------------------
# 5. SPLIT INTO RAW THEMES
# ------------------------------------------------

theme_rows = []

for _, row in original_df.iterrows():

    response = str(
        row["Original_Response"]
    )

    response = response.replace(
        "\n", ","
    )

    response = response.replace(
        ";", ","
    )

    response = response.replace(
        "•", ","
    )

    themes = response.split(",")

    for theme in themes:

        theme = str(
            theme
        ).strip()

        if theme == "":
            continue

        theme_rows.append({

            "Participant":
                row["Participant"],

            "Construct":
                "RCC",

            "Raw_Theme":
                theme,

            "Source_Sheet":
                row["Source_Sheet"],

            "Source_Row":
                row["Source_Row"]
        })


raw_df = pd.DataFrame(
    theme_rows
)


# ------------------------------------------------
# 6. CLEAN THEMES
# ------------------------------------------------

clean_df = raw_df.copy()

clean_df["Theme_Clean"] = (
    clean_df["Raw_Theme"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

clean_df["Theme_Key"] = (
    clean_df["Theme_Clean"]
    .str.lower()
    .str.strip()
)

clean_df = clean_df[
    clean_df["Theme_Key"] != ""
]

clean_df = clean_df.drop_duplicates(
    subset=[
        "Participant",
        "Construct",
        "Theme_Key"
    ]
)

clean_df = clean_df.reset_index(
    drop=True
)

print("\nRCC cleaned theme observations:")
print(len(clean_df))


# ------------------------------------------------
# 7. DISPLAY RAW RCC THEMES
# ------------------------------------------------

raw_theme_list = (
    clean_df[
        [
            "Raw_Theme",
            "Theme_Clean",
            "Theme_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Theme_Key"
    )
    .reset_index(
        drop=True
    )
)

print("\n")
print("=" * 60)
print("RCC RAW THEMES")
print("=" * 60)

print(
    raw_theme_list.to_string(
        index=False
    )
)


# ------------------------------------------------
# 8. RCC NORMALIZATION DICTIONARY
# ------------------------------------------------
#
# Add researcher-approved RCC mappings here after reviewing
# the actual RCC raw themes.
#
# Example format:
#
# "coordinated risk actions":
#     "Coordinated risk response",
#
# "joint risk response":
#     "Coordinated risk response",
#
# ------------------------------------------------

RCC_NORMALIZATION = {

    # ADD RCC MAPPINGS HERE

}


# ------------------------------------------------
# 9. APPLY NORMALIZATION
# ------------------------------------------------

coding_df = raw_theme_list.copy()

coding_df["Normalized_Theme"] = ""

coding_df["Decision"] = ""

coding_df["Reason"] = ""


for i, row in coding_df.iterrows():

    theme_key = str(
        row["Theme_Key"]
    ).strip().lower()

    if theme_key in RCC_NORMALIZATION:

        normalized = RCC_NORMALIZATION[
            theme_key
        ]

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = normalized

        if theme_key == normalized.lower():

            coding_df.loc[
                i,
                "Decision"
            ] = "Keep"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Retained as a distinct conceptual theme."
            )

        else:

            coding_df.loc[
                i,
                "Decision"
            ] = "Merge"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Merged with a semantically equivalent "
                "theme."
            )

    else:

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = row["Theme_Clean"]

        coding_df.loc[
            i,
            "Decision"
        ] = "Keep"

        coding_df.loc[
            i,
            "Reason"
        ] = (
            "Retained pending conceptual review."
        )


# ------------------------------------------------
# 10. MAP TO PARTICIPANTS
# ------------------------------------------------

mapping = coding_df[
    [
        "Theme_Key",
        "Normalized_Theme",
        "Decision"
    ]
]

coded_df = clean_df.merge(
    mapping,
    on="Theme_Key",
    how="left"
)


# ------------------------------------------------
# 11. PARTICIPANT × RCC THEME MATRIX
# ------------------------------------------------

participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

matrix_source = (
    coded_df
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
)

matrix = pd.crosstab(
    matrix_source["Normalized_Theme"],
    matrix_source["Participant"]
)

matrix = matrix.reindex(
    columns=participants,
    fill_value=0
)

matrix = matrix.reset_index()

matrix["Frequency"] = matrix[
    participants
].sum(axis=1)

matrix["Percentage"] = (
    matrix["Frequency"]
    / len(participants)
    * 100
).round(1)


# ------------------------------------------------
# 12. THEME SUMMARY
# ------------------------------------------------

theme_summary = matrix[
    [
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].sort_values(
    "Frequency",
    ascending=False
)


# ------------------------------------------------
# 13. DECISION SUMMARY
# ------------------------------------------------

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Count"
    )
)


# ------------------------------------------------
# 14. PARTICIPANT COVERAGE
# ------------------------------------------------

participant_coverage = (
    coded_df
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "RCC_Theme_Count"
]


# ------------------------------------------------
# 15. SAVE EXCEL
# ------------------------------------------------

output_file = Path(
    "RCC_Coding_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".xlsx"
)

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_df.to_excel(
        writer,
        sheet_name="04_Coding_Dictionary",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="07_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="08_Participant_Coverage",
        index=False
    )


# ------------------------------------------------
# 16. FINAL REPORT
# ------------------------------------------------

print("\n")
print("=" * 60)
print("RCC CODING COMPLETED")
print("=" * 60)

print(
    "Participants:",
    clean_df["Participant"].nunique()
)

print(
    "Raw theme observations:",
    len(clean_df)
)

print(
    "Unique RCC raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Normalized RCC themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print("\nDECISION SUMMARY:")

print(
    decision_summary.to_string(
        index=False
    )
)

print("\nOUTPUT FILE:")

print(
    output_file.resolve()
)

print("\nDONE.")

Participant sheets found:
P01 -> 1
P02 -> 2
P03 -> 3
P04 -> 4
P05 -> 5
P06 -> 6
P07 -> 7
P08 -> 8
P09 -> 9
P10 -> 10
P11 -> 11
P12 -> 12
P13 -> 13
P14 -> 14
P15 -> 15
P16 -> 16
P17 -> 17
P18 -> 18
P19 -> 19
P20 -> 20
P21 -> 21
P22 -> 22
P23 -> 23
P24 -> 24
P25 -> 25
P26 -> 26

RCC original responses:
26

RCC cleaned theme observations:
94


RCC RAW THEMES
                      Raw_Theme                     Theme_Clean                       Theme_Key
               Action agreement                Action agreement                action agreement
               action alignment                action alignment                action alignment
                 action clarity                  action clarity                  action clarity
                      alignment                       alignment                       alignment
                      authority                       authority                       authority
            changing priorities             changing priorities   

In [12]:
# ================================================================
# COMPLETE RCC QUALITATIVE CODING ANALYSIS
# Input: RCC_Coding_20260827_132134.xlsx
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# ================================================================
# 1. INPUT FILE
# ================================================================

input_file = Path("RCC_Coding_20260827_132134.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"\nFile not found:\n{input_file.resolve()}\n\n"
        "Make sure the Excel file is in the same folder as this notebook."
    )

print("=" * 70)
print("RCC QUALITATIVE CODING ANALYSIS")
print("=" * 70)

excel = pd.ExcelFile(input_file)

print("\nInput file:", input_file.resolve())
print("\nSheets found:")
for s in excel.sheet_names:
    print(" -", s)


# ================================================================
# 2. FIND SHEETS FLEXIBLY
# ================================================================

def find_sheet(possible_names):

    for name in possible_names:
        if name in excel.sheet_names:
            return name

    for sheet in excel.sheet_names:

        a = (
            str(sheet)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for name in possible_names:

            b = (
                str(name)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if b in a:
                return sheet

    return None


original_sheet = find_sheet([
    "01_Original_RCC",
    "01_Original",
    "Original_RCC",
    "Original"
])

raw_sheet = find_sheet([
    "02_Raw_RCC",
    "02_Raw",
    "Raw_RCC",
    "Raw"
])

clean_sheet = find_sheet([
    "03_Cleaned_RCC",
    "03_Cleaned",
    "Cleaned_RCC",
    "Cleaned"
])

coding_sheet = find_sheet([
    "04_Normalized_Coding",
    "04_Coding",
    "Normalized_Coding",
    "Coding"
])


print("\nSelected sheets:")
print("Original:", original_sheet)
print("Raw:", raw_sheet)
print("Cleaned:", clean_sheet)
print("Coding:", coding_sheet)


# ================================================================
# 3. READ SHEETS
# ================================================================

original_df = (
    pd.read_excel(input_file, sheet_name=original_sheet)
    if original_sheet else pd.DataFrame()
)

raw_df = (
    pd.read_excel(input_file, sheet_name=raw_sheet)
    if raw_sheet else pd.DataFrame()
)

clean_df = (
    pd.read_excel(input_file, sheet_name=clean_sheet)
    if clean_sheet else pd.DataFrame()
)

if coding_sheet is None:
    raise ValueError(
        "\nCoding/Normalized Coding sheet could not be found.\n\n"
        "Available sheets:\n" +
        "\n".join(str(x) for x in excel.sheet_names)
    )

coding_df = pd.read_excel(
    input_file,
    sheet_name=coding_sheet
)


# ================================================================
# 4. STANDARDIZE COLUMN NAMES
# ================================================================

for df in [
    original_df,
    raw_df,
    clean_df,
    coding_df
]:
    if not df.empty:
        df.columns = [
            str(c).strip()
            for c in df.columns
        ]


print("\nCoding columns:")
print(list(coding_df.columns))


# ================================================================
# 5. FIND COLUMNS
# ================================================================

def find_column(df, candidates):

    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    for column in df.columns:

        a = (
            str(column)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for candidate in candidates:

            b = (
                str(candidate)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if a == b:
                return column

    return None


theme_key_col = find_column(
    coding_df,
    ["Theme_Key", "Theme Key", "ThemeKey"]
)

raw_theme_col = find_column(
    coding_df,
    ["Raw_Theme", "Raw Theme", "RawTheme"]
)

normalized_col = find_column(
    coding_df,
    [
        "Normalized_Theme",
        "Normalized Theme",
        "NormalizedTheme"
    ]
)

decision_col = find_column(
    coding_df,
    ["Decision"]
)


if theme_key_col is None:
    raise ValueError("Theme_Key column not found.")

if raw_theme_col is None:
    raise ValueError("Raw_Theme column not found.")

if normalized_col is None:
    raise ValueError("Normalized_Theme column not found.")

if decision_col is None:
    raise ValueError("Decision column not found.")


coding_df = coding_df.rename(
    columns={
        theme_key_col: "Theme_Key",
        raw_theme_col: "Raw_Theme",
        normalized_col: "Normalized_Theme",
        decision_col: "Decision"
    }
)


# ================================================================
# 6. PARTICIPANT COLUMN
# ================================================================

participant_col = find_column(
    clean_df,
    [
        "Participant",
        "Participant_ID",
        "Participant ID",
        "ParticipantID"
    ]
)

if participant_col is None:

    raise ValueError(
        "\nParticipant column not found.\n\n"
        "Available columns:\n" +
        str(list(clean_df.columns))
    )

if participant_col != "Participant":

    clean_df = clean_df.rename(
        columns={
            participant_col: "Participant"
        }
    )


# ================================================================
# 7. THEME KEY IN CLEANED DATA
# ================================================================

clean_theme_key = find_column(
    clean_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

if clean_theme_key is None:

    raise ValueError(
        "\nTheme_Key not found in Cleaned sheet.\n\n"
        "Available columns:\n" +
        str(list(clean_df.columns))
    )

if clean_theme_key != "Theme_Key":

    clean_df = clean_df.rename(
        columns={
            clean_theme_key: "Theme_Key"
        }
    )


# ================================================================
# 8. CLEAN CODING DICTIONARY
# ================================================================

coding_df["Theme_Key"] = (
    coding_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

coding_df["Raw_Theme"] = (
    coding_df["Raw_Theme"]
    .fillna("")
    .astype(str)
    .str.strip()
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .replace(
        ["nan", "None", "", "NaN"],
        np.nan
    )
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .astype("string")
    .str.strip()
)

coding_df["Decision"] = (
    coding_df["Decision"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 9. CLEAN PARTICIPANT DATA
# ================================================================

clean_df["Theme_Key"] = (
    clean_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

clean_df["Participant"] = (
    clean_df["Participant"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 10. PARTICIPANTS
# ================================================================

participants = sorted(
    clean_df["Participant"]
    .replace("", np.nan)
    .dropna()
    .unique()
)

print("\nNumber of participants:", len(participants))
print("Participants:", participants)


# ================================================================
# 11. MERGE CLEANED THEMES WITH CODING DICTIONARY
# ================================================================

coded_df = clean_df.merge(
    coding_df[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision"
        ]
    ],
    on="Theme_Key",
    how="left"
)


# ================================================================
# 12. UNMAPPED THEMES
# ================================================================

missing_mapping = coded_df[
    coded_df["Normalized_Theme"].isna()
].copy()

print(
    "\nNumber of unmapped themes:",
    len(missing_mapping)
)

if len(missing_mapping) > 0:

    print("\nUnmapped themes:")

    print(
        missing_mapping[
            [
                "Participant",
                "Theme_Key"
            ]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )


# ================================================================
# 13. PARTICIPANT × NORMALIZED RCC THEMES
# ================================================================

participant_theme = (
    coded_df[
        [
            "Participant",
            "Normalized_Theme"
        ]
    ]
    .dropna()
    .drop_duplicates()
)


# ================================================================
# 14. PARTICIPANT × THEME MATRIX
# ================================================================

matrix = pd.crosstab(
    participant_theme["Normalized_Theme"],
    participant_theme["Participant"]
)


# ================================================================
# 15. ORDER PARTICIPANTS P01–P26
# ================================================================

expected_participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

existing = [
    p
    for p in expected_participants
    if p in matrix.columns
]

other = [
    p
    for p in matrix.columns
    if p not in existing
]

matrix = matrix.reindex(
    columns=existing + other,
    fill_value=0
)

matrix = matrix.reset_index()

participant_columns = existing + other


# ================================================================
# 16. FREQUENCY
# ================================================================

matrix["Frequency"] = (
    matrix[participant_columns]
    .sum(axis=1)
)


# ================================================================
# 17. PERCENTAGE OF EXPERTS
# ================================================================

total_participants = len(participants)

if total_participants > 0:

    matrix["Percentage"] = (
        matrix["Frequency"]
        / total_participants
        * 100
    ).round(1)

else:

    matrix["Percentage"] = 0


# ================================================================
# 18. RANK THEMES
# ================================================================

matrix = matrix.sort_values(
    [
        "Frequency",
        "Normalized_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)

matrix.insert(
    0,
    "Rank",
    range(1, len(matrix) + 1)
)


# ================================================================
# 19. RCC THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Rank",
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].copy()

theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme": "Final_RCC_Theme",
        "Frequency": "Experts_Mentioning",
        "Percentage": "Percentage_of_Experts"
    }
)


# ================================================================
# 20. PREVALENCE CATEGORY
# ================================================================

def prevalence_category(p):

    if p >= 75:
        return "Very High"

    elif p >= 50:
        return "High"

    elif p >= 25:
        return "Moderate"

    else:
        return "Low"


theme_summary["Prevalence_Category"] = (
    theme_summary["Percentage_of_Experts"]
    .apply(prevalence_category)
)


# ================================================================
# 21. CODING AUDIT
# ================================================================

theme_participant_counts = (
    participant_theme
    .groupby("Normalized_Theme")
    .size()
    .reset_index(
        name="Experts_Mentioning"
    )
)

coding_audit = coding_df.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)

coding_audit["Experts_Mentioning"] = (
    coding_audit["Experts_Mentioning"]
    .fillna(0)
    .astype(int)
)

if total_participants > 0:

    coding_audit["Percentage_of_Experts"] = (
        coding_audit["Experts_Mentioning"]
        / total_participants
        * 100
    ).round(1)

else:

    coding_audit["Percentage_of_Experts"] = 0


coding_audit = coding_audit.sort_values(
    [
        "Normalized_Theme",
        "Decision",
        "Raw_Theme"
    ]
).reset_index(drop=True)


# ================================================================
# 22. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Number_of_Raw_Themes"
    )
)

if len(coding_df) > 0:

    decision_summary["Percentage"] = (
        decision_summary["Number_of_Raw_Themes"]
        / len(coding_df)
        * 100
    ).round(1)

else:

    decision_summary["Percentage"] = 0


# ================================================================
# 23. NORMALIZATION SUMMARY
# ================================================================

normalization_summary = (
    coding_df
    .dropna(subset=["Normalized_Theme"])
    .groupby("Normalized_Theme")
    .agg(
        Raw_Themes=("Raw_Theme", "count"),

        Keep_Count=(
            "Decision",
            lambda x: (x == "Keep").sum()
        ),

        Merge_Count=(
            "Decision",
            lambda x: (x == "Merge").sum()
        )
    )
    .reset_index()
)

normalization_summary = (
    normalization_summary.merge(
        theme_participant_counts,
        on="Normalized_Theme",
        how="left"
    )
)

normalization_summary["Experts_Mentioning"] = (
    normalization_summary["Experts_Mentioning"]
    .fillna(0)
    .astype(int)
)

if total_participants > 0:

    normalization_summary["Percentage_of_Experts"] = (
        normalization_summary["Experts_Mentioning"]
        / total_participants
        * 100
    ).round(1)

else:

    normalization_summary["Percentage_of_Experts"] = 0


normalization_summary = (
    normalization_summary
    .sort_values(
        "Experts_Mentioning",
        ascending=False
    )
    .reset_index(drop=True)
)


# ================================================================
# 24. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    participant_theme
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "Number_of_Normalized_RCC_Themes"
]


# ================================================================
# 25. RCC CONSTRUCT STATISTICS
# ================================================================

construct_statistics = pd.DataFrame({

    "Construct": ["RCC"],

    "Participants": [
        total_participants
    ],

    "Original_Response_Rows": [
        len(original_df)
    ],

    "Raw_Theme_Observations": [
        len(raw_df)
    ],

    "Cleaned_Theme_Observations": [
        len(clean_df)
    ],

    "Unique_Raw_Themes": [
        coding_df["Theme_Key"].nunique()
    ],

    "Final_Normalized_Themes": [
        coding_df["Normalized_Theme"].nunique()
    ],

    "Merged_Raw_Themes": [
        (
            coding_df["Decision"] == "Merge"
        ).sum()
    ],

    "Kept_Raw_Themes": [
        (
            coding_df["Decision"] == "Keep"
        ).sum()
    ],

    "Unmapped_Themes": [
        len(missing_mapping)
    ]
})


# ================================================================
# 26. FINAL RCC EVIDENCE
# ================================================================

final_evidence = normalization_summary[
    [
        "Normalized_Theme",
        "Raw_Themes",
        "Experts_Mentioning",
        "Percentage_of_Experts",
        "Keep_Count",
        "Merge_Count"
    ]
].copy()

final_evidence = final_evidence.rename(
    columns={
        "Normalized_Theme": "Final_RCC_Theme",
        "Raw_Themes": "Number_of_Raw_Themes",
        "Percentage_of_Experts": "Expert_Prevalence_%",
        "Keep_Count": "Raw_Themes_Kept",
        "Merge_Count": "Raw_Themes_Merged"
    }
)

final_evidence["Prevalence_Category"] = (
    final_evidence["Expert_Prevalence_%"]
    .apply(prevalence_category)
)


# ================================================================
# 27. QUALITY CHECKS
# ================================================================

duplicate_count = (
    len(coded_df)
    -
    len(
        coded_df[
            [
                "Participant",
                "Normalized_Theme"
            ]
        ]
        .dropna()
        .drop_duplicates()
    )
)

quality_checks = pd.DataFrame({

    "Check": [

        "Number of participants",
        "Original response rows",
        "Raw theme observations",
        "Unique raw themes",
        "Final normalized themes",
        "Unmapped themes",
        "Duplicate participant-theme records",
        "Keep decisions",
        "Merge decisions"

    ],

    "Result": [

        total_participants,
        len(original_df),
        len(raw_df),

        coding_df["Theme_Key"].nunique(),

        coding_df["Normalized_Theme"].nunique(),

        len(missing_mapping),

        duplicate_count,

        (
            coding_df["Decision"] == "Keep"
        ).sum(),

        (
            coding_df["Decision"] == "Merge"
        ).sum()

    ],

    "Status": [

        "PASS" if total_participants > 0 else "CHECK",

        "PASS" if len(original_df) > 0 else "CHECK",

        "PASS" if len(raw_df) > 0 else "CHECK",

        "PASS"
        if coding_df["Theme_Key"].nunique() > 0
        else "CHECK",

        "PASS"
        if coding_df["Normalized_Theme"].nunique() > 0
        else "CHECK",

        "PASS"
        if len(missing_mapping) == 0
        else "CHECK",

        "INFO",
        "PASS",
        "PASS"
    ]
})


# ================================================================
# 28. SAVE COMPLETE RCC WORKBOOK
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_file = Path(
    f"RCC_FINAL_QUALITATIVE_ANALYSIS_{timestamp}.xlsx"
)

counter = 1

while output_file.exists():

    output_file = Path(
        f"RCC_FINAL_QUALITATIVE_ANALYSIS_"
        f"{timestamp}_{counter}.xlsx"
    )

    counter += 1


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_RCC_Evidence",
        index=False
    )

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    missing_mapping.to_excel(
        writer,
        sheet_name="13_Unmapped_Themes",
        index=False
    )


# ================================================================
# 29. FINAL REPORT
# ================================================================

print("\n")
print("=" * 70)
print("RCC ANALYSIS COMPLETED")
print("=" * 70)

print("\nParticipants:", total_participants)

print("Original responses:", len(original_df))

print("Raw theme observations:", len(raw_df))

print("Cleaned theme observations:", len(clean_df))

print(
    "Unique raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Final normalized RCC themes:",
    coding_df["Normalized_Theme"].nunique()
)

print(
    "Keep decisions:",
    (
        coding_df["Decision"] == "Keep"
    ).sum()
)

print(
    "Merge decisions:",
    (
        coding_df["Decision"] == "Merge"
    ).sum()
)

print(
    "Unmapped themes:",
    len(missing_mapping)
)


# ================================================================
# 30. DISPLAY FINAL RCC THEMES
# ================================================================

print("\n")
print("=" * 70)
print("FINAL RCC THEMES")
print("=" * 70)

print(
    final_evidence[
        [
            "Final_RCC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Prevalence_Category"
        ]
    ].to_string(index=False)
)


# ================================================================
# 31. OUTPUT LOCATION
# ================================================================

print("\n")
print("=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(output_file.resolve())

print(
    "\nComplete RCC qualitative analysis workbook "
    "created successfully."
)

RCC QUALITATIVE CODING ANALYSIS

Input file: C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Decision_Intelligence_Risk_Coordination_Supply_Chain_Resilience_1\RCC_Coding_20260827_132134.xlsx

Sheets found:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Dictionary
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Decision_Summary
 - 08_Participant_Coverage

Selected sheets:
Original: 01_Original_Responses
Raw: 02_Raw_Themes
Cleaned: 03_Cleaned_Themes
Coding: 04_Coding_Dictionary

Coding columns:
['Raw_Theme', 'Theme_Clean', 'Theme_Key', 'Normalized_Theme', 'Decision', 'Reason']

Number of participants: 26
Participants: ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26']

Number of unmapped themes: 0


RCC ANALYSIS COMPLETED

Participants: 26
Original responses: 26
Raw theme observations: 94
Cl

In [ ]:
# ================================================================
# RCC — FINAL QUALITATIVE EVIDENCE + DIMENSION ANALYSIS
# ================================================================
#
# INPUT:
# RCC_FINAL_QUALITATIVE_ANALYSIS_20260827_132212.xlsx
#
# OUTPUT:
# RCC_FINAL_EVIDENCE_AND_DIMENSIONS_20260827_132311.xlsx
#
# PURPOSE:
# 1. Preserve the completed RCC qualitative coding
# 2. Extract final RCC evidence
# 3. Organize themes into evidence-based RCC dimensions
# 4. Produce dimension/theme tables for questionnaire development
# 5. Perform quality checks
#
# IMPORTANT:
# Dimensions generated here are qualitative groupings.
# They are NOT automatically treated as separate constructs.
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import os


# ================================================================
# 1. LOCATE INPUT FILE
# ================================================================

TARGET = "RCC_FINAL_QUALITATIVE_ANALYSIS_20260827_132212"

search_locations = [
    Path("/mnt/data"),
    Path("."),
    Path.home(),
    Path.home() / "Downloads",
    Path.home() / "Documents",
    Path.home() / "Desktop"
]

possible_files = []

for location in search_locations:

    if not location.exists():
        continue

    try:

        for ext in [".xlsx", ".xlsm", ".xls"]:

            possible_files.extend(
                location.rglob(TARGET + ext)
            )

    except Exception:
        pass


# Remove duplicates
possible_files = list(
    dict.fromkeys(
        [p.resolve() for p in possible_files]
    )
)


# ---------------------------------------------------------------
# Exact filename first
# ---------------------------------------------------------------

exact_files = [
    p for p in possible_files
    if p.stem == TARGET
]


if len(exact_files) > 0:

    INPUT_FILE = exact_files[0]


else:

    # -----------------------------------------------------------
    # Uploaded file may have been renamed by the system.
    # Identify it using the RCC-specific evidence sheet.
    # -----------------------------------------------------------

    rcc_candidates = []

    for p in possible_files:

        try:

            xl_test = pd.ExcelFile(
                p,
                engine="openpyxl"
            )

            if "11_Final_RCC_Evidence" in xl_test.sheet_names:

                rcc_candidates.append(p)

        except Exception:
            pass


    if len(rcc_candidates) > 0:

        INPUT_FILE = rcc_candidates[0]

    else:

        raise FileNotFoundError(
            "\nRCC input workbook could not be found.\n"
            "Expected:\n"
            f"{TARGET}.xlsx"
        )


print("=" * 85)
print("RCC FINAL QUALITATIVE ANALYSIS")
print("=" * 85)

print("\nInput file:")
print(INPUT_FILE)


# ================================================================
# 2. READ WORKBOOK
# ================================================================

xls = pd.ExcelFile(
    INPUT_FILE,
    engine="openpyxl"
)

print("\nAvailable sheets:")

for s in xls.sheet_names:
    print(" -", s)


# ================================================================
# 3. LOAD ALL SHEETS
# ================================================================

sheets = {}

for sheet in xls.sheet_names:

    try:

        sheets[sheet] = pd.read_excel(
            INPUT_FILE,
            sheet_name=sheet
        )

    except Exception as e:

        print(
            f"Warning: could not read {sheet}: {e}"
        )


# ================================================================
# 4. HELPER FUNCTIONS
# ================================================================

def find_sheet(keyword):

    for s in sheets.keys():

        if keyword.lower() in s.lower():

            return s

    return None


def clean_text(x):

    if pd.isna(x):

        return ""

    x = str(x).strip()

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x


def prevalence_category(x):

    try:

        x = float(x)

    except:

        return "Not available"


    if x >= 75:

        return "Very High"

    elif x >= 50:

        return "High"

    elif x >= 25:

        return "Moderate"

    else:

        return "Low"


# ================================================================
# 5. IDENTIFY ALL SUPPORTING SHEETS
# ================================================================

original_sheet = find_sheet(
    "01_Original_Responses"
)

raw_sheet = find_sheet(
    "02_Raw_Themes"
)

cleaned_sheet = find_sheet(
    "03_Cleaned_Themes"
)

audit_sheet = find_sheet(
    "04_Coding_Audit"
)

matrix_sheet = find_sheet(
    "05_Participant_Matrix"
)

theme_sheet = find_sheet(
    "06_Theme_Summary"
)

normalization_sheet = find_sheet(
    "07_Normalization_Summary"
)

decision_sheet = find_sheet(
    "08_Decision_Summary"
)

coverage_sheet = find_sheet(
    "09_Participant_Coverage"
)

statistics_sheet = find_sheet(
    "10_Construct_Statistics"
)

final_sheet = find_sheet(
    "11_Final_RCC_Evidence"
)

quality_sheet = find_sheet(
    "12_Quality_Checks"
)

unmapped_sheet = find_sheet(
    "13_Unmapped_Themes"
)


if final_sheet is None:

    raise ValueError(
        "11_Final_RCC_Evidence was not found."
    )


# ================================================================
# 6. LOAD FINAL RCC EVIDENCE
# ================================================================

final_evidence = sheets[
    final_sheet
].copy()


final_evidence.columns = [
    str(c).strip()
    for c in final_evidence.columns
]


# ================================================================
# 7. IDENTIFY RCC THEME COLUMN
# ================================================================

theme_col = None


if "Final_RCC_Theme" in final_evidence.columns:

    theme_col = "Final_RCC_Theme"


else:

    for c in final_evidence.columns:

        cstr = str(c)

        if (
            "RCC" in cstr
            and "Theme" in cstr
        ):

            theme_col = c

            break


# Fallback
if theme_col is None:

    for c in final_evidence.columns:

        if "Theme" in str(c):

            theme_col = c

            break


if theme_col is None:

    raise ValueError(
        "Could not identify the final RCC theme column."
    )


final_evidence[
    "Final_RCC_Theme"
] = (
    final_evidence[
        theme_col
    ]
    .apply(clean_text)
)


# ================================================================
# 8. STANDARDIZE EXPERT COLUMNS
# ================================================================

if "Experts_Mentioning" not in final_evidence.columns:

    for c in final_evidence.columns:

        cstr = str(c)

        if (
            "Experts" in cstr
            and "Mention" in cstr
        ):

            final_evidence[
                "Experts_Mentioning"
            ] = final_evidence[c]

            break


if "Expert_Prevalence_%" not in final_evidence.columns:

    for c in final_evidence.columns:

        cstr = str(c)

        if (
            "Prevalence" in cstr
            and "%" in cstr
        ):

            final_evidence[
                "Expert_Prevalence_%"
            ] = final_evidence[c]

            break


# ================================================================
# 9. REMOVE EMPTY THEMES
# ================================================================

final_evidence = final_evidence[
    final_evidence[
        "Final_RCC_Theme"
    ] != ""
].copy()


# ================================================================
# 10. PARTICIPANT MATRIX
# ================================================================

participant_matrix = None


if matrix_sheet is not None:

    participant_matrix = sheets[
        matrix_sheet
    ].copy()


participants = []


if participant_matrix is not None:

    for c in participant_matrix.columns:

        cstr = str(c).strip()

        if re.match(
            r"^P\d+$",
            cstr,
            flags=re.IGNORECASE
        ):

            participants.append(
                cstr
            )


# If participant columns cannot be detected,
# use the known study participant count.
if len(participants) > 0:

    n_participants = len(
        participants
    )

else:

    n_participants = 26


print(
    "\nParticipants used:",
    n_participants
)


# ================================================================
# 11. PREVALENCE CATEGORY
# ================================================================

if "Expert_Prevalence_%" in final_evidence.columns:

    final_evidence[
        "Prevalence_Category"
    ] = (
        final_evidence[
            "Expert_Prevalence_%"
        ]
        .apply(
            prevalence_category
        )
    )


# ================================================================
# 12. SORT FINAL EVIDENCE
# ================================================================

if "Experts_Mentioning" in final_evidence.columns:

    final_evidence[
        "_Sort_Experts"
    ] = pd.to_numeric(
        final_evidence[
            "Experts_Mentioning"
        ],
        errors="coerce"
    ).fillna(0)


    final_evidence = (
        final_evidence
        .sort_values(
            "_Sort_Experts",
            ascending=False
        )
        .drop(
            columns=["_Sort_Experts"]
        )
        .reset_index(drop=True)
    )


# ================================================================
# 13. CREATE FINAL RCC EVIDENCE TABLE
# ================================================================

preferred_columns = [

    "Final_RCC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Raw_Themes_Kept",

    "Raw_Themes_Merged",

    "Prevalence_Category"

]


evidence_columns = [

    c

    for c in preferred_columns

    if c in final_evidence.columns

]


final_rcc_evidence = final_evidence[
    evidence_columns
].copy()


# Avoid duplicate Rank errors
if "Rank" in final_rcc_evidence.columns:

    final_rcc_evidence = (
        final_rcc_evidence
        .drop(
            columns=["Rank"]
        )
    )


final_rcc_evidence.insert(
    0,
    "Rank",
    range(
        1,
        len(final_rcc_evidence) + 1
    )
)


# ================================================================
# 14. RCC DIMENSION MAPPING
# ================================================================
#
# The dimensions represent qualitative aspects of RCC.
#
# They are NOT automatically separate constructs.
#
# Their purpose is questionnaire-item development.
# ================================================================

def map_rcc_dimension(theme):

    t = str(theme).strip().lower()


    # ------------------------------------------------------------
    # 1. INFORMATION SHARING & COMMUNICATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "information sharing",
            "information exchange",
            "information flow",
            "information flows",
            "communication",
            "communicate",
            "sharing information",
            "timely information",
            "risk information"

        ]
    ):

        return (
            "Risk Information Sharing & Communication"
        )


    # ------------------------------------------------------------
    # 2. CROSS-FUNCTIONAL COORDINATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "cross-functional",
            "cross functional",
            "functional coordination",
            "department coordination",
            "interdepartmental",
            "between departments",
            "between functions",
            "internal coordination",
            "cross-unit",
            "cross unit"

        ]
    ):

        return (
            "Cross-Functional Risk Coordination"
        )


    # ------------------------------------------------------------
    # 3. SUPPLY-CHAIN PARTNER COORDINATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "supplier coordination",
            "supplier collaboration",
            "supplier cooperation",
            "customer coordination",
            "customer collaboration",
            "partner coordination",
            "partner collaboration",
            "external coordination",
            "supply chain partners",
            "supply-chain partners",
            "supplier",
            "customer",
            "partner"

        ]
    ):

        return (
            "Supply-Chain Partner Coordination"
        )


    # ------------------------------------------------------------
    # 4. JOINT RISK RESPONSE
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "joint response",
            "joint action",
            "collective response",
            "collective action",
            "coordinated response",
            "coordinated action",
            "joint risk",
            "shared response",
            "response coordination"

        ]
    ):

        return (
            "Joint Risk Response"
        )


    # ------------------------------------------------------------
    # 5. RESOURCE & RESPONSIBILITY COORDINATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "resource coordination",
            "resource allocation",
            "resource sharing",
            "resource deployment",
            "responsibility",
            "responsibilities",
            "role clarity",
            "roles",
            "task allocation",
            "allocation of resources",
            "resource"

        ]
    ):

        return (
            "Resource & Responsibility Coordination"
        )


    # ------------------------------------------------------------
    # 6. COORDINATED DECISION-MAKING
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "decision coordination",
            "decision making",
            "decision-making",
            "joint decision",
            "shared decision",
            "collective decision",
            "coordinated decisions",
            "decision alignment",
            "alignment of decisions"

        ]
    ):

        return (
            "Coordinated Risk Decision-Making"
        )


    # ------------------------------------------------------------
    # 7. COORDINATION MONITORING & ALIGNMENT
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "monitor coordination",
            "coordination monitoring",
            "monitoring",
            "tracking",
            "alignment",
            "synchronization",
            "synchronisation",
            "follow-up",
            "follow up",
            "coordination effectiveness"

        ]
    ):

        return (
            "Coordination Monitoring & Alignment"
        )


    # ------------------------------------------------------------
    # 8. REVIEW REQUIRED
    # ------------------------------------------------------------

    return "Review Required"


final_rcc_evidence[
    "RCC_Dimension"
] = (
    final_rcc_evidence[
        "Final_RCC_Theme"
    ]
    .apply(
        map_rcc_dimension
    )
)


# ================================================================
# 15. EXACT THEME OVERRIDES
# ================================================================
#
# These only apply when a final theme exactly matches the
# specified wording.
# ================================================================

manual_mapping = {

    "information sharing":
        "Risk Information Sharing & Communication",

    "information exchange":
        "Risk Information Sharing & Communication",

    "information flow":
        "Risk Information Sharing & Communication",

    "communication":
        "Risk Information Sharing & Communication",

    "cross-functional coordination":
        "Cross-Functional Risk Coordination",

    "cross functional coordination":
        "Cross-Functional Risk Coordination",

    "supplier coordination":
        "Supply-Chain Partner Coordination",

    "supplier collaboration":
        "Supply-Chain Partner Coordination",

    "partner coordination":
        "Supply-Chain Partner Coordination",

    "joint response":
        "Joint Risk Response",

    "joint action":
        "Joint Risk Response",

    "collective response":
        "Joint Risk Response",

    "coordinated response":
        "Joint Risk Response",

    "resource coordination":
        "Resource & Responsibility Coordination",

    "resource allocation":
        "Resource & Responsibility Coordination",

    "role clarity":
        "Resource & Responsibility Coordination",

    "decision coordination":
        "Coordinated Risk Decision-Making",

    "joint decision":
        "Coordinated Risk Decision-Making",

    "shared decision":
        "Coordinated Risk Decision-Making",

    "coordination monitoring":
        "Coordination Monitoring & Alignment",

    "coordination effectiveness":
        "Coordination Monitoring & Alignment"

}


for theme, dimension in manual_mapping.items():

    mask = (
        final_rcc_evidence[
            "Final_RCC_Theme"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq(
            theme.lower()
        )
    )


    final_rcc_evidence.loc[
        mask,
        "RCC_Dimension"
    ] = dimension


# ================================================================
# 16. DIMENSION SUMMARY
# ================================================================

dimension_rows = []


for dimension, group in (
    final_rcc_evidence
    .groupby(
        "RCC_Dimension",
        dropna=False
    )
):

    themes = (
        group[
            "Final_RCC_Theme"
        ]
        .astype(str)
        .tolist()
    )


    experts_mentioning = 0


    # ------------------------------------------------------------
    # Participant matrix method
    # ------------------------------------------------------------

    if (
        participant_matrix is not None
        and len(participants) > 0
    ):

        temp = participant_matrix.copy()


        theme_candidates = [

            c

            for c in temp.columns

            if "theme" in str(c).lower()

        ]


        if len(theme_candidates) > 0:

            matrix_theme_col = (
                theme_candidates[0]
            )

        else:

            matrix_theme_col = (
                temp.columns[0]
            )


        temp["_Theme"] = (
            temp[
                matrix_theme_col
            ]
            .astype(str)
            .str.strip()
        )


        theme_lower = {

            str(x)
            .strip()
            .lower()

            for x in themes

        }


        matched = temp[
            temp[
                "_Theme"
            ]
            .str.lower()
            .isin(
                theme_lower
            )
        ]


        available_participants = [

            p

            for p in participants

            if p in matched.columns

        ]


        if (
            len(matched) > 0
            and len(available_participants) > 0
        ):

            vals = (
                matched[
                    available_participants
                ]
                .apply(
                    pd.to_numeric,
                    errors="coerce"
                )
                .fillna(0)
            )


            experts_mentioning = int(
                (
                    vals.sum(axis=0) > 0
                ).sum()
            )


    # ------------------------------------------------------------
    # Fallback to final evidence
    # ------------------------------------------------------------

    if experts_mentioning == 0:

        if (
            "Experts_Mentioning"
            in group.columns
        ):

            try:

                experts_mentioning = int(
                    pd.to_numeric(
                        group[
                            "Experts_Mentioning"
                        ],
                        errors="coerce"
                    )
                    .max()
                )

            except:

                experts_mentioning = 0


    prevalence = (

        experts_mentioning
        /
        n_participants
        *
        100

        if n_participants > 0

        else 0

    )


    dimension_rows.append({

        "RCC_Dimension":
            dimension,

        "Number_of_Normalized_Themes":
            len(themes),

        "Experts_Mentioning":
            experts_mentioning,

        "Expert_Prevalence_%":
            round(
                prevalence,
                1
            ),

        "Prevalence_Category":
            prevalence_category(
                prevalence
            ),

        "Included_RCC_Themes":
            "; ".join(
                themes
            )

    })


dimension_summary = pd.DataFrame(
    dimension_rows
)


# ================================================================
# 17. SORT DIMENSION SUMMARY
# ================================================================

if len(dimension_summary) > 0:

    dimension_summary[
        "_sort"
    ] = pd.to_numeric(
        dimension_summary[
            "Experts_Mentioning"
        ],
        errors="coerce"
    ).fillna(0)


    dimension_summary = (
        dimension_summary
        .sort_values(
            "_sort",
            ascending=False
        )
        .drop(
            columns=["_sort"]
        )
        .reset_index(drop=True)
    )


    if "Rank" in dimension_summary.columns:

        dimension_summary = (
            dimension_summary
            .drop(
                columns=["Rank"]
            )
        )


    dimension_summary.insert(
        0,
        "Rank",
        range(
            1,
            len(dimension_summary) + 1
        )
    )


# ================================================================
# 18. DIMENSION × THEME TABLE
# ================================================================

dimension_theme_columns = [

    "RCC_Dimension",

    "Final_RCC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Prevalence_Category"

]


dimension_theme_columns = [

    c

    for c in dimension_theme_columns

    if c in final_rcc_evidence.columns

]


dimension_themes = final_rcc_evidence[
    dimension_theme_columns
].copy()


if "Experts_Mentioning" in dimension_themes.columns:

    dimension_themes[
        "_sort"
    ] = pd.to_numeric(
        dimension_themes[
            "Experts_Mentioning"
        ],
        errors="coerce"
    ).fillna(0)


    dimension_themes = (
        dimension_themes
        .sort_values(
            [
                "RCC_Dimension",
                "_sort"
            ],
            ascending=[
                True,
                False
            ]
        )
        .drop(
            columns=["_sort"]
        )
    )


# ================================================================
# 19. THEMES REQUIRING MANUAL REVIEW
# ================================================================

unmapped_check = final_rcc_evidence[
    final_rcc_evidence[
        "RCC_Dimension"
    ]
    ==
    "Review Required"
].copy()


if len(unmapped_check) > 0:

    review_columns = [

        "Final_RCC_Theme",

        "Experts_Mentioning",

        "Expert_Prevalence_%",

        "RCC_Dimension"

    ]


    review_columns = [

        c

        for c in review_columns

        if c in unmapped_check.columns

    ]


    unmapped_check = unmapped_check[
        review_columns
    ].copy()


    unmapped_check.insert(
        0,
        "Status",
        "REVIEW REQUIRED"
    )


else:

    unmapped_check = pd.DataFrame({

        "Status": [
            "PASS — All RCC themes mapped to a dimension."
        ]

    })


# ================================================================
# 20. CODING AUDIT CHECK
# ================================================================

coding_audit = None


if audit_sheet is not None:

    coding_audit = sheets[
        audit_sheet
    ].copy()


if coding_audit is not None:

    decision_columns = [

        c

        for c in coding_audit.columns

        if "Decision" in str(c)

    ]


    if len(decision_columns) > 0:

        decision_col = (
            decision_columns[0]
        )


        missing_decisions = coding_audit[
            coding_audit[
                decision_col
            ].isna()
        ]


        if len(missing_decisions) == 0:

            decision_check = pd.DataFrame({

                "Status": [
                    "PASS — All RCC coding decisions are present."
                ]

            })

        else:

            decision_check = missing_decisions.copy()


    else:

        decision_check = pd.DataFrame({

            "Status": [
                "CHECK — Decision column not detected."
            ]

        })

else:

    decision_check = pd.DataFrame({

        "Status": [
            "CHECK — Coding audit unavailable."
        ]

    })


# ================================================================
# 21. QUALITY CHECKS
# ================================================================

quality_rows = []


quality_rows.append({

    "Quality_Check":
        "Input workbook found",

    "Result":
        "PASS",

    "Details":
        INPUT_FILE.name

})


quality_rows.append({

    "Quality_Check":
        "Final RCC evidence available",

    "Result":
        "PASS"
        if len(final_rcc_evidence) > 0
        else "FAIL",

    "Details":
        f"{len(final_rcc_evidence)} final RCC themes"

})


quality_rows.append({

    "Quality_Check":
        "Participants",

    "Result":
        n_participants,

    "Details":
        "Participant count used for prevalence"

})


quality_rows.append({

    "Quality_Check":
        "RCC dimensions generated",

    "Result":
        len(dimension_summary),

    "Details":
        "Qualitative evidence-based dimensions"

})


review_count = (

    len(unmapped_check)

    if "Final_RCC_Theme"
    in unmapped_check.columns

    else 0

)


quality_rows.append({

    "Quality_Check":
        "Themes requiring manual review",

    "Result":
        review_count,

    "Details":
        "Themes assigned to Review Required"

})


duplicate_count = int(
    final_rcc_evidence[
        "Final_RCC_Theme"
    ]
    .str.lower()
    .duplicated()
    .sum()
)


quality_rows.append({

    "Quality_Check":
        "Duplicate theme labels",

    "Result":
        duplicate_count,

    "Details":
        "Case-insensitive duplicate labels"

})


quality_checks_new = pd.DataFrame(
    quality_rows
)


# ================================================================
# 22. LOAD SUPPORTING SHEETS
# ================================================================

def get_sheet_or_empty(sheet):

    if sheet is not None:

        return sheets[
            sheet
        ].copy()

    return pd.DataFrame()


original_responses = get_sheet_or_empty(
    original_sheet
)

raw_themes = get_sheet_or_empty(
    raw_sheet
)

cleaned_themes = get_sheet_or_empty(
    cleaned_sheet
)

theme_summary_existing = get_sheet_or_empty(
    theme_sheet
)

normalization_summary = get_sheet_or_empty(
    normalization_sheet
)

decision_summary = get_sheet_or_empty(
    decision_sheet
)

participant_coverage = get_sheet_or_empty(
    coverage_sheet
)

construct_statistics = get_sheet_or_empty(
    statistics_sheet
)


# ================================================================
# 23. WRITE FINAL OUTPUT
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"RCC_FINAL_EVIDENCE_AND_DIMENSIONS_{timestamp}.xlsx"
)


with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:


    original_responses.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )


    raw_themes.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )


    cleaned_themes.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )


    if coding_audit is not None:

        coding_audit.to_excel(
            writer,
            sheet_name="04_Coding_Audit",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="04_Coding_Audit",
            index=False
        )


    if participant_matrix is not None:

        participant_matrix.to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )


    theme_summary_existing.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )


    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )


    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )


    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )


    unmapped_check.to_excel(
        writer,
        sheet_name="10_Unmapped_Check",
        index=False
    )


    decision_check.to_excel(
        writer,
        sheet_name="11_Decision_Check",
        index=False
    )


    quality_checks_new.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )


    final_rcc_evidence.to_excel(
        writer,
        sheet_name="13_Final_RCC_Evidence",
        index=False
    )


    dimension_summary.to_excel(
        writer,
        sheet_name="14_RCC_Dimension_Summary",
        index=False
    )


    dimension_themes.to_excel(
        writer,
        sheet_name="15_RCC_Dimension_Themes",
        index=False
    )


    construct_statistics.to_excel(
        writer,
        sheet_name="16_Construct_Statistics",
        index=False
    )


# ================================================================
# 24. DISPLAY RESULTS
# ================================================================

print("\n")
print("=" * 85)
print("RCC PROCESS COMPLETED SUCCESSFULLY")
print("=" * 85)

print("\nOutput file:")
print(
    OUTPUT_FILE.resolve()
)


print("\nFinal RCC themes:")
print(
    len(final_rcc_evidence)
)


print("\nRCC dimensions:")
print(
    len(dimension_summary)
)


print("\n")
print("-" * 85)
print("RCC DIMENSION SUMMARY")
print("-" * 85)

if len(dimension_summary) > 0:

    print(
        dimension_summary.to_string(
            index=False
        )
    )

else:

    print(
        "No dimensions were generated."
    )


print("\n")
print("-" * 85)
print("QUALITY CHECKS")
print("-" * 85)

print(
    quality_checks_new.to_string(
        index=False
    )
)


print("\n")
print("=" * 85)
print("NEXT QUESTIONNAIRE STAGE")
print("=" * 85)

print("""
Use these two sheets for RCC questionnaire development:

14_RCC_Dimension_Summary
15_RCC_Dimension_Themes

The dimensions are qualitative evidence-based groupings.
They should NOT automatically be treated as separate
latent constructs.

The next step is to examine the strongest RCC themes,
remove conceptual overlap, and convert the retained themes
into candidate reflective questionnaire items.
""")

RCC FINAL QUALITATIVE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Decision_Intelligence_Risk_Coordination_Supply_Chain_Resilience_1\RCC_FINAL_QUALITATIVE_ANALYSIS_20260827_132212.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Audit
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Normalization_Summary
 - 08_Decision_Summary
 - 09_Participant_Coverage
 - 10_Construct_Statistics
 - 11_Final_RCC_Evidence
 - 12_Quality_Checks
 - 13_Unmapped_Themes

Participants used: 26


RCC PROCESS COMPLETED SUCCESSFULLY

Output file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Decision_Intelligence_Risk_Coordination_Supply_Chain_Resilience_1\RCC_FINAL_EVIDENCE_AND_DIMENSIONS_20260827_132311.xlsx

Final RCC themes:
64

RCC dimensions:
7


-------------------------------------------------------------------------------------
RCC DIMENSION SUMMARY
-----

## Data Trust & Governance (DTG)

In [14]:
# ================================================================
# DTG CODING
# INPUT FILE: Themes.xlsx
# ================================================================

import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# ------------------------------------------------
# 1. INPUT FILE
# ------------------------------------------------

input_file = Path("Themes.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"Themes.xlsx was not found in:\n{Path.cwd()}"
    )

excel = pd.ExcelFile(input_file)


# ------------------------------------------------
# 2. FIND PARTICIPANT SHEETS P01-P26
# ------------------------------------------------

participant_sheets = []

for sheet in excel.sheet_names:

    match = re.fullmatch(
        r"P?\s*0*(\d+)",
        str(sheet).strip(),
        flags=re.IGNORECASE
    )

    if match:

        number = int(match.group(1))

        if 1 <= number <= 26:
            participant_sheets.append(
                (number, sheet)
            )

participant_sheets = sorted(
    participant_sheets,
    key=lambda x: x[0]
)

print("Participant sheets found:")

for number, sheet in participant_sheets:
    print(f"P{number:02d} -> {sheet}")


# ------------------------------------------------
# 3. EXTRACT DTG RESPONSES
# ------------------------------------------------

raw_rows = []

for number, sheet in participant_sheets:

    participant = f"P{number:02d}"

    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        header=None
    )

    for row_number, row in df.iterrows():

        values = []

        for value in row.tolist():

            if pd.isna(value):
                continue

            text = str(value).strip()

            if text != "":
                values.append(text)

        if len(values) < 2:
            continue

        construct_position = None

        for position, value in enumerate(values):

            if value.strip().upper() == "DTG":

                construct_position = position
                break

        if construct_position is None:
            continue

        remaining = values[
            construct_position + 1:
        ]

        if not remaining:
            continue

        response = " ".join(
            remaining
        ).strip()

        if response:

            raw_rows.append({

                "Participant":
                    participant,

                "Construct":
                    "DTG",

                "Original_Response":
                    response,

                "Source_Sheet":
                    sheet,

                "Source_Row":
                    row_number + 1
            })


# ------------------------------------------------
# 4. CHECK
# ------------------------------------------------

original_df = pd.DataFrame(
    raw_rows
)

if original_df.empty:

    raise ValueError(
        """
        No DTG responses were found.

        Check that DTG appears in the participant sheets.
        """
    )

print("\nDTG original responses:")
print(len(original_df))


# ------------------------------------------------
# 5. SPLIT RESPONSES INTO RAW THEMES
# ------------------------------------------------

theme_rows = []

for _, row in original_df.iterrows():

    response = str(
        row["Original_Response"]
    )

    response = response.replace(
        "\n", ","
    )

    response = response.replace(
        ";", ","
    )

    response = response.replace(
        "•", ","
    )

    themes = response.split(",")

    for theme in themes:

        theme = str(
            theme
        ).strip()

        if theme == "":
            continue

        theme_rows.append({

            "Participant":
                row["Participant"],

            "Construct":
                "DTG",

            "Raw_Theme":
                theme,

            "Source_Sheet":
                row["Source_Sheet"],

            "Source_Row":
                row["Source_Row"]
        })


raw_df = pd.DataFrame(
    theme_rows
)


# ------------------------------------------------
# 6. CLEAN THEMES
# ------------------------------------------------

clean_df = raw_df.copy()

clean_df["Theme_Clean"] = (
    clean_df["Raw_Theme"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

clean_df["Theme_Key"] = (
    clean_df["Theme_Clean"]
    .str.lower()
    .str.strip()
)

clean_df = clean_df[
    clean_df["Theme_Key"] != ""
]

clean_df = clean_df.drop_duplicates(
    subset=[
        "Participant",
        "Construct",
        "Theme_Key"
    ]
)

clean_df = clean_df.reset_index(
    drop=True
)

print("\nDTG cleaned theme observations:")
print(len(clean_df))


# ------------------------------------------------
# 7. DISPLAY RAW DTG THEMES
# ------------------------------------------------

raw_theme_list = (
    clean_df[
        [
            "Raw_Theme",
            "Theme_Clean",
            "Theme_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Theme_Key"
    )
    .reset_index(
        drop=True
    )
)

print("\n")
print("=" * 60)
print("DTG RAW THEMES")
print("=" * 60)

print(
    raw_theme_list.to_string(
        index=False
    )
)


# ------------------------------------------------
# 8. DTG NORMALIZATION DICTIONARY
# ------------------------------------------------
#
# IMPORTANT:
# Do NOT put arbitrary categories here.
#
# After running the code, review the actual DTG RAW THEMES
# and create researcher-approved mappings.
#
# Example format:
#
# "data integrity":
#     "Data integrity",
#
# "accurate shared data":
#     "Data integrity",
#
# "data provenance":
#     "Data provenance",
#
# ------------------------------------------------

DTG_NORMALIZATION = {

    # ADD DTG MAPPINGS HERE

}


# ------------------------------------------------
# 9. APPLY NORMALIZATION
# ------------------------------------------------

coding_df = raw_theme_list.copy()

coding_df["Normalized_Theme"] = ""

coding_df["Decision"] = ""

coding_df["Reason"] = ""


for i, row in coding_df.iterrows():

    theme_key = str(
        row["Theme_Key"]
    ).strip().lower()

    if theme_key in DTG_NORMALIZATION:

        normalized = DTG_NORMALIZATION[
            theme_key
        ]

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = normalized

        if theme_key == normalized.lower():

            coding_df.loc[
                i,
                "Decision"
            ] = "Keep"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Retained as a distinct conceptual theme."
            )

        else:

            coding_df.loc[
                i,
                "Decision"
            ] = "Merge"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Merged with a semantically equivalent "
                "theme."
            )

    else:

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = row["Theme_Clean"]

        coding_df.loc[
            i,
            "Decision"
        ] = "Keep"

        coding_df.loc[
            i,
            "Reason"
        ] = (
            "Retained pending conceptual review."
        )


# ------------------------------------------------
# 10. MAP NORMALIZED THEMES BACK TO PARTICIPANTS
# ------------------------------------------------

mapping = coding_df[
    [
        "Theme_Key",
        "Normalized_Theme",
        "Decision"
    ]
]

coded_df = clean_df.merge(
    mapping,
    on="Theme_Key",
    how="left"
)


# ------------------------------------------------
# 11. PARTICIPANT × DTG THEME MATRIX
# ------------------------------------------------

participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

matrix_source = (
    coded_df
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
)

matrix = pd.crosstab(
    matrix_source["Normalized_Theme"],
    matrix_source["Participant"]
)

matrix = matrix.reindex(
    columns=participants,
    fill_value=0
)

matrix = matrix.reset_index()

matrix["Frequency"] = matrix[
    participants
].sum(axis=1)

matrix["Percentage"] = (
    matrix["Frequency"]
    / len(participants)
    * 100
).round(1)


# ------------------------------------------------
# 12. THEME SUMMARY
# ------------------------------------------------

theme_summary = matrix[
    [
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].sort_values(
    "Frequency",
    ascending=False
)


# ------------------------------------------------
# 13. DECISION SUMMARY
# ------------------------------------------------

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Count"
    )
)


# ------------------------------------------------
# 14. PARTICIPANT COVERAGE
# ------------------------------------------------

participant_coverage = (
    coded_df
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "DTG_Theme_Count"
]


# ------------------------------------------------
# 15. SAVE EVERYTHING TO EXCEL
# ------------------------------------------------

output_file = Path(
    "DTG_Coding_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".xlsx"
)

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_df.to_excel(
        writer,
        sheet_name="04_Coding_Dictionary",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="07_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="08_Participant_Coverage",
        index=False
    )


# ------------------------------------------------
# 16. FINAL REPORT
# ------------------------------------------------

print("\n")
print("=" * 60)
print("DTG CODING COMPLETED")
print("=" * 60)

print(
    "Participants:",
    clean_df["Participant"].nunique()
)

print(
    "Raw theme observations:",
    len(clean_df)
)

print(
    "Unique DTG raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Normalized DTG themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print("\nDECISION SUMMARY:")

print(
    decision_summary.to_string(
        index=False
    )
)

print("\nOUTPUT FILE:")

print(
    output_file.resolve()
)

print("\nDONE.")

Participant sheets found:
P01 -> 1
P02 -> 2
P03 -> 3
P04 -> 4
P05 -> 5
P06 -> 6
P07 -> 7
P08 -> 8
P09 -> 9
P10 -> 10
P11 -> 11
P12 -> 12
P13 -> 13
P14 -> 14
P15 -> 15
P16 -> 16
P17 -> 17
P18 -> 18
P19 -> 19
P20 -> 20
P21 -> 21
P22 -> 22
P23 -> 23
P24 -> 24
P25 -> 25
P26 -> 26

DTG original responses:
26

DTG cleaned theme observations:
107


DTG RAW THEMES
                      Raw_Theme                     Theme_Clean                       Theme_Key
                         access                          access                          access
      access and change control       access and change control       access and change control
                 access control                  access control                  access control
               analytical trust                analytical trust                analytical trust
             Authoritative data              Authoritative data              authoritative data
                  authorization                   authorization  

In [16]:
# ================================================================
# COMPLETE DTG QUALITATIVE CODING ANALYSIS
# Input: DTG_Coding_20260827_132426.xlsx
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# ================================================================
# 1. INPUT
# ================================================================

input_file = Path("DTG_Coding_20260827_132426.xlsx")
CONSTRUCT = "DTG"

if not input_file.exists():
    raise FileNotFoundError(
        f"\nFile not found:\n{input_file.resolve()}\n\n"
        "Make sure the Excel file is in the same folder as this notebook."
    )

print("=" * 70)
print("DTG QUALITATIVE CODING ANALYSIS")
print("=" * 70)

excel = pd.ExcelFile(input_file)

print("\nInput file:", input_file.resolve())
print("\nAvailable sheets:")
for s in excel.sheet_names:
    print(" -", s)


# ================================================================
# 2. FLEXIBLE SHEET FINDER
# ================================================================

def find_sheet(possible_names):

    for name in possible_names:
        if name in excel.sheet_names:
            return name

    for sheet in excel.sheet_names:

        a = (
            str(sheet)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for name in possible_names:

            b = (
                str(name)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if b in a:
                return sheet

    return None


original_sheet = find_sheet([
    "01_Original_DTG",
    "01_Original",
    "Original_DTG",
    "Original"
])

raw_sheet = find_sheet([
    "02_Raw_DTG",
    "02_Raw",
    "Raw_DTG",
    "Raw"
])

clean_sheet = find_sheet([
    "03_Cleaned_DTG",
    "03_Cleaned",
    "Cleaned_DTG",
    "Cleaned"
])

coding_sheet = find_sheet([
    "04_Normalized_Coding",
    "04_Coding",
    "Normalized_Coding",
    "Coding"
])


print("\nSelected sheets:")
print("Original:", original_sheet)
print("Raw:", raw_sheet)
print("Cleaned:", clean_sheet)
print("Coding:", coding_sheet)


# ================================================================
# 3. READ DATA
# ================================================================

original_df = (
    pd.read_excel(input_file, sheet_name=original_sheet)
    if original_sheet else pd.DataFrame()
)

raw_df = (
    pd.read_excel(input_file, sheet_name=raw_sheet)
    if raw_sheet else pd.DataFrame()
)

clean_df = (
    pd.read_excel(input_file, sheet_name=clean_sheet)
    if clean_sheet else pd.DataFrame()
)

if coding_sheet is None:
    raise ValueError(
        "\nCould not find the Coding/Normalized Coding sheet.\n\n"
        "Available sheets:\n" +
        "\n".join(str(x) for x in excel.sheet_names)
    )

coding_df = pd.read_excel(
    input_file,
    sheet_name=coding_sheet
)


# ================================================================
# 4. STANDARDIZE COLUMN NAMES
# ================================================================

for df in [
    original_df,
    raw_df,
    clean_df,
    coding_df
]:

    if not df.empty:

        df.columns = [
            str(c).strip()
            for c in df.columns
        ]


print("\nCoding columns:")
print(list(coding_df.columns))


# ================================================================
# 5. FLEXIBLE COLUMN FINDER
# ================================================================

def find_column(df, candidates):

    for candidate in candidates:

        if candidate in df.columns:
            return candidate

    for column in df.columns:

        a = (
            str(column)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for candidate in candidates:

            b = (
                str(candidate)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if a == b:
                return column

    return None


# ================================================================
# 6. IDENTIFY CODING COLUMNS
# ================================================================

theme_key_col = find_column(
    coding_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

raw_theme_col = find_column(
    coding_df,
    [
        "Raw_Theme",
        "Raw Theme",
        "RawTheme"
    ]
)

normalized_col = find_column(
    coding_df,
    [
        "Normalized_Theme",
        "Normalized Theme",
        "NormalizedTheme"
    ]
)

decision_col = find_column(
    coding_df,
    [
        "Decision"
    ]
)


if theme_key_col is None:
    raise ValueError(
        "Theme_Key column was not found."
    )

if raw_theme_col is None:
    raise ValueError(
        "Raw_Theme column was not found."
    )

if normalized_col is None:
    raise ValueError(
        "Normalized_Theme column was not found."
    )

if decision_col is None:
    raise ValueError(
        "Decision column was not found."
    )


coding_df = coding_df.rename(
    columns={
        theme_key_col: "Theme_Key",
        raw_theme_col: "Raw_Theme",
        normalized_col: "Normalized_Theme",
        decision_col: "Decision"
    }
)


# ================================================================
# 7. PARTICIPANT COLUMN
# ================================================================

participant_col = find_column(
    clean_df,
    [
        "Participant",
        "Participant_ID",
        "Participant ID",
        "ParticipantID"
    ]
)

if participant_col is None:

    raise ValueError(
        "\nParticipant column was not found.\n\n"
        "Available columns:\n" +
        str(list(clean_df.columns))
    )

if participant_col != "Participant":

    clean_df = clean_df.rename(
        columns={
            participant_col: "Participant"
        }
    )


# ================================================================
# 8. THEME KEY IN CLEANED DATA
# ================================================================

clean_theme_key = find_column(
    clean_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

if clean_theme_key is None:

    raise ValueError(
        "\nTheme_Key was not found in the cleaned data.\n\n"
        "Available columns:\n" +
        str(list(clean_df.columns))
    )

if clean_theme_key != "Theme_Key":

    clean_df = clean_df.rename(
        columns={
            clean_theme_key: "Theme_Key"
        }
    )


# ================================================================
# 9. CLEAN CODING DICTIONARY
# ================================================================

coding_df["Theme_Key"] = (
    coding_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

coding_df["Raw_Theme"] = (
    coding_df["Raw_Theme"]
    .fillna("")
    .astype(str)
    .str.strip()
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .replace(
        [
            "nan",
            "None",
            "",
            "NaN"
        ],
        np.nan
    )
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .astype("string")
    .str.strip()
)

coding_df["Decision"] = (
    coding_df["Decision"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 10. CLEAN PARTICIPANT DATA
# ================================================================

clean_df["Theme_Key"] = (
    clean_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

clean_df["Participant"] = (
    clean_df["Participant"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 11. PARTICIPANTS
# ================================================================

participants = sorted(
    clean_df["Participant"]
    .replace("", np.nan)
    .dropna()
    .unique()
)

print("\nNumber of participants:", len(participants))
print("Participants:", participants)


# ================================================================
# 12. MERGE CLEANED DATA + CODING DICTIONARY
# ================================================================

coded_df = clean_df.merge(
    coding_df[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision"
        ]
    ],
    on="Theme_Key",
    how="left"
)


# ================================================================
# 13. CHECK UNMAPPED THEMES
# ================================================================

missing_mapping = coded_df[
    coded_df["Normalized_Theme"].isna()
].copy()

print(
    "\nNumber of unmapped themes:",
    len(missing_mapping)
)

if len(missing_mapping) > 0:

    print("\nUnmapped themes:")

    print(
        missing_mapping[
            [
                "Participant",
                "Theme_Key"
            ]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )


# ================================================================
# 14. PARTICIPANT × NORMALIZED DTG THEMES
# ================================================================

participant_theme = (
    coded_df[
        [
            "Participant",
            "Normalized_Theme"
        ]
    ]
    .dropna()
    .drop_duplicates()
)


# ================================================================
# 15. PARTICIPANT × THEME MATRIX
# ================================================================

matrix = pd.crosstab(
    participant_theme["Normalized_Theme"],
    participant_theme["Participant"]
)


# ================================================================
# 16. ORDER PARTICIPANTS P01–P26
# ================================================================

expected_participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

existing = [
    p
    for p in expected_participants
    if p in matrix.columns
]

other = [
    p
    for p in matrix.columns
    if p not in existing
]

matrix = matrix.reindex(
    columns=existing + other,
    fill_value=0
)

matrix = matrix.reset_index()

participant_columns = existing + other


# ================================================================
# 17. FREQUENCY
# ================================================================

matrix["Frequency"] = (
    matrix[participant_columns]
    .sum(axis=1)
)


# ================================================================
# 18. EXPERT PREVALENCE
# ================================================================

total_participants = len(participants)

if total_participants > 0:

    matrix["Percentage"] = (
        matrix["Frequency"]
        / total_participants
        * 100
    ).round(1)

else:

    matrix["Percentage"] = 0


# ================================================================
# 19. RANK DTG THEMES
# ================================================================

matrix = matrix.sort_values(
    [
        "Frequency",
        "Normalized_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)

matrix.insert(
    0,
    "Rank",
    range(
        1,
        len(matrix) + 1
    )
)


# ================================================================
# 20. DTG THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Rank",
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].copy()

theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme": "Final_DTG_Theme",
        "Frequency": "Experts_Mentioning",
        "Percentage": "Percentage_of_Experts"
    }
)


# ================================================================
# 21. PREVALENCE CATEGORY
# ================================================================

def prevalence_category(p):

    if p >= 75:
        return "Very High"

    elif p >= 50:
        return "High"

    elif p >= 25:
        return "Moderate"

    else:
        return "Low"


theme_summary["Prevalence_Category"] = (
    theme_summary[
        "Percentage_of_Experts"
    ].apply(prevalence_category)
)


# ================================================================
# 22. CODING AUDIT
# ================================================================

theme_participant_counts = (
    participant_theme
    .groupby("Normalized_Theme")
    .size()
    .reset_index(
        name="Experts_Mentioning"
    )
)

coding_audit = coding_df.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)

coding_audit["Experts_Mentioning"] = (
    coding_audit["Experts_Mentioning"]
    .fillna(0)
    .astype(int)
)

if total_participants > 0:

    coding_audit["Percentage_of_Experts"] = (
        coding_audit["Experts_Mentioning"]
        / total_participants
        * 100
    ).round(1)

else:

    coding_audit["Percentage_of_Experts"] = 0


coding_audit = coding_audit.sort_values(
    [
        "Normalized_Theme",
        "Decision",
        "Raw_Theme"
    ]
).reset_index(drop=True)


# ================================================================
# 23. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Number_of_Raw_Themes"
    )
)

if len(coding_df) > 0:

    decision_summary["Percentage"] = (
        decision_summary[
            "Number_of_Raw_Themes"
        ]
        / len(coding_df)
        * 100
    ).round(1)

else:

    decision_summary["Percentage"] = 0


# ================================================================
# 24. NORMALIZATION SUMMARY
# ================================================================

normalization_summary = (
    coding_df
    .dropna(
        subset=["Normalized_Theme"]
    )
    .groupby("Normalized_Theme")
    .agg(
        Raw_Themes=(
            "Raw_Theme",
            "count"
        ),

        Keep_Count=(
            "Decision",
            lambda x:
            (x == "Keep").sum()
        ),

        Merge_Count=(
            "Decision",
            lambda x:
            (x == "Merge").sum()
        )
    )
    .reset_index()
)


normalization_summary = (
    normalization_summary.merge(
        theme_participant_counts,
        on="Normalized_Theme",
        how="left"
    )
)

normalization_summary[
    "Experts_Mentioning"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)

if total_participants > 0:

    normalization_summary[
        "Percentage_of_Experts"
    ] = (
        normalization_summary[
            "Experts_Mentioning"
        ]
        / total_participants
        * 100
    ).round(1)

else:

    normalization_summary[
        "Percentage_of_Experts"
    ] = 0


normalization_summary = (
    normalization_summary
    .sort_values(
        "Experts_Mentioning",
        ascending=False
    )
    .reset_index(drop=True)
)


# ================================================================
# 25. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    participant_theme
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "Number_of_Normalized_DTG_Themes"
]


# ================================================================
# 26. DTG CONSTRUCT STATISTICS
# ================================================================

construct_statistics = pd.DataFrame({

    "Construct": [CONSTRUCT],

    "Participants": [
        total_participants
    ],

    "Original_Response_Rows": [
        len(original_df)
    ],

    "Raw_Theme_Observations": [
        len(raw_df)
    ],

    "Cleaned_Theme_Observations": [
        len(clean_df)
    ],

    "Unique_Raw_Themes": [
        coding_df["Theme_Key"].nunique()
    ],

    "Final_Normalized_Themes": [
        coding_df["Normalized_Theme"].nunique()
    ],

    "Merged_Raw_Themes": [
        (
            coding_df["Decision"] == "Merge"
        ).sum()
    ],

    "Kept_Raw_Themes": [
        (
            coding_df["Decision"] == "Keep"
        ).sum()
    ],

    "Unmapped_Themes": [
        len(missing_mapping)
    ]
})


# ================================================================
# 27. FINAL DTG EVIDENCE
# ================================================================

final_evidence = normalization_summary[
    [
        "Normalized_Theme",
        "Raw_Themes",
        "Experts_Mentioning",
        "Percentage_of_Experts",
        "Keep_Count",
        "Merge_Count"
    ]
].copy()

final_evidence = final_evidence.rename(
    columns={
        "Normalized_Theme": "Final_DTG_Theme",
        "Raw_Themes": "Number_of_Raw_Themes",
        "Percentage_of_Experts": "Expert_Prevalence_%",
        "Keep_Count": "Raw_Themes_Kept",
        "Merge_Count": "Raw_Themes_Merged"
    }
)

final_evidence[
    "Prevalence_Category"
] = (
    final_evidence[
        "Expert_Prevalence_%"
    ].apply(prevalence_category)
)


# ================================================================
# 28. QUALITY CHECKS
# ================================================================

duplicate_count = (
    len(coded_df)
    -
    len(
        coded_df[
            [
                "Participant",
                "Normalized_Theme"
            ]
        ]
        .dropna()
        .drop_duplicates()
    )
)

quality_checks = pd.DataFrame({

    "Check": [

        "Number of participants",
        "Original response rows",
        "Raw theme observations",
        "Unique raw themes",
        "Final normalized themes",
        "Unmapped themes",
        "Duplicate participant-theme records",
        "Keep decisions",
        "Merge decisions"

    ],

    "Result": [

        total_participants,
        len(original_df),
        len(raw_df),

        coding_df[
            "Theme_Key"
        ].nunique(),

        coding_df[
            "Normalized_Theme"
        ].nunique(),

        len(missing_mapping),

        duplicate_count,

        (
            coding_df["Decision"]
            == "Keep"
        ).sum(),

        (
            coding_df["Decision"]
            == "Merge"
        ).sum()

    ],

    "Status": [

        "PASS"
        if total_participants > 0
        else "CHECK",

        "PASS"
        if len(original_df) > 0
        else "CHECK",

        "PASS"
        if len(raw_df) > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Theme_Key"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Normalized_Theme"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if len(missing_mapping) == 0
        else "CHECK",

        "INFO",
        "PASS",
        "PASS"
    ]
})


# ================================================================
# 29. SAVE COMPLETE DTG WORKBOOK
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_file = Path(
    f"DTG_FINAL_QUALITATIVE_ANALYSIS_"
    f"{timestamp}.xlsx"
)

counter = 1

while output_file.exists():

    output_file = Path(
        f"DTG_FINAL_QUALITATIVE_ANALYSIS_"
        f"{timestamp}_{counter}.xlsx"
    )

    counter += 1


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_DTG_Evidence",
        index=False
    )

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    missing_mapping.to_excel(
        writer,
        sheet_name="13_Unmapped_Themes",
        index=False
    )


# ================================================================
# 30. FINAL REPORT
# ================================================================

print("\n")
print("=" * 70)
print("DTG ANALYSIS COMPLETED")
print("=" * 70)

print("\nParticipants:", total_participants)

print("Original responses:", len(original_df))

print("Raw theme observations:", len(raw_df))

print("Cleaned theme observations:", len(clean_df))

print(
    "Unique raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Final normalized DTG themes:",
    coding_df["Normalized_Theme"].nunique()
)

print(
    "Keep decisions:",
    (
        coding_df["Decision"] == "Keep"
    ).sum()
)

print(
    "Merge decisions:",
    (
        coding_df["Decision"] == "Merge"
    ).sum()
)

print(
    "Unmapped themes:",
    len(missing_mapping)
)


# ================================================================
# 31. DISPLAY FINAL DTG THEMES
# ================================================================

print("\n")
print("=" * 70)
print("FINAL DTG THEMES")
print("=" * 70)

print(
    final_evidence[
        [
            "Final_DTG_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Prevalence_Category"
        ]
    ].to_string(index=False)
)


# ================================================================
# 32. OUTPUT LOCATION
# ================================================================

print("\n")
print("=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(output_file.resolve())

print(
    "\nComplete DTG qualitative analysis workbook "
    "created successfully."
)

DTG QUALITATIVE CODING ANALYSIS

Input file: C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Decision_Intelligence_Risk_Coordination_Supply_Chain_Resilience_1\DTG_Coding_20260827_132426.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Dictionary
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Decision_Summary
 - 08_Participant_Coverage

Selected sheets:
Original: 01_Original_Responses
Raw: 02_Raw_Themes
Cleaned: 03_Cleaned_Themes
Coding: 04_Coding_Dictionary

Coding columns:
['Raw_Theme', 'Theme_Clean', 'Theme_Key', 'Normalized_Theme', 'Decision', 'Reason']

Number of participants: 26
Participants: ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26']

Number of unmapped themes: 0


DTG ANALYSIS COMPLETED

Participants: 26
Original responses: 26
Raw theme observations: 1

In [ ]:
# ================================================================
# DTG — FINAL QUALITATIVE EVIDENCE + DIMENSION ANALYSIS
# ================================================================
#
# INPUT:
# DTG_FINAL_QUALITATIVE_ANALYSIS_20260827_132512.xlsx
#
# OUTPUT:
# DTG_FINAL_EVIDENCE_AND_DIMENSIONS_20260827_132558.xlsx
#
# PURPOSE:
# 1. Read completed DTG qualitative analysis
# 2. Extract final DTG evidence
# 3. Remove capitalization duplicates
# 4. Group DTG themes into evidence-based content dimensions
# 5. Prepare the evidence for questionnaire-item development
#
# IMPORTANT:
# Dimensions are CONTENT DOMAINS, not additional constructs.
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import os


# ================================================================
# 1. INPUT FILE
# ================================================================

TARGET = "DTG_FINAL_QUALITATIVE_ANALYSIS_20260827_132512"

search_locations = [
    Path("/mnt/data"),
    Path("."),
    Path.home() / "Downloads",
    Path.home() / "Documents",
    Path.home() / "Desktop"
]

possible_files = []

for location in search_locations:

    if not location.exists():
        continue

    try:

        for ext in [".xlsx", ".xlsm", ".xls"]:

            possible_files.extend(
                location.rglob(TARGET + ext)
            )

    except Exception:
        pass


possible_files = list(
    dict.fromkeys(
        [p.resolve() for p in possible_files]
    )
)


if len(possible_files) > 0:

    INPUT_FILE = possible_files[0]

else:

    # Search for workbook containing DTG final evidence
    candidates = []

    for location in search_locations:

        if not location.exists():
            continue

        try:

            for p in location.rglob("*.xlsx"):

                try:

                    test_xls = pd.ExcelFile(
                        p,
                        engine="openpyxl"
                    )

                    if "11_Final_DTG_Evidence" in test_xls.sheet_names:
                        candidates.append(
                            p.resolve()
                        )

                except Exception:
                    pass

        except Exception:
            pass


    if len(candidates) == 0:

        raise FileNotFoundError(
            "\nDTG input workbook was not found.\n"
            f"Expected: {TARGET}.xlsx"
        )

    INPUT_FILE = candidates[0]


print("=" * 90)
print("DTG FINAL QUALITATIVE ANALYSIS")
print("=" * 90)

print("\nInput file:")
print(INPUT_FILE)


# ================================================================
# 2. READ WORKBOOK
# ================================================================

xls = pd.ExcelFile(
    INPUT_FILE,
    engine="openpyxl"
)

print("\nAvailable sheets:")

for s in xls.sheet_names:
    print(" -", s)


# ================================================================
# 3. LOAD ALL SHEETS
# ================================================================

sheets = {}

for sheet in xls.sheet_names:

    try:

        sheets[sheet] = pd.read_excel(
            INPUT_FILE,
            sheet_name=sheet
        )

    except Exception as e:

        print(
            f"Warning: could not read {sheet}: {e}"
        )


# ================================================================
# 4. HELPER FUNCTIONS
# ================================================================

def find_sheet(keyword):

    for s in sheets.keys():

        if keyword.lower() in s.lower():
            return s

    return None


def clean_text(x):

    if pd.isna(x):
        return ""

    x = str(x).strip()

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x


def prevalence_category(x):

    try:
        x = float(x)
    except:
        return "Not available"

    if x >= 75:
        return "Very High"

    elif x >= 50:
        return "High"

    elif x >= 25:
        return "Moderate"

    else:
        return "Low"


# ================================================================
# 5. IDENTIFY SHEETS
# ================================================================

original_sheet = find_sheet(
    "01_Original_Responses"
)

raw_sheet = find_sheet(
    "02_Raw_Themes"
)

cleaned_sheet = find_sheet(
    "03_Cleaned_Themes"
)

audit_sheet = find_sheet(
    "04_Coding_Audit"
)

matrix_sheet = find_sheet(
    "05_Participant_Matrix"
)

theme_sheet = find_sheet(
    "06_Theme_Summary"
)

normalization_sheet = find_sheet(
    "07_Normalization_Summary"
)

decision_sheet = find_sheet(
    "08_Decision_Summary"
)

coverage_sheet = find_sheet(
    "09_Participant_Coverage"
)

statistics_sheet = find_sheet(
    "10_Construct_Statistics"
)

final_sheet = find_sheet(
    "11_Final_DTG_Evidence"
)

quality_sheet = find_sheet(
    "12_Quality_Checks"
)

unmapped_sheet = find_sheet(
    "13_Unmapped_Themes"
)


if final_sheet is None:

    raise ValueError(
        "11_Final_DTG_Evidence was not found."
    )


# ================================================================
# 6. LOAD FINAL DTG EVIDENCE
# ================================================================

final_evidence = sheets[
    final_sheet
].copy()


final_evidence.columns = [
    str(c).strip()
    for c in final_evidence.columns
]


# ================================================================
# 7. IDENTIFY THEME COLUMN
# ================================================================

theme_col = None


if "Final_DTG_Theme" in final_evidence.columns:

    theme_col = "Final_DTG_Theme"

else:

    for c in final_evidence.columns:

        if (
            "DTG" in str(c)
            and "Theme" in str(c)
        ):

            theme_col = c
            break


if theme_col is None:

    for c in final_evidence.columns:

        if "Theme" in str(c):

            theme_col = c
            break


if theme_col is None:

    raise ValueError(
        "Could not identify the final DTG theme column."
    )


final_evidence[
    "Final_DTG_Theme"
] = (
    final_evidence[
        theme_col
    ]
    .apply(clean_text)
)


# ================================================================
# 8. STANDARDIZE IMPORTANT COLUMNS
# ================================================================

if "Experts_Mentioning" not in final_evidence.columns:

    for c in final_evidence.columns:

        if (
            "Experts" in str(c)
            and "Mention" in str(c)
        ):

            final_evidence[
                "Experts_Mentioning"
            ] = final_evidence[c]

            break


if "Expert_Prevalence_%" not in final_evidence.columns:

    for c in final_evidence.columns:

        if (
            "Prevalence" in str(c)
            and "%" in str(c)
        ):

            final_evidence[
                "Expert_Prevalence_%"
            ] = final_evidence[c]

            break


# ================================================================
# 9. REMOVE EMPTY THEMES
# ================================================================

final_evidence = final_evidence[
    final_evidence[
        "Final_DTG_Theme"
    ] != ""
].copy()


# ================================================================
# 10. PARTICIPANT COUNT
# ================================================================

participant_matrix = None

if matrix_sheet is not None:

    participant_matrix = sheets[
        matrix_sheet
    ].copy()


participants = []

if participant_matrix is not None:

    for c in participant_matrix.columns:

        cstr = str(c).strip()

        if re.match(
            r"^P\d+$",
            cstr,
            flags=re.IGNORECASE
        ):

            participants.append(cstr)


if len(participants) > 0:

    n_participants = len(participants)

else:

    # Your qualitative coding dataset
    n_participants = 26


print(
    "\nParticipants used:",
    n_participants
)


# ================================================================
# 11. PREVALENCE CATEGORY
# ================================================================

if "Expert_Prevalence_%" in final_evidence.columns:

    final_evidence[
        "Prevalence_Category"
    ] = (
        final_evidence[
            "Expert_Prevalence_%"
        ]
        .apply(
            prevalence_category
        )
    )


# ================================================================
# 12. NORMALIZE CAPITALIZATION DUPLICATES
# ================================================================

final_evidence[
    "_Theme_Normalized"
] = (
    final_evidence[
        "Final_DTG_Theme"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)


if "Expert_Prevalence_%" in final_evidence.columns:

    final_evidence[
        "_Prevalence_Numeric"
    ] = pd.to_numeric(
        final_evidence[
            "Expert_Prevalence_%"
        ],
        errors="coerce"
    ).fillna(0)

else:

    final_evidence[
        "_Prevalence_Numeric"
    ] = 0


final_evidence = (
    final_evidence
    .sort_values(
        "_Prevalence_Numeric",
        ascending=False
    )
)


final_evidence = (
    final_evidence
    .drop_duplicates(
        subset=[
            "_Theme_Normalized"
        ],
        keep="first"
    )
    .copy()
)


final_evidence = (
    final_evidence
    .drop(
        columns=[
            "_Theme_Normalized",
            "_Prevalence_Numeric"
        ],
        errors="ignore"
    )
)


# ================================================================
# 13. SORT FINAL EVIDENCE
# ================================================================

if "Experts_Mentioning" in final_evidence.columns:

    final_evidence[
        "_Sort"
    ] = pd.to_numeric(
        final_evidence[
            "Experts_Mentioning"
        ],
        errors="coerce"
    ).fillna(0)


    final_evidence = (
        final_evidence
        .sort_values(
            "_Sort",
            ascending=False
        )
        .drop(
            columns=["_Sort"]
        )
        .reset_index(drop=True)
    )


# ================================================================
# 14. FINAL DTG EVIDENCE TABLE
# ================================================================

preferred_columns = [

    "Final_DTG_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Raw_Themes_Kept",

    "Raw_Themes_Merged",

    "Prevalence_Category"

]


evidence_columns = [

    c
    for c in preferred_columns
    if c in final_evidence.columns

]


final_dtg_evidence = final_evidence[
    evidence_columns
].copy()


if "Rank" in final_dtg_evidence.columns:

    final_dtg_evidence = (
        final_dtg_evidence
        .drop(
            columns=["Rank"]
        )
    )


final_dtg_evidence.insert(
    0,
    "Rank",
    range(
        1,
        len(final_dtg_evidence) + 1
    )
)


# ================================================================
# 15. DTG DIMENSION MAPPING
# ================================================================
#
# These are qualitative CONTENT DOMAINS for item development.
# They are NOT separate constructs.
#
# DTG = Data Trust & Governance
#
# Core domains:
#   1. Data accuracy & quality
#   2. Data integrity & consistency
#   3. Data security & protection
#   4. Data access & availability
#   5. Data transparency & traceability
#   6. Data ownership & accountability
#   7. Governance policies & standards
#   8. Data sharing & controlled access
#   9. Data validation & monitoring
#  10. Compliance & responsible data governance
# ================================================================

def map_dtg_dimension(theme):

    t = str(theme).strip().lower()


    # ------------------------------------------------------------
    # 1. DATA ACCURACY & QUALITY
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "accuracy",
            "accurate data",
            "data quality",
            "quality of data",
            "high-quality data",
            "reliable data",
            "data reliability",
            "completeness",
            "complete data",
            "timely data",
            "data consistency",
            "consistency of data"

        ]
    ):

        return (
            "Data Accuracy & Quality"
        )


    # ------------------------------------------------------------
    # 2. DATA INTEGRITY
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "data integrity",
            "integrity",
            "tamper",
            "tamper-proof",
            "unaltered",
            "data authenticity",
            "authenticity",
            "data validation",
            "validation"

        ]
    ):

        return (
            "Data Integrity & Validation"
        )


    # ------------------------------------------------------------
    # 3. DATA SECURITY & PROTECTION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "security",
            "data security",
            "cybersecurity",
            "cyber security",
            "data protection",
            "protection of data",
            "privacy",
            "data privacy",
            "unauthorized access",
            "access protection",
            "confidentiality"

        ]
    ):

        return (
            "Data Security & Protection"
        )


    # ------------------------------------------------------------
    # 4. DATA ACCESS & AVAILABILITY
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "data access",
            "accessibility",
            "accessible data",
            "availability",
            "data availability",
            "real-time access",
            "timely access",
            "access to information",
            "information access"

        ]
    ):

        return (
            "Data Access & Availability"
        )


    # ------------------------------------------------------------
    # 5. TRANSPARENCY & TRACEABILITY
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "transparency",
            "transparent",
            "traceability",
            "traceable",
            "data provenance",
            "provenance",
            "audit trail",
            "visibility",
            "data visibility",
            "tracking"

        ]
    ):

        return (
            "Data Transparency & Traceability"
        )


    # ------------------------------------------------------------
    # 6. OWNERSHIP & ACCOUNTABILITY
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "ownership",
            "data owner",
            "accountability",
            "responsibility",
            "responsible data",
            "data stewardship",
            "stewardship",
            "custodian",
            "data custodian"

        ]
    ):

        return (
            "Data Ownership & Accountability"
        )


    # ------------------------------------------------------------
    # 7. GOVERNANCE POLICIES & STANDARDS
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "governance",
            "governance policy",
            "governance policies",
            "data policy",
            "data policies",
            "standards",
            "data standards",
            "rules",
            "governance framework",
            "governance structure",
            "procedures",
            "protocols"

        ]
    ):

        return (
            "Data Governance Policies & Standards"
        )


    # ------------------------------------------------------------
    # 8. DATA SHARING & COLLABORATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "data sharing",
            "information sharing",
            "sharing data",
            "controlled sharing",
            "secure sharing",
            "data exchange",
            "information exchange",
            "interorganizational data",
            "data collaboration",
            "collaboration"

        ]
    ):

        return (
            "Controlled Data Sharing & Collaboration"
        )


    # ------------------------------------------------------------
    # 9. MONITORING & AUDITING
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "monitoring",
            "data monitoring",
            "continuous monitoring",
            "audit",
            "auditing",
            "data audit",
            "quality monitoring",
            "governance monitoring",
            "checking",
            "review"

        ]
    ):

        return (
            "Data Monitoring & Auditing"
        )


    # ------------------------------------------------------------
    # 10. COMPLIANCE
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "compliance",
            "regulatory",
            "regulation",
            "legal",
            "law",
            "gdpr",
            "ethical",
            "ethics",
            "responsible use"

        ]
    ):

        return (
            "Data Compliance & Responsible Governance"
        )


    # ------------------------------------------------------------
    # 11. GENERAL DATA TRUST
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "trust",
            "trusted data",
            "data trust",
            "confidence in data",
            "confidence"

        ]
    ):

        return (
            "Data Trust & Confidence"
        )


    # ------------------------------------------------------------
    # 12. REVIEW
    # ------------------------------------------------------------

    return "Review Required"


final_dtg_evidence[
    "DTG_Dimension"
] = (
    final_dtg_evidence[
        "Final_DTG_Theme"
    ]
    .apply(
        map_dtg_dimension
    )
)


# ================================================================
# 16. MANUAL EXACT-THEME OVERRIDES
# ================================================================

manual_mapping = {

    "data accuracy":
        "Data Accuracy & Quality",

    "accuracy":
        "Data Accuracy & Quality",

    "data quality":
        "Data Accuracy & Quality",

    "data consistency":
        "Data Accuracy & Quality",

    "consistency":
        "Data Accuracy & Quality",

    "data integrity":
        "Data Integrity & Validation",

    "integrity":
        "Data Integrity & Validation",

    "data validation":
        "Data Integrity & Validation",

    "validation":
        "Data Integrity & Validation",

    "data security":
        "Data Security & Protection",

    "security":
        "Data Security & Protection",

    "data protection":
        "Data Security & Protection",

    "data privacy":
        "Data Security & Protection",

    "privacy":
        "Data Security & Protection",

    "confidentiality":
        "Data Security & Protection",

    "data access":
        "Data Access & Availability",

    "data availability":
        "Data Access & Availability",

    "availability":
        "Data Access & Availability",

    "accessibility":
        "Data Access & Availability",

    "transparency":
        "Data Transparency & Traceability",

    "traceability":
        "Data Transparency & Traceability",

    "data provenance":
        "Data Transparency & Traceability",

    "provenance":
        "Data Transparency & Traceability",

    "audit trail":
        "Data Transparency & Traceability",

    "data ownership":
        "Data Ownership & Accountability",

    "ownership":
        "Data Ownership & Accountability",

    "accountability":
        "Data Ownership & Accountability",

    "responsibility":
        "Data Ownership & Accountability",

    "data governance":
        "Data Governance Policies & Standards",

    "governance":
        "Data Governance Policies & Standards",

    "governance policies":
        "Data Governance Policies & Standards",

    "data policies":
        "Data Governance Policies & Standards",

    "data standards":
        "Data Governance Policies & Standards",

    "data sharing":
        "Controlled Data Sharing & Collaboration",

    "information sharing":
        "Controlled Data Sharing & Collaboration",

    "controlled data sharing":
        "Controlled Data Sharing & Collaboration",

    "data exchange":
        "Controlled Data Sharing & Collaboration",

    "monitoring":
        "Data Monitoring & Auditing",

    "data monitoring":
        "Data Monitoring & Auditing",

    "audit":
        "Data Monitoring & Auditing",

    "auditing":
        "Data Monitoring & Auditing",

    "compliance":
        "Data Compliance & Responsible Governance",

    "regulatory compliance":
        "Data Compliance & Responsible Governance",

    "ethical data use":
        "Data Compliance & Responsible Governance",

    "data trust":
        "Data Trust & Confidence",

    "trust":
        "Data Trust & Confidence",

    "trusted data":
        "Data Trust & Confidence"

}


for theme, dimension in manual_mapping.items():

    mask = (
        final_dtg_evidence[
            "Final_DTG_Theme"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq(
            theme.lower()
        )
    )

    final_dtg_evidence.loc[
        mask,
        "DTG_Dimension"
    ] = dimension


# ================================================================
# 17. DIMENSION SUMMARY
# ================================================================

dimension_rows = []


for dimension, group in (
    final_dtg_evidence
    .groupby(
        "DTG_Dimension",
        dropna=False
    )
):

    themes = (
        group[
            "Final_DTG_Theme"
        ]
        .astype(str)
        .tolist()
    )


    experts = 0


    if "Experts_Mentioning" in group.columns:

        try:

            experts = int(
                pd.to_numeric(
                    group[
                        "Experts_Mentioning"
                    ],
                    errors="coerce"
                )
                .max()
            )

        except:

            experts = 0


    prevalence = (

        experts
        /
        n_participants
        *
        100

        if n_participants > 0
        else 0

    )


    dimension_rows.append({

        "DTG_Dimension":
            dimension,

        "Number_of_Normalized_Themes":
            len(themes),

        "Experts_Mentioning":
            experts,

        "Expert_Prevalence_%":
            round(
                prevalence,
                1
            ),

        "Prevalence_Category":
            prevalence_category(
                prevalence
            ),

        "Included_DTG_Themes":
            "; ".join(
                themes
            )

    })


dimension_summary = pd.DataFrame(
    dimension_rows
)


# ================================================================
# 18. SORT DIMENSIONS
# ================================================================

if len(dimension_summary) > 0:

    dimension_summary[
        "_sort"
    ] = pd.to_numeric(
        dimension_summary[
            "Experts_Mentioning"
        ],
        errors="coerce"
    ).fillna(0)


    dimension_summary = (
        dimension_summary
        .sort_values(
            "_sort",
            ascending=False
        )
        .drop(
            columns=["_sort"]
        )
        .reset_index(drop=True)
    )


    if "Rank" in dimension_summary.columns:

        dimension_summary = (
            dimension_summary
            .drop(
                columns=["Rank"]
            )
        )


    dimension_summary.insert(
        0,
        "Rank",
        range(
            1,
            len(dimension_summary) + 1
        )
    )


# ================================================================
# 19. DIMENSION × THEME TABLE
# ================================================================

dimension_theme_columns = [

    "DTG_Dimension",

    "Final_DTG_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Prevalence_Category"

]


dimension_theme_columns = [

    c
    for c in dimension_theme_columns
    if c in final_dtg_evidence.columns

]


dimension_themes = final_dtg_evidence[
    dimension_theme_columns
].copy()


if "Experts_Mentioning" in dimension_themes.columns:

    dimension_themes[
        "_sort"
    ] = pd.to_numeric(
        dimension_themes[
            "Experts_Mentioning"
        ],
        errors="coerce"
    ).fillna(0)


    dimension_themes = (
        dimension_themes
        .sort_values(
            [
                "DTG_Dimension",
                "_sort"
            ],
            ascending=[
                True,
                False
            ]
        )
        .drop(
            columns=["_sort"]
        )
    )


# ================================================================
# 20. THEMES REQUIRING REVIEW
# ================================================================

unmapped_check = final_dtg_evidence[
    final_dtg_evidence[
        "DTG_Dimension"
    ]
    ==
    "Review Required"
].copy()


if len(unmapped_check) > 0:

    review_columns = [

        "Final_DTG_Theme",

        "Experts_Mentioning",

        "Expert_Prevalence_%",

        "DTG_Dimension"

    ]


    review_columns = [

        c
        for c in review_columns
        if c in unmapped_check.columns

    ]


    unmapped_check = unmapped_check[
        review_columns
    ].copy()


    unmapped_check.insert(
        0,
        "Status",
        "REVIEW REQUIRED"
    )


else:

    unmapped_check = pd.DataFrame({

        "Status": [
            "PASS — All DTG themes mapped to a dimension."
        ]

    })


# ================================================================
# 21. QUALITY CHECKS
# ================================================================

quality_rows = []


quality_rows.append({

    "Quality_Check":
        "Input workbook found",

    "Result":
        "PASS",

    "Details":
        INPUT_FILE.name

})


quality_rows.append({

    "Quality_Check":
        "Final DTG evidence available",

    "Result":
        "PASS"
        if len(final_dtg_evidence) > 0
        else "FAIL",

    "Details":
        f"{len(final_dtg_evidence)} normalized DTG themes"

})


quality_rows.append({

    "Quality_Check":
        "Participants",

    "Result":
        n_participants,

    "Details":
        "Participant count used"

})


quality_rows.append({

    "Quality_Check":
        "DTG dimensions generated",

    "Result":
        len(dimension_summary),

    "Details":
        "Evidence-based content domains"

})


review_count = (

    len(unmapped_check)

    if "Final_DTG_Theme"
    in unmapped_check.columns

    else 0

)


quality_rows.append({

    "Quality_Check":
        "Themes requiring manual review",

    "Result":
        review_count,

    "Details":
        "Themes assigned Review Required"

})


duplicate_count = int(
    final_dtg_evidence[
        "Final_DTG_Theme"
    ]
    .astype(str)
    .str.lower()
    .duplicated()
    .sum()
)


quality_rows.append({

    "Quality_Check":
        "Remaining capitalization duplicates",

    "Result":
        duplicate_count,

    "Details":
        "Should normally be zero"

})


quality_checks_new = pd.DataFrame(
    quality_rows
)


# ================================================================
# 22. SUPPORTING SHEETS
# ================================================================

def get_sheet_or_empty(sheet):

    if sheet is not None:
        return sheets[sheet].copy()

    return pd.DataFrame()


original_responses = get_sheet_or_empty(
    original_sheet
)

raw_themes = get_sheet_or_empty(
    raw_sheet
)

cleaned_themes = get_sheet_or_empty(
    cleaned_sheet
)

coding_audit = get_sheet_or_empty(
    audit_sheet
)

theme_summary_existing = get_sheet_or_empty(
    theme_sheet
)

normalization_summary = get_sheet_or_empty(
    normalization_sheet
)

decision_summary = get_sheet_or_empty(
    decision_sheet
)

participant_coverage = get_sheet_or_empty(
    coverage_sheet
)

construct_statistics = get_sheet_or_empty(
    statistics_sheet
)


# ================================================================
# 23. OUTPUT FILE
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"DTG_FINAL_EVIDENCE_AND_DIMENSIONS_{timestamp}.xlsx"
)


# ================================================================
# 24. WRITE OUTPUT
# ================================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    original_responses.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_themes.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    cleaned_themes.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    if participant_matrix is not None:

        participant_matrix.to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    theme_summary_existing.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    unmapped_check.to_excel(
        writer,
        sheet_name="11_Unmapped_Check",
        index=False
    )

    quality_checks_new.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    final_dtg_evidence.to_excel(
        writer,
        sheet_name="13_Final_DTG_Evidence",
        index=False
    )

    dimension_summary.to_excel(
        writer,
        sheet_name="14_DTG_Dimension_Summary",
        index=False
    )

    dimension_themes.to_excel(
        writer,
        sheet_name="15_DTG_Dimension_Themes",
        index=False
    )


# ================================================================
# 25. FINAL REPORT
# ================================================================

print("\n")
print("=" * 90)
print("DTG PROCESS COMPLETED SUCCESSFULLY")
print("=" * 90)

print("\nOutput file:")
print(
    OUTPUT_FILE.resolve()
)

print("\nNormalized final DTG themes:")
print(
    len(final_dtg_evidence)
)

print("\nDTG content dimensions:")
print(
    len(dimension_summary)
)


print("\n")
print("-" * 90)
print("DTG DIMENSION SUMMARY")
print("-" * 90)

if len(dimension_summary) > 0:

    print(
        dimension_summary.to_string(
            index=False
        )
    )

else:

    print(
        "No dimensions generated."
    )


print("\n")
print("-" * 90)
print("QUALITY CHECKS")
print("-" * 90)

print(
    quality_checks_new.to_string(
        index=False
    )
)


print("\n")
print("=" * 90)
print("NEXT STAGE")
print("=" * 90)

print("""
Use these three sheets for DTG questionnaire development:

13_Final_DTG_Evidence
14_DTG_Dimension_Summary
15_DTG_Dimension_Themes

The DTG dimensions are qualitative content domains.
They are NOT separate constructs.

The next step is to convert the strongest and
conceptually distinct DTG themes into candidate
reflective questionnaire items, while avoiding
duplicate or overlapping items.
""")

DTG FINAL QUALITATIVE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Decision_Intelligence_Risk_Coordination_Supply_Chain_Resilience_1\DTG_FINAL_QUALITATIVE_ANALYSIS_20260827_132512.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Audit
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Normalization_Summary
 - 08_Decision_Summary
 - 09_Participant_Coverage
 - 10_Construct_Statistics
 - 11_Final_DTG_Evidence
 - 12_Quality_Checks
 - 13_Unmapped_Themes

Participants used: 26


DTG PROCESS COMPLETED SUCCESSFULLY

Output file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Decision_Intelligence_Risk_Coordination_Supply_Chain_Resilience_1\DTG_FINAL_EVIDENCE_AND_DIMENSIONS_20260827_132558.xlsx

Normalized final DTG themes:
47

DTG content dimensions:
7


------------------------------------------------------------------------------------------
DTG